# Certified classical and quantum KKT solves inside CVXOPT

Companion notebook of the paper *Certified Classical and Quantum Linear Solves inside a Production Interior-Point Method: Equilibration, Low-Precision Refinement, and Conditions for a Quantum Advantage*.

The notebook replaces the KKT factorisation of `cvxopt.solvers.coneqp` through its `kktsolver` callback. Every replacement, classical or quantum, runs under the same acceptance test, the same classical-fallback protocol and the same accounting. The notebook contains

* instance generators: synthetic QP families with a known primal-dual solution, a subset of the Maros-Mészáros QP collection and, optionally, SDPLIB problems solved with `conelp`;
* classical KKT solvers: LU and reduced (CHOLMOD) direct references, equilibrated MINRES with a baseline and a latency-optimised kernel (NumPy or CuPy), CG on the reduced and null-space systems, lazy refactorisation;
* a gate-level statevector simulator, phase-estimation HHL, and a polynomial linear solver (qubitised walk with a linear combination of unitaries) for the augmented and the null-space target systems;
* every experiment of the paper, the figures, a summary report and a CSV export of all tables.

All quantum results are classical simulations of ideal circuits with oracle block encoding and state preparation.

## Running the notebook

There are two builds. They differ only in the default device in the configuration cell.

| build | Kaggle accelerator | measured run time, profile `medium` |
|---|---|---|
| `quantum_kkt_cvxopt_cpu.ipynb` | None | 4.7 h |
| `quantum_kkt_cvxopt_gpu.ipynb` | GPU T4 | 5.8 h |

Switch Internet access on in the Kaggle session settings. The notebook downloads the Maros-Mészáros and SDPLIB files at run time and installs Qiskit for the simulator cross-check; without Internet the affected experiments are reported as `skipped` and everything else runs.

The problem sizes are chosen by `RUN_PROFILE` in the configuration cell or by the environment variable `QKKT_PROFILE`: `smoke` (a few minutes, exercises every code path), `small` (default), `medium` (used for the paper) and `large`. All output goes to `/kaggle/working/results/` (`./kaggle_working_local/results/` outside Kaggle) and is compressed into `results_bundle_<cpu|gpu>_<profile>.zip` at the end.

## Notebook sections and the paper

| section | experiment id | paper |
|---|---|---|
| 1-9 | configuration, generators, KKT assembly, solvers, runner | Sections 3, 4 and 6 |
| 10 | `E1_contract_tiny` | contract check of the callback, Section 3.1, Table 4 |
| 10 | `E2_main` | acceptance-test ablation, Section 7.1, Table 5, Fig. 2 |
| 10 | `E3_scaling` | scaling, Section 7.3, Table 6, Figs. 4-5 |
| 10, 11 | `E4_noise`, `E4b_tolerance_sweep` | precision of a Newton direction, Section 7.6, Table 8, Fig. 8 |
| 12 | `E8_maros_meszaros` | real-world QPs, Section 7.5, Table 7, Fig. 7 |
| 16 | `Q1_captured`, `Q2_qiskit_crosscheck`, `Q3_integrated` | HHL, Section 8.2, Fig. 9(a) |
| 17 | `Q4_regularization_qp`, `Q4_regularization_sdplib` | regularised target with refinement; not reported in the paper |
| 18 | `D1_dequantisation`, `E5_outer_refinement`, `E6_gpu_matvec` | Sections 7.4, 7.6 and 7.7 |
| 19 | `E7_spectrum_deflation` | conditioning along the IPM run, Section 7.2, Fig. 3 |
| 20 | `Q5ab_cks_captured` | polynomial solver and refinement cost, Sections 8.3-8.4, Tables 10-11, Figs. 9-11 |
| 20 | `Q5c_integrated_cks`, `Q5e_shot_sweep` | quantum solvers inside the IPM, Sections 8.5-8.7, Tables 12-14, Figs. 12-14 |
| 21 | `S1_sdplib` | SDPLIB, Appendix B, Table 18 |

## 1. Configuration

Everything that controls the size and scope of the experiments is in the next cell. `RUN_PROFILE` (or the environment variable `QKKT_PROFILE`) selects the problem sizes; `smoke` runs every code path in a few minutes, `small` is the default and `medium` is the profile used for the paper. `DEFAULT_DEVICE` is `"cpu"` in the CPU build and `"gpu"` in the GPU build; the environment variable `QKKT_DEVICE` overrides it. The GPU build also extends the two sparse families to KKT dimension 79 360.

The configuration and a hash of it are written to `results/config.json`; every result row carries the hash.

In [ ]:
# Configuration: the only cell that normally needs editing.
import os, sys, json, time, platform, hashlib

CODE_VERSION = "1.0.0"
DEFAULT_DEVICE = "gpu"          # "cpu" | "gpu"; without a usable GPU every GPU method is reported as skipped
RUN_PROFILE = os.environ.get("QKKT_PROFILE", "small")     # "smoke" | "small" | "medium" | "large"

CONFIG = {
    "code_version": CODE_VERSION, "run_profile": RUN_PROFILE,
    "device": os.environ.get("QKKT_DEVICE", DEFAULT_DEVICE),
    "global_seed": 20260911,
    "output_root": "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./kaggle_working_local"),

    # CVXOPT algorithm parameters (CVXOPT defaults, identical for every method)
    "cvxopt": {"maxiters": 100, "abstol": 1e-7, "reltol": 1e-6, "feastol": 1e-7, "refinement_qp": 0, "refinement_sdp": 1},

    # acceptance of the final iterate, recomputed independently of the solver's own report
    "accept": {"pres": 1e-6, "dres": 1e-6, "relgap": 1e-5, "cone_viol": 1e-8, "obj_rel_err": 1e-5},

    # synthetic benchmark families
    "families": {
        "dense_kappa": {"structure": "dense", "n_list": [10, 20, 40], "m_ratio": 2.0, "p_ratio": 0.2,
                        "kappa_list": [1e0, 1e3, 1e6], "seeds": [0, 1, 2], "active_frac": 0.3, "density": 1.0},
        "dense_noeq": {"structure": "dense", "n_list": [10, 40], "m_ratio": 2.0, "p_ratio": 0.0,
                       "kappa_list": [1e0, 1e6], "seeds": [0, 1], "active_frac": 0.3, "density": 1.0},
        # random 5 % density: G'G fills in completely, so this family is kept small
        "sparse_scale": {"structure": "random_sparse", "n_list": [25, 50, 100, 200], "m_ratio": 2.0, "p_ratio": 0.1,
                         "kappa_list": [1e3], "seeds": [0], "active_frac": 0.3, "density": 0.05},
        # banded P, 4 nonzeros per row of G and A in random columns: P + G'W^-2 G is a random sparse graph whose
        # Cholesky factor fills in almost completely (high-fill family); `seeds_large` applies from `seeds_large_from_n` on
        "sparse_banded": {"structure": "banded", "n_list": [100, 200, 400, 800, 1600, 3200], "m_ratio": 2.0, "p_ratio": 0.1,
                          "kappa_list": [1e3], "seeds": [0, 1, 2], "seeds_large": [0], "seeds_large_from_n": 1600, "active_frac": 0.3, "density": 0.0, "nnz_per_row": 4, "bandwidth": 3},
        # same sizes, but the nonzeros of each row of G and A lie in a window of 13 neighbouring columns: P + G'W^-2 G is
        # banded and sparse Cholesky is O(n) (low-fill control family, where a direct solver should win)
        "sparse_lowfill": {"structure": "banded_local", "n_list": [100, 200, 400, 800, 1600, 3200], "m_ratio": 2.0, "p_ratio": 0.1,
                           "kappa_list": [1e3], "seeds": [0, 1], "seeds_large": [0], "seeds_large_from_n": 1600, "active_frac": 0.3, "density": 0.0, "nnz_per_row": 4, "bandwidth": 3},
    },
    "repeats": 1,
    "inexact": {
        # acceptance tests for one KKT solve (section 3.4): "relres" = ||Ku - r|| / ||r||; "backward" = normwise backward error
        # ||Ku - r|| / (||K||_2 ||u|| + ||r||); "block" = per-block forcing ||e_b|| <= eta * max(||r_b||, floor_b); "combined" = block AND backward
        "be_tols": [1e-6, 1e-8, 1e-10, 1e-12],
        "be_floor": 1e-14,                  # backward error of a direct solve; used by "relres_floor"
        "full_ablation": False,             # True: full criterion x scaling x tolerance grid in E2; False: essential method set
        "warm_start": True,                 # combined-direction solve starts from the affine direction (minres_base_*_ws_*)
        "switch_policy": True,              # inexact solver until its first rejected direction, exact solver afterwards (*_switch_*)
        "block_etas": [0.5, 0.1, 0.01],
        "relres_tol": 1e-6,                 # tolerance of the relative-residual test
        "accept_factor": 10.0,              # "relres" / "backward": accept if value <= accept_factor * tolerance
        "block_abs_floor": 1e-9,            # absolute floor per block, times max(1, ||q||), max(1, ||b||), max(1, ||h||)
        "max_refine": 2, "fallback": True,
        "check_adaptive_div": 16,           # fused kernel: next explicit check after max(check_every, it // 16) iterations
        "gpu_fuse": True,                   # fuse the vector recurrences with cupy.fuse (self-tested; plain code otherwise)
        "gpu_auto_min_dim": 20000,          # `*_auto` methods use the GPU from this KKT dimension on
        "oss_max_n": 3200,                  # null-space methods form a dense n x (n - p) basis of null(A): only up to this n
        "fallback_reduced": True,           # fallback factors the reduced quasi-definite system (CHOLMOD); False: LU of the full KKT matrix
        "check_every": 8,                   # Krylov iterations between explicit acceptance checks
        "norm_est_iters": 20,               # power iterations for ||K||_2 (once per factorisation)
        "adaptive_nt": {"eta": 1e-2, "tol_min": 1e-10, "tol_max": 1e-2},
        "noise_iso": [1e-8, 1e-7, 1e-6],    # isotropic error model: delta * ||u|| spread over all components
        "noise_block": [1e-1, 1e-2, 1e-3],  # block-relative error model: delta * ||u_b|| per block
        "minres_maxiter_cap": 20000, "minres_maxiter_factor": 2, "ruiz_iters": 6,
    },
    "cond_limit": 600,
    "memory_sampling": {"enabled": True, "interval_s": 0.05},     # peak RSS (and CuPy pool) sampled by a thread during every coneqp call
    # Maros-Meszaros QPs: .mat files of github.com/qpsolvers/maros_meszaros_qpbenchmark, min 1/2 x'Px + q'x + r  s.t.  l <= Ax <= u
    "maros_meszaros": {
        "enabled": True, "search_dirs": ["/kaggle/input", "./maros_meszaros", "./data/maros_meszaros"],
        "download_if_missing": True, "download_url": "https://raw.githubusercontent.com/qpsolvers/maros_meszaros_qpbenchmark/main/data/{name}.mat",
        # KKT dimension after conversion: 91 ... 1 260 in the default profile.  Left out because the unmodified coneqp run
        # ends with status 'unknown': QSHARE2B, QRECIPE, QPCBOEI2, QFORPLAN, QCAPRI, QSCFXM1-3, QGFRDXPN.
        "instances": ["QAFIRO", "DUAL1", "CVXQP1_S", "QPCBLEND", "QADLITTL", "DUALC1", "QSCAGR7", "PRIMALC1", "QISRAEL", "QSC205", "PRIMAL1", "QSCTAP1", "KSIP"],
        "methods": ["cvxopt_default", "custom_exact_reduced", "gmres_stalelu_comb", "minres_fused_comb_cpu", "minres_fused_comb_switch_cpu", "minres_fused_comb_gpu_mm"],
        "budget_s": 120, "max_kkt_dim": 12000,
    },
    # E4b: IPM tolerance level tau x direction error; acceptance thresholds scale with tau (tau = 1e-7 reproduces the defaults)
    "tolerance_sweep": {"levels": [1e-4, 1e-6, 1e-8], "noise_iso": [1e-8, 1e-7, 1e-6, 1e-5], "noise_block": [1e-1, 1e-2, 1e-3], "extra_methods": ["cvxopt_default", "minres_fused_comb_cpu"],
                        "families": ["dense_kappa", "dense_noeq"], "n_max": 40, "max_instances": 30},
    "capture": {"max_dim": 160, "instances": ["tiny_A", "tiny_B", "tiny_C", "dense_kappa_n20_k6_s0", "dense_kappa_n40_k6_s0"]},   # KKT systems saved from the reference run

    # quantum linear solvers (statevector simulation)
    "quantum": {
        "enabled": True, "max_qubits": 17, "max_state_bytes": 64 * 2**20,
        "n_clock_list": [4, 6, 8, 10], "shots_list": [1000, 10000, 100000],
        "eff_shots_list": [100, 1000, 10000],   # accepted samples per setting with amplitude-amplification-emulated readout
        "sampling_repeats": 3, "scalings": ["none", "ruiz"], "reciprocal_modes": ["resolution", "kappa_known"],
        "ir_rounds": 4,                         # refinement rounds with HHL on the true residual (Q1)
        # Q4: regularisation delta added to the Ruiz-scaled system of ill-conditioned captured systems; corrector = LU of the
        # regularised system (classical diagnostic) or HHL with ideal readout (padded dimension <= quantum_max_dim)
        "reg_experiment": {"deltas": [0.0, 1e-4, 1e-2, 1e-1], "ir_rounds": 6, "min_kappa_scaled": 50.0, "classical_max_dim": 512, "quantum_max_dim": 32, "quantum_n_clock": [6, 8]},
        "spectral_bound": "power", "power_iters": 40, "power_safety": 1.15, "reference": "uniform", "captured_iters": "auto",
        "integrated": {
            "instances": ["tiny_A", "tiny_B"],
            "runs": [   # combined acceptance test unless 'criterion' overrides it
                {"name": "hhl_noscale_res_nc8_fb",     "recon": "statevector", "n_clock": 8, "scaling": "none", "reciprocal": "resolution", "reg": 0.0, "fallback": True},
                {"name": "hhl_scaled_kk_nc8_fb",       "recon": "statevector", "n_clock": 8, "scaling": "ruiz", "reciprocal": "kappa_known", "reg": 0.0, "fallback": True},
                {"name": "hhl_scaled_kk_nc8_nofb",     "recon": "statevector", "n_clock": 8, "scaling": "ruiz", "reciprocal": "kappa_known", "reg": 0.0, "fallback": False},
                {"name": "hhl_scaled_kk_nc8_aa1000_fb", "recon": "aa_shots", "eff_shots": 1000, "n_clock": 8, "scaling": "ruiz", "reciprocal": "kappa_known", "reg": 0.0, "fallback": True},
                {"name": "hhl_scaled_kk_nc12_nofb",    "recon": "statevector", "n_clock": 12, "scaling": "ruiz", "reciprocal": "kappa_known", "reg": 0.0, "fallback": False},
                {"name": "hhl_scaled_kk_nc12_switch",  "recon": "statevector", "n_clock": 12, "scaling": "ruiz", "reciprocal": "kappa_known", "reg": 0.0, "fallback": True, "switch_policy": True},
                {"name": "hhl_scaled_kk_nc8_blockonly_nofb", "recon": "statevector", "n_clock": 8, "scaling": "ruiz", "reciprocal": "kappa_known", "reg": 0.0, "fallback": False, "criterion": "block", "instances_only": ["tiny_B"]},   # block test alone (ablation)
            ],
            "accept_criterion": "combined", "accept_eta": 0.1, "accept_be_tol": 1e-8, "max_refine": 4, "maxiters": 40,
        },
        # polynomial solver (qubitised walk + LCU), section 15
        "cks": {"poly": "optimal",                       # "optimal" (Sunderhauf et al.) | "cks" (Childs-Kothari-Somma expansion)
                "eps_list": [3e-1, 1e-2, 1e-6, 1e-10], "max_degree": 400000,
                # Q5b grid: circuit precision x accepted samples per refinement round (both measurement settings together)
                "ir_eps_list": [3e-1, 1e-2, 1e-6], "ir_samples_per_round": [200, 500, 1000, 4000, 20000], "ir_rounds": 120, "ir_max_systems": 9,
                # instances whose first KKT matrix already needs more walk steps than max_degree (e.g. dense_noeq_n10_k6) are not used
                "integrated_extra_instances": ["dense_kappa_n10_k0_s0", "dense_kappa_n10_k3_s0", "dense_noeq_n10_k0_s0", "dense_kappa_n10_k0_s1", "dense_kappa_n10_k3_s1"],
                "integrated_be_tol": 1e-9,                 # tau of the combined test in the quantum runs (classical comb methods: 1e-10)
                "integrated_max_refine": 60, "hhl_fair_extra_instances": ["dense_kappa_n10_k0_s0"], "hhl_fair_extra_runs": ["hhl_nc10_ideal_ir60_nofb"], "hhl_fair_maxiters": 20,
                "integrated_runs": [{"name": "lcu_opt_eps1e-10_ideal_nofb", "eps": 1e-10, "recon": "statevector"},                                 # high-precision circuit, ideal readout
                                    {"name": "lcu_opt_eps0.3_ideal_ir_nofb", "eps": 3e-1, "recon": "statevector"},                                  # low-precision circuit + refinement
                                    {"name": "lcu_opt_eps0.3_aa500_ir_nofb", "eps": 3e-1, "recon": "aa_shots", "eff_shots": 250},                   # eps = 0.3, 500 samples per round
                                    {"name": "lcu_cks_eps1e-6_aa2e4_ir_nofb", "eps": 1e-6, "recon": "aa_shots", "eff_shots": 10000, "poly": "cks"}],  # CKS expansion, high precision
                # HHL with the same refinement budget as the polynomial solver
                "hhl_fair_runs": [{"name": "hhl_nc8_ideal_ir60_nofb", "n_clock": 8, "recon": "statevector"}, {"name": "hhl_nc10_ideal_ir60_nofb", "n_clock": 10, "recon": "statevector"},
                                  {"name": "hhl_nc10_aa500_ir60_nofb", "n_clock": 10, "recon": "aa_shots", "eff_shots": 250}],
                # Q5e: finite-shot runs over sampling seeds x accepted samples per refinement round (both settings together)
                "shot_sweep": {"eps": 3e-1, "samples_per_round": [100, 250, 1000, 4000], "seeds": [0, 1, 2, 3, 4], "max_instances": 8},
                # Q5c/Q5d: SPD polynomial on the null-space target Z'SZ instead of the indefinite augmented system
                "oss_runs": [{"name": "lcu_oss_eps0.3_ideal_ir_nofb", "eps": 3e-1, "recon": "statevector"},
                             {"name": "lcu_oss_eps0.3_aa500_ir_nofb", "eps": 3e-1, "recon": "aa_shots", "eff_shots": 250}]},
        "qiskit_crosscheck": True, "qiskit_max_clock": 6, "qiskit_max_qubits": 12,
        "install_qiskit_if_missing": True, "qiskit_pins": "qiskit>=1.2,<3 qiskit-aer>=0.14",
    },

    # SDPLIB (optional, conelp)
    "sdplib": {
        "enabled": True,
        "search_dirs": ["/kaggle/input", "./sdplib", "./data/sdplib"],
        "download_if_missing": True, "download_url": "https://raw.githubusercontent.com/vsdp/SDPLIB/master/data/{name}.dat-s",
        "instances": ["truss1", "truss4", "hinf2", "control1", "truss3", "truss2", "qap5"],   # unmodified conelp fails on hinf1/3/10-15
        "max_kkt_dim_custom": 900, "max_kkt_dim_inexact": 200, "quantum_instances": ["truss1"],
    },
    # Per-method budgets.  A method that exceeds max_run_seconds on an instance is skipped for larger instances of the same
    # family only.  With predictive_budget, it is skipped at the next size when its extrapolated time (power law through its
    # last two sizes, exponent clipped to [1, 3.5]; 2.0 with one point) exceeds budget_margin * max_run_seconds.  A running
    # coneqp call cannot be interrupted safely, so budgets act between runs; partial results are flushed after every run.
    # The LU reference is not run above exact_max_dim (fill-in makes it O(N^2.5)).
    "limits": {"max_dense_kkt_dim": 3000, "max_run_seconds": 180, "exact_max_dim": 5000,
               "predictive_budget": True, "budget_margin": 1.5},
}

if RUN_PROFILE == "smoke":
    CONFIG["families"] = {
        "dense_kappa": {"structure": "dense", "n_list": [10], "m_ratio": 2.0, "p_ratio": 0.2, "kappa_list": [1e0, 1e6], "seeds": [0], "active_frac": 0.3, "density": 1.0},
        "dense_noeq": {"structure": "dense", "n_list": [10], "m_ratio": 2.0, "p_ratio": 0.0, "kappa_list": [1e6], "seeds": [0], "active_frac": 0.3, "density": 1.0},
        "sparse_scale": {"structure": "random_sparse", "n_list": [25], "m_ratio": 2.0, "p_ratio": 0.1, "kappa_list": [1e3], "seeds": [0], "active_frac": 0.3, "density": 0.05},
        "sparse_banded": {"structure": "banded", "n_list": [100, 200], "m_ratio": 2.0, "p_ratio": 0.1, "kappa_list": [1e3], "seeds": [0], "active_frac": 0.3, "density": 0.0, "nnz_per_row": 4, "bandwidth": 3},
        "sparse_lowfill": {"structure": "banded_local", "n_list": [100, 200], "m_ratio": 2.0, "p_ratio": 0.1, "kappa_list": [1e3], "seeds": [0], "active_frac": 0.3, "density": 0.0, "nnz_per_row": 4, "bandwidth": 3},
    }
    CONFIG["inexact"].update({"be_tols": [1e-8, 1e-12], "block_etas": [0.1], "noise_iso": [1e-6], "noise_block": [1e-2]})
    CONFIG["quantum"].update({"n_clock_list": [4, 6], "shots_list": [1000, 10000], "eff_shots_list": [100, 1000], "sampling_repeats": 2, "ir_rounds": 2, "qiskit_max_clock": 4})
    CONFIG["quantum"]["reg_experiment"].update({"deltas": [0.0, 1e-2], "ir_rounds": 3, "quantum_n_clock": [6]})
    CONFIG["quantum"]["integrated"]["instances"] = ["tiny_A", "tiny_B"]
    CONFIG["quantum"]["integrated"]["runs"] = [r for r in CONFIG["quantum"]["integrated"]["runs"] if r["name"] in ("hhl_scaled_kk_nc8_fb", "hhl_scaled_kk_nc8_nofb", "hhl_scaled_kk_nc12_switch")]
    for r in CONFIG["quantum"]["integrated"]["runs"]: r["n_clock"] = 6 if r["n_clock"] == 8 else 8
    CONFIG["quantum"]["integrated"]["maxiters"] = 20
    CONFIG["quantum"]["cks"].update({"eps_list": [3e-1, 1e-6], "ir_eps_list": [3e-1, 1e-6], "ir_samples_per_round": [500, 4000], "ir_rounds": 60, "ir_max_systems": 3, "max_degree": 30000, "integrated_extra_instances": [], "hhl_fair_extra_instances": [], "integrated_max_refine": 60})
    CONFIG["quantum"]["cks"]["shot_sweep"].update({"samples_per_round": [250, 1000], "seeds": [0, 1], "max_instances": 2})
    CONFIG["families"]["sparse_banded"]["seeds"] = [0, 1]; CONFIG["families"]["sparse_lowfill"]["seeds"] = [0]
    CONFIG["maros_meszaros"]["instances"] = ["QAFIRO", "CVXQP1_S"]; CONFIG["maros_meszaros"]["methods"] = ["cvxopt_default", "custom_exact_reduced", "minres_fused_comb_cpu", "minres_fused_comb_switch_cpu", "minres_fused_comb_gpu"]
    CONFIG["tolerance_sweep"].update({"levels": [1e-4, 1e-8], "noise_iso": [1e-7, 1e-5], "noise_block": [1e-2], "max_instances": 4})
    CONFIG["quantum"]["cks"]["hhl_fair_runs"] = [{"name": "hhl_nc8_ideal_ir60_nofb", "n_clock": 8, "recon": "statevector"}]
    CONFIG["capture"]["instances"] = ["tiny_A", "tiny_B", "tiny_C", "dense_kappa_n10_k6_s0"]
    CONFIG["sdplib"]["instances"] = ["truss1", "hinf2"]
elif RUN_PROFILE == "medium":
    CONFIG["families"]["dense_kappa"]["n_list"] = [10, 20, 40, 80]; CONFIG["families"]["dense_kappa"]["seeds"] = [0, 1, 2, 3, 4]
    CONFIG["families"]["sparse_banded"].update({"n_list": [100, 200, 400, 800, 1600, 3200, 6400], "seeds": [0, 1, 2, 3, 4], "seeds_large": [0, 1], "seeds_large_from_n": 6400})
    CONFIG["families"]["sparse_lowfill"].update({"n_list": [100, 200, 400, 800, 1600, 3200, 6400], "seeds": [0, 1, 2], "seeds_large": [0], "seeds_large_from_n": 6400})
    CONFIG["maros_meszaros"]["instances"] += ["DUAL2", "CVXQP2_S", "CVXQP3_S", "QSHARE1B", "VALUES", "QBRANDY", "QBEACONF", "PRIMAL2", "QE226", "QBORE3D", "QGROW7", "QSCORPIO", "QBANDM", "QPCSTAIR",
                                              "QPCBOEI1", "QSCAGR25", "PRIMAL4", "QSCSD1", "QETAMACR", "GOULDQP3", "QSTANDAT", "QSCSD6", "LASER", "QSEBA", "CVXQP2_M", "QSHIP04S", "CVXQP1_M", "CVXQP3_M",
                                              "MOSARQP2", "QSHELL", "Q25FV47", "QSCTAP2", "MOSARQP1", "QSCTAP3", "CONT-050"]      # 48 instances, KKT dimension up to 10 192
    CONFIG["maros_meszaros"]["budget_s"] = 420      # MINRES needs up to ~350 s on QSHELL and Q25FV47; a lower budget would skip larger instances
    CONFIG["repeats"] = 2; CONFIG["inexact"]["full_ablation"] = True
    CONFIG["quantum"].update({"n_clock_list": [4, 6, 8, 10], "shots_list": [1000, 10000, 100000, 1000000], "eff_shots_list": [100, 1000, 10000, 100000], "max_qubits": 17})
    CONFIG["quantum"]["reg_experiment"].update({"quantum_n_clock": [6, 8, 10]})
    CONFIG["quantum"]["integrated"]["instances"] = ["tiny_A", "tiny_B", "tiny_C"]
    CONFIG["quantum"]["integrated"]["runs"] += [{"name": "hhl_scaled_kk_nc10_nofb", "recon": "statevector", "n_clock": 10, "scaling": "ruiz", "reciprocal": "kappa_known", "reg": 0.0, "fallback": False}]
    CONFIG["quantum"]["integrated"]["maxiters"] = 60
    CONFIG["sdplib"]["instances"] += ["control2", "hinf1", "theta1"]; CONFIG["sdplib"]["max_kkt_dim_inexact"] = 350
    CONFIG["cond_limit"] = 1200; CONFIG["limits"]["max_run_seconds"] = 300; CONFIG["limits"]["exact_max_dim"] = 5000
elif RUN_PROFILE == "large":
    CONFIG["families"]["dense_kappa"]["n_list"] = [10, 20, 40, 80, 160]; CONFIG["families"]["dense_kappa"]["seeds"] = [0, 1, 2, 3, 4]
    CONFIG["families"]["sparse_banded"].update({"n_list": [100, 200, 400, 800, 1600, 3200, 6400, 12800], "seeds": [0, 1, 2, 3, 4], "seeds_large": [0, 1, 2], "seeds_large_from_n": 6400})
    CONFIG["families"]["sparse_lowfill"].update({"n_list": [100, 200, 400, 800, 1600, 3200, 6400, 12800], "seeds": [0, 1, 2], "seeds_large": [0], "seeds_large_from_n": 6400})
    CONFIG["maros_meszaros"]["instances"] += ["DUAL2", "CVXQP2_S", "CVXQP3_S", "QSHARE1B", "VALUES", "QBRANDY", "QBEACONF", "PRIMAL2", "QE226", "QBORE3D", "QGROW7", "QSCORPIO", "QBANDM", "QPCSTAIR",
                                              "QPCBOEI1", "QSCAGR25", "PRIMAL4", "QSCSD1", "QETAMACR", "GOULDQP3", "QSTANDAT", "QSCSD6", "LASER", "QSEBA", "CVXQP2_M", "QSHIP04S", "CVXQP1_M", "CVXQP3_M",
                                              "MOSARQP2", "QSHELL", "Q25FV47", "QSCTAP2", "MOSARQP1", "QSCTAP3", "CONT-050", "QSHIP08S", "AUG3DCQP", "STCQP2", "QSHIP08L"]
    CONFIG["maros_meszaros"].update({"budget_s": 600, "max_kkt_dim": 20000})
    CONFIG["repeats"] = 3; CONFIG["inexact"]["full_ablation"] = True
    CONFIG["quantum"].update({"n_clock_list": [4, 6, 8, 10, 12], "shots_list": [1000, 10000, 100000, 1000000], "eff_shots_list": [100, 1000, 10000, 100000], "max_qubits": 18})
    CONFIG["quantum"]["integrated"]["instances"] = ["tiny_A", "tiny_B", "tiny_C"]
    CONFIG["quantum"]["integrated"]["runs"] += [{"name": "hhl_scaled_kk_nc10_nofb", "recon": "statevector", "n_clock": 10, "scaling": "ruiz", "reciprocal": "kappa_known", "reg": 0.0, "fallback": False},
                                                {"name": "hhl_scaled_kk_nc12_nofb", "recon": "statevector", "n_clock": 12, "scaling": "ruiz", "reciprocal": "kappa_known", "reg": 0.0, "fallback": False}]
    CONFIG["quantum"]["reg_experiment"].update({"quantum_n_clock": [6, 8, 10, 12], "classical_max_dim": 1024})
    CONFIG["quantum"]["integrated"]["maxiters"] = 60
    CONFIG["sdplib"]["instances"] += ["control2", "control3", "hinf1", "hinf3", "theta1"]; CONFIG["sdplib"]["max_kkt_dim_inexact"] = 500
    CONFIG["cond_limit"] = 2000; CONFIG["limits"]["max_run_seconds"] = 900; CONFIG["limits"]["exact_max_dim"] = 10000

if CONFIG["device"] == "gpu" and RUN_PROFILE != "smoke":
    # GPU build: extend the sparse families to sizes where a GPU can matter (KKT dimension up to ~79 000)
    CONFIG["families"]["sparse_banded"]["n_list"] = sorted(set(CONFIG["families"]["sparse_banded"]["n_list"]) | {6400, 12800, 25600})
    CONFIG["families"]["sparse_lowfill"]["n_list"] = sorted(set(CONFIG["families"]["sparse_lowfill"]["n_list"]) | {6400, 12800, 25600})
    CONFIG["limits"]["max_run_seconds"] = max(CONFIG["limits"]["max_run_seconds"], 300)
    CONFIG["quantum"]["max_qubits"] = max(CONFIG["quantum"]["max_qubits"], 20); CONFIG["quantum"]["max_state_bytes"] = 512 * 2**20
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:10]
CONFIG["config_hash"] = CONFIG_HASH
RESULTS_DIR = os.path.join(CONFIG["output_root"], "results")
PLOTS_DIR = os.path.join(RESULTS_DIR, "plots")
PARTIAL_DIR = os.path.join(RESULTS_DIR, "partial")
NPZ_DIR = os.path.join(RESULTS_DIR, "instances_npz")
for d in (RESULTS_DIR, PLOTS_DIR, PARTIAL_DIR, NPZ_DIR):
    os.makedirs(d, exist_ok=True)
NOTEBOOK_T0 = time.time()
with open(os.path.join(RESULTS_DIR, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"version={CODE_VERSION} profile={RUN_PROFILE} device={CONFIG['device']} config_hash={CONFIG_HASH} output_root={CONFIG['output_root']}")

## 2. Environment

* Required: `numpy`, `scipy`, `pandas`, `matplotlib` and `cvxopt` (1.3). Kaggle images ship all of them; a missing package is installed with `pip` before `numpy` is imported, so no kernel restart is needed. Installed packages are never upgraded.
* Optional: `qiskit` and `qiskit-aer`, used only to cross-check the statevector simulator and to obtain transpiled gate counts. If Qiskit cannot be installed, the cross-check is reported as `skipped`.
* GPU: with `device = "gpu"` the MINRES kernels and the simulator run on CuPy. The cell checks that a CUDA device is usable; if not, every GPU method is reported as `skipped` and the CPU methods run unchanged. Setting `QKKT_FAKE_GPU=1` replaces CuPy by NumPy to test the GPU code paths on a machine without a GPU (no GPU timing is meaningful in that mode).

Package versions, hardware, thread settings and the seed are written to `results/environment.json`.

In [ ]:
# Environment: install missing packages before numpy is imported, record versions and hardware.
import importlib, subprocess, sys, os, json, time, platform

ENV = {"python": sys.version, "platform": platform.platform(), "notebook_started": time.strftime("%Y-%m-%d %H:%M:%S"), "code_version": CODE_VERSION,
       "install_log": [], "kaggle": os.path.isdir("/kaggle/working"), "run_profile": CONFIG["run_profile"]}

def _have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

def _pip_install(spec):
    """Install missing packages only (never upgrades what is present). Returns (ok, message)."""
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check"] + spec.split()
    t0 = time.time()
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
        ok = (r.returncode == 0)
        msg = (r.stdout + r.stderr)[-2000:]
    except Exception as e:
        ok, msg = False, repr(e)
    ENV["install_log"].append({"spec": spec, "ok": ok, "seconds": round(time.time() - t0, 1), "tail": msg[-500:]})
    return ok, msg

# --- required packages ---
REQUIRED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "matplotlib": "matplotlib", "cvxopt": "cvxopt>=1.3,<1.4"}
for mod, spec in REQUIRED.items():
    if not _have(mod):
        print(f"[env] {mod} missing -> pip install {spec}  (requires Internet)")
        ok, msg = _pip_install(spec)
        print("      ->", "installed" if ok else "FAILED: " + msg[-300:])
missing = [m for m in REQUIRED if not _have(m)]
if missing:
    raise RuntimeError(f"Required packages could not be imported/installed: {missing}. "
                       "On Kaggle: Settings -> Internet -> ON, then re-run this cell.")

# --- optional Qiskit (cross-check only) ---
QISKIT_STATUS = {"available": False, "reason": "", "version": None, "aer": False}
if CONFIG["quantum"]["qiskit_crosscheck"]:
    if not _have("qiskit") and CONFIG["quantum"]["install_qiskit_if_missing"]:
        print("[env] qiskit missing -> optional pip install (Internet needed; failure is non-fatal)")
        ok, msg = _pip_install(CONFIG["quantum"]["qiskit_pins"])
        print("      ->", "installed" if ok else "not installed (" + msg[-200:].strip() + ")")
    try:
        import qiskit
        QISKIT_STATUS.update({"available": True, "version": qiskit.__version__})
        try:
            import qiskit_aer; QISKIT_STATUS["aer"] = True; QISKIT_STATUS["aer_version"] = qiskit_aer.__version__
        except Exception as e:
            QISKIT_STATUS["aer"] = False; QISKIT_STATUS["aer_reason"] = repr(e)[:200]
    except Exception as e:
        QISKIT_STATUS.update({"available": False, "reason": f"import failed: {e!r}"[:300]})
else:
    QISKIT_STATUS["reason"] = "disabled in CONFIG"

# --- imports used everywhere ---
import numpy as np, scipy, scipy.linalg as sla, scipy.sparse as sp, scipy.sparse.linalg as spla
import pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cvxopt
from cvxopt import matrix, spmatrix, solvers, misc
import warnings, io, contextlib, re, math, hashlib, shutil, zipfile, traceback, inspect
warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- compute device: CuPy if requested and usable, otherwise NumPy ---
GPU_STATUS = {"requested": CONFIG["device"], "available": False, "reason": "", "name": None}
DEVICE_XP = np; DEVICE_SPARSE = None
if CONFIG["device"] == "gpu":
    if os.environ.get("QKKT_FAKE_GPU") == "1":            # NumPy stand-in to test the GPU code paths without a GPU
        class _FakeSparse:
            csr_matrix = staticmethod(lambda a, shape=None: sp.csr_matrix(a, shape=shape))
        GPU_STATUS.update({"available": True, "name": "FAKE (numpy stand-in, logic test only)"}); DEVICE_SPARSE = _FakeSparse
    else:
        try:
            try:
                import cupy
            except ImportError:                                # Kaggle GPU images ship CuPy; otherwise try one install
                if shutil.which("nvidia-smi"):
                    print("[env] cupy missing -> pip install cupy-cuda12x"); print("   ", _pip_install("cupy-cuda12x"))
            import cupy, cupyx.scipy.sparse as _cusp
            if cupy.cuda.runtime.getDeviceCount() < 1: raise RuntimeError("no CUDA device")
            _t = cupy.arange(8, dtype=float); assert float((_t @ _t)) == 140.0
            DEVICE_XP = cupy; DEVICE_SPARSE = _cusp
            GPU_STATUS.update({"available": True, "name": cupy.cuda.runtime.getDeviceProperties(0)["name"].decode(), "cupy": cupy.__version__,
                               "memory_gb": round(cupy.cuda.runtime.memGetInfo()[1] / 2**30, 2)})
        except Exception as e:
            GPU_STATUS["reason"] = f"{type(e).__name__}: {e}"[:300]
    if not GPU_STATUS["available"]:
        print("[env] GPU requested but not usable ->", GPU_STATUS["reason"], "-> every GPU method is reported as skipped; CPU methods run normally")
GPU_OK = GPU_STATUS["available"]
CONFIG["quantum"]["device"] = "gpu" if (GPU_OK and DEVICE_XP is not np) else "cpu"
def to_device(a):
    """numpy / scipy.sparse -> device array (cupy / cupyx sparse); identity on CPU."""
    if not GPU_OK: return a
    if sp.issparse(a): return DEVICE_SPARSE.csr_matrix(a.tocsr())
    return DEVICE_XP.asarray(a)
def to_host(a):
    """cupy -> numpy (no-op for numpy)."""
    return a.get() if hasattr(a, "get") else np.asarray(a)
def dev_sync():
    if GPU_OK and DEVICE_XP is not np: DEVICE_XP.cuda.Stream.null.synchronize()

# --- hardware ---
def _hardware():
    hw = {"cpu_count": os.cpu_count(), "machine": platform.machine(), "processor": platform.processor()}
    try:
        import psutil
        hw["memory_total_gb"] = round(psutil.virtual_memory().total / 2**30, 2)
        hw["memory_available_gb"] = round(psutil.virtual_memory().available / 2**30, 2)
    except Exception:
        try:
            with open("/proc/meminfo") as f:
                for line in f:
                    if line.startswith("MemTotal"):
                        hw["memory_total_gb"] = round(int(line.split()[1]) / 2**20, 2)
        except Exception:
            pass
    try:
        with open("/proc/cpuinfo") as f:
            for line in f:
                if "model name" in line:
                    hw["cpu_model"] = line.split(":", 1)[1].strip(); break
    except Exception:
        pass
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
        hw["gpu_present"] = r.stdout.strip() if r.returncode == 0 else "none detected"
    except Exception:
        hw["gpu_present"] = "none detected"
    return hw

ENV["hardware"] = _hardware()
ENV["packages"] = {"numpy": np.__version__, "scipy": scipy.__version__, "pandas": pd.__version__,
                   "matplotlib": matplotlib.__version__, "cvxopt": cvxopt.__version__,
                   "qiskit": QISKIT_STATUS.get("version"), "qiskit_aer": QISKIT_STATUS.get("aer_version")}
ENV["qiskit_status"] = QISKIT_STATUS; ENV["gpu_status"] = GPU_STATUS
ENV["seed"] = CONFIG["global_seed"]
ENV["blas"] = {k: v for k, v in np.__config__.CONFIG.get("Build Dependencies", {}).items()} if hasattr(np.__config__, "CONFIG") else {}
# thread count used by BLAS (affects timing reproducibility)
ENV["thread_env"] = {k: os.environ.get(k) for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS")}
with open(os.path.join(RESULTS_DIR, "environment.json"), "w") as f:
    json.dump(ENV, f, indent=2, default=str)

print(json.dumps({k: ENV[k] for k in ("python", "platform", "hardware", "packages", "qiskit_status", "gpu_status", "seed")}, indent=1, default=str))
print("Internet needed for installation:", "NO (everything already available)" if not ENV["install_log"] else "YES (see install_log above)")

## 3. Formulation

### 3.1 Problem and optimality conditions

`cvxopt.solvers.coneqp` solves, for the linear cone used here,

$$
\min_x\; \tfrac12 x^\top P x + q^\top x \quad\text{s.t.}\quad Gx + s = h,\; Ax = b,\; s \ge 0,
\qquad P = P^\top \succeq 0,\; x\in\mathbb R^n,\; s\in\mathbb R^m,\; b\in\mathbb R^p .
$$

With multipliers $z \ge 0$ and $y$ the KKT conditions are

$$
P x + q + G^\top z + A^\top y = 0,\qquad Ax = b,\qquad Gx + s = h,\qquad s\ge 0,\; z \ge 0,\qquad s_i z_i = 0 .
$$

CVXOPT's primal-dual path-following method (Vandenberghe 2010) uses a Mehrotra-type predictor-corrector and the Nesterov-Todd scaling $W$; for the nonnegative orthant $W = \mathrm{diag}(d)$ with $d_i = \sqrt{s_i/z_i}$. CVXOPT passes the scaling as the dictionary `W` (`W['d']`, `W['di'] = 1/d`, and `v`, `beta`, `r`, `rti` for second-order and semidefinite cones).

### 3.2 The KKT system replaced by the callback

Each interior-point iteration computes an affine and a combined direction, each with one solve of

$$
\begin{bmatrix} P & A^\top & G^\top \\ A & 0 & 0 \\ G & 0 & -W^\top W \end{bmatrix}
\begin{bmatrix} u_x \\ u_y \\ u_z \end{bmatrix} =
\begin{bmatrix} b_x \\ b_y \\ b_z \end{bmatrix},
$$

plus one solve at initialisation with $W = I$. The callback contract (CVXOPT user guide, *Exploiting structure*, and `misc.kkt_ldl`) is: `f = kktsolver(W)` returns a function `f(x, y, z)` that receives $b_x, b_y, b_z$ in `x, y, z` and overwrites them with $u_x$, $u_y$ and $W u_z$. With $w = W u_z$ and the third block row multiplied by $W^{-\top}$, this is the symmetric system that CVXOPT's own `'ldl'` solver factors and that every method in this notebook forms, measures and solves:

$$
\tilde K \tilde u = \tilde r,\qquad
\tilde K = \begin{bmatrix} P & A^\top & G^\top W^{-1} \\ A & 0 & 0 \\ W^{-\top} G & 0 & -I \end{bmatrix},\quad
\tilde u = \begin{bmatrix} u_x \\ u_y \\ w \end{bmatrix},\quad
\tilde r = \begin{bmatrix} b_x \\ b_y \\ W^{-\top} b_z \end{bmatrix}.
$$

$\tilde K$ is symmetric indefinite of dimension $N = n + p + m$. Eliminating $w$ gives the reduced system with $S = P + G^\top W^{-1}W^{-\top}G$, which is positive definite and is what CVXOPT's default `'chol2'` solver factors. Neither $\kappa(\tilde K)$ nor $\kappa(S)$ is controlled by $\kappa(P)$ alone, because $d_i \to 0$ for active and $d_i \to \infty$ for inactive constraints as $\mu \to 0$ (paper, Section 3).

CVXOPT's `refinement` option wraps any `kktsolver` in an outer refinement loop. It is fixed for every method (0 for QPs, the CVXOPT default for cone LPs), so that an inexact solver is never corrected silently; the refinement and fallback of the solvers below are logged separately.

### 3.3 Stopping conditions

With `abstol` $=10^{-7}$, `reltol` $=10^{-6}$ and `feastol` $=10^{-7}$, CVXOPT returns `status = 'optimal'` when

$$
\text{pres} = \max\!\Big(\tfrac{\|Ax-b\|_2}{\max(1,\|b\|_2)},\, \tfrac{\|Gx+s-h\|_2}{\max(1,\|h\|_2)}\Big)\le \texttt{feastol},\qquad
\text{dres} = \tfrac{\|Px+q+G^\top z+A^\top y\|_2}{\max(1,\|q\|_2)} \le \texttt{feastol},
$$

and either $s^\top z \le \texttt{abstol}$ or the relative gap is at most `reltol`; otherwise it stops with `'unknown'` after `maxiters`. The runner recomputes all these quantities from the returned $(x, y, s, z)$, together with a valid duality gap and the objective error against the known optimum.

### 3.4 Acceptance tests for an inexact KKT solve

Let $e = \tilde K\tilde u - \tilde r$ be the residual of a computed direction and $\tau$ the tolerance of the method. The notebook implements five tests, all evaluated on the original (unscaled, unregularised) system. The relative-residual and backward-error tests accept at $10\,\tau$ (`accept_factor` in the configuration); tolerances in method names are the values of $\tau$.

* `relres`: $\|e\|/\|\tilde r\| \le 10\,\tau$. In floating point the smallest attainable residual is of order $u\,(\|\tilde K\|\|\tilde u\| + \|\tilde r\|)$ with unit round-off $u$. Late on the central path $\|\tilde K\|\|\tilde u\| \gg \|\tilde r\|$, so this test can be out of reach even for an LU solve (figure `04_conditioning_across_iterations.png`), and it does not distinguish the blocks of the right-hand side.
* `relres_floor`: `relres`, or accept when the backward error is already at the level of a direct solve ($10^{-14}$).
* `backward`: the normwise backward error $\omega = \|e\| / (\|\tilde K\|_2\|\tilde u\| + \|\tilde r\|) \le 10\,\tau$ (Higham, *Accuracy and Stability of Numerical Algorithms*, Thm. 7.1). A direction with backward error $\omega$ solves a system whose matrix and right-hand side are perturbed by at most $\omega$ in relative norm. $\|\tilde K\|_2$ is estimated once per factorisation by 20 power iterations.
* `block`: block-wise forcing $\|e_b\| \le \eta \max(\|\tilde r_b\|, \epsilon_{\text{abs}}\rho_b)$ for $b \in \{x, y, z\}$ with $\rho_x = \max(1,\|q\|)$, $\rho_y = \max(1,\|b\|)$, $\rho_z = \max(1,\|h\|)$ and $\epsilon_{\text{abs}} = 10^{-9}$. The $x$ and $y$ blocks of $\tilde r$ are the dual and primal residuals that CVXOPT drives to zero; the $z$ block scales with $\sqrt{s^\top z}$, so the test tightens along the path.
* `combined`: `block` with $\eta = 0.1$ and `backward` together, i.e. $\max(\beta/\eta,\ \omega/(10\,\tau)) \le 1$. This is the test of the paper (Section 4.1, Eq. (5)); $\tau = 10^{-10}$ for the classical `*_comb*` methods and $10^{-9}$ for the quantum solvers, so $\omega \le 10^{-9}$ and $\omega \le 10^{-8}$.

For any computed $\tilde u$, $\|\delta u\|/\|\tilde u\| \le \kappa(\tilde K)\,\|e\|/(\|\tilde K\|\|\tilde u\|)$. A backward error $\omega$ therefore bounds the forward error by about $\kappa\,\omega$; a relative residual $\rho$ bounds it by $\kappa\rho\,\|\tilde r\|/(\|\tilde K\|\|\tilde u\|)$; the block test bounds the block residuals and gives no forward-error bound by itself.

The tests are evaluated every `check_every` Krylov iterations and after the solve. The protocol around them (refinement, fallback, accounting) is implemented in `KKTSolveEngine.solve`, Section 8.

### 3.5 Regularised target for the quantum solvers

$\tilde K_\delta = \tilde K + \delta J$ with $J = \mathrm{diag}(I_n, -I_{p+m})$, applied to the Ruiz-scaled system, keeps the quasi-definite structure and bounds $\kappa(\tilde K_\delta) \lesssim (1+\delta)/\delta$. The direction is refined on the true system, so the error introduced by $\delta$ is measured (Section 17).

## 4. Utilities

Seeded random streams (`rng_for` hashes the global seed with the instance keys, so every instance is reproducible on its own), timers, conversions between CVXOPT and NumPy, and the result tables. Rows are flushed to `results/partial/` every 20 s, so a session that stops early still leaves usable CSV files. `PeakMemory` samples the process RSS (and the CuPy pool) during each `coneqp` call.

In [ ]:
# Utilities: seeded RNG, timing, CVXOPT/NumPy conversion, result tables with periodic flushing to results/partial/.
GLOBAL_SEED = CONFIG["global_seed"]

def rng_for(*keys):
    """Deterministic RNG derived from the global seed and any hashable keys."""
    h = hashlib.sha256(("|".join(map(str, keys))).encode()).digest()
    return np.random.default_rng([GLOBAL_SEED, int.from_bytes(h[:4], "little")])

class Timer:
    def __init__(self): self.t = 0.0
    def __enter__(self): self._t0 = time.perf_counter(); return self
    def __exit__(self, *a): self.t += time.perf_counter() - self._t0

def to_np(m):
    """cvxopt matrix -> numpy (column vector -> 1-D)."""
    a = np.array(m, dtype=float)
    return a.ravel() if a.ndim == 2 and a.shape[1] == 1 else a

def to_cvx(a):
    a = np.asarray(a, dtype=float)
    if a.ndim == 1: a = a.reshape(-1, 1)
    return matrix(np.ascontiguousarray(a), tc="d")

def to_cvx_sparse(a):
    c = sp.coo_matrix(a)
    return spmatrix(c.data.astype(float).tolist(), c.row.astype(int).tolist(), c.col.astype(int).tolist(), c.shape, tc="d")

def cvx_matrix_from(a, sparse):
    return to_cvx_sparse(a) if sparse else to_cvx(np.asarray(a.todense()) if sp.issparse(a) else a)

def write_back(cvx_vec, values):
    """Overwrite a cvxopt column matrix in place with numpy values."""
    v = np.ascontiguousarray(np.asarray(values, dtype=float).reshape(-1, 1))
    cvx_vec[:] = matrix(v, tc="d")

def relnorm(a, b):
    nb = np.linalg.norm(b)
    return float(np.linalg.norm(a) / nb) if nb > 0 else float(np.linalg.norm(a))

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return float("nan")

class Records:
    """All exported tables live here; every append can be flushed to results/partial/*.csv."""
    TABLES = ["benchmark_results", "iteration_logs", "linear_solve_logs", "quantum_results", "failures", "experiment_status"]
    def __init__(self):
        for t in self.TABLES: setattr(self, t, [])
        self._last_flush = time.time()
    def add(self, table, rec):
        getattr(self, table).append(rec)
    def status(self, experiment, status, reason="", **extra):
        rec = {"experiment": experiment, "status": status, "reason": reason, "timestamp": time.strftime("%H:%M:%S")}
        rec.update(extra)
        # replace an earlier status line for the same experiment
        self.experiment_status = [r for r in self.experiment_status if r["experiment"] != experiment] + [rec]
        print(f"[status] {experiment}: {status}" + (f" ({reason})" if reason else ""))
    def failure(self, run_id, stage, reason, **extra):
        rec = {"run_id": run_id, "stage": stage, "reason": str(reason)[:500]}
        rec.update(extra); self.failures.append(rec)
    def df(self, table):
        return pd.DataFrame(getattr(self, table))
    def flush(self, final=False, min_interval=20.0):
        if not final and time.time() - self._last_flush < min_interval: return
        self._last_flush = time.time()
        outdir = RESULTS_DIR if final else PARTIAL_DIR
        for t in self.TABLES:
            rows = getattr(self, t)
            if rows or final:
                pd.DataFrame(rows).to_csv(os.path.join(outdir, f"{t}.csv"), index=False)

REC = Records()

import threading
try:
    import psutil as _psutil
except Exception:
    _psutil = None
class PeakMemory:
    """Samples the resident set size of the process (and the CuPy pool when a GPU is used) from a daemon thread during a coneqp call.
    The RSS is process-wide and memory is not returned to the OS between runs, so the increase over the start value is what the
    method needed on top of what was already held."""
    def __init__(self, interval=None):
        c = CONFIG.get("memory_sampling", {}); self.interval = float(c.get("interval_s", 0.05) if interval is None else interval); self.enabled = bool(c.get("enabled", True)) and _psutil is not None
        self.proc = _psutil.Process() if self.enabled else None; self.pool = None
        if self.enabled and GPU_OK and DEVICE_XP is not np:
            try: self.pool = DEVICE_XP.get_default_memory_pool()
            except Exception: self.pool = None
        self.rss0 = self.rss_peak = self.gpu0 = self.gpu_peak = float("nan"); self.n = 0; self._stop = threading.Event(); self._thr = None
    def _sample(self):
        try:
            r = float(self.proc.memory_info().rss); self.rss_peak = max(self.rss_peak, r) if np.isfinite(self.rss_peak) else r
        except Exception: pass
        if self.pool is not None:
            try:
                g = float(self.pool.used_bytes()); self.gpu_peak = max(self.gpu_peak, g) if np.isfinite(self.gpu_peak) else g
            except Exception: self.pool = None
        self.n += 1
    def _loop(self):
        while not self._stop.wait(self.interval): self._sample()
    def __enter__(self):
        if not self.enabled: return self
        self._sample(); self.rss0 = self.rss_peak; self.gpu0 = self.gpu_peak
        self._thr = threading.Thread(target=self._loop, daemon=True); self._thr.start(); return self
    def __exit__(self, *a):
        if not self.enabled: return
        self._stop.set(); self._thr.join(timeout=2.0); self._sample()
    def fields(self):
        mb = 1.0 / 2 ** 20
        return {"mem_rss_start_mb": self.rss0 * mb, "mem_rss_peak_mb": self.rss_peak * mb, "mem_rss_increase_mb": max(0.0, (self.rss_peak - self.rss0) * mb) if np.isfinite(self.rss_peak) else float("nan"),
                "mem_gpu_pool_start_mb": self.gpu0 * mb, "mem_gpu_pool_peak_mb": self.gpu_peak * mb, "mem_gpu_pool_increase_mb": max(0.0, (self.gpu_peak - self.gpu0) * mb) if np.isfinite(self.gpu_peak) else float("nan"),
                "mem_samples": self.n}
print("utilities ready; partial results dir:", PARTIAL_DIR, "| memory sampling:", "psutil" if _psutil is not None else "unavailable (psutil missing)")

## 5. Synthetic QP instances

For given $(n, m, p)$, a target condition number $\kappa_P$, a density and a seed:

1. $P = Q\,\mathrm{diag}(\lambda)\,Q^\top$ with $\lambda$ log-spaced in $[1, \kappa_P]$ for the dense families, so $\kappa_2(P) = \kappa_P$. The sparse families use $P = \mathrm{diag}(\lambda) + E$ with a sparse symmetric $E$ whose Gershgorin radius is at most $0.1\,\lambda_{\min}$; then $P \succ 0$ and $\kappa_2(P)$ is within a factor 1.3 of $\kappa_P$. The measured $\kappa_2(P)$ is stored.
2. $G$ ($m\times n$) and $A$ ($p \times n$, full row rank, checked) are random dense or sparse matrices.
3. $x^\star$ is random. An active set $\mathcal S$ with $|\mathcal S| = \lceil \text{active\_frac}\cdot m\rceil \le n - p - 1$ gets $s^\star_i = 0,\; z^\star_i \in [0.5, 2]$, the other rows $s^\star_i \in [0.5, 2],\; z^\star_i = 0$ (strict complementarity); $y^\star$ is random.
4. $q = -(Px^\star + G^\top z^\star + A^\top y^\star)$, $h = Gx^\star + s^\star$, $b = Ax^\star$. Then $(x^\star, y^\star, z^\star, s^\star)$ satisfies the KKT conditions exactly and $f^\star = \tfrac12 x^{\star\top} P x^\star + q^\top x^\star$ is the optimal value.

| family | structure | role |
|---|---|---|
| `dense_kappa`, `dense_noeq` | dense, $\kappa_P \in \{1, 10^3, 10^6\}$ | acceptance-test ablation, precision of a direction |
| `sparse_scale` | 5 % random density | small sizes only: $G^\top G$ fills in completely |
| `sparse_banded` | banded $P$, 4 nonzeros per row of $G$ and $A$ in random columns | high-fill scaling family |
| `sparse_lowfill` | as above, nonzeros within a window of 13 columns | low-fill control: $P + G^\top W^{-2} G$ is banded and sparse Cholesky is $O(n)$ |

The three tiny instances `tiny_A` ($n=2$, $m=2$), `tiny_B` ($n=3$, $m=4$, $p=1$) and `tiny_C` ($n=4$, $m=4$) have KKT dimension 4, 8 and 8; their KKT systems are the ones passed to the quantum solvers.

In [ ]:
# Synthetic QP families with a known primal-dual solution (x*, y*, z*, s*) and prescribed kappa(P).
def _sparse_random(rng, rows, cols, density):
    nnz = max(rows, int(round(density * rows * cols)))
    I = np.concatenate([np.arange(rows), rng.integers(0, rows, nnz - rows)])      # every row gets >= 1 entry
    J = np.concatenate([rng.integers(0, cols, rows), rng.integers(0, cols, nnz - rows)])
    V = rng.standard_normal(nnz)
    M = sp.coo_matrix((V, (I, J)), shape=(rows, cols)).tocsr()
    M.sum_duplicates()
    return M

def _banded_rows(rng, rows, cols, nnz_per_row):
    """Sparse matrix with exactly nnz_per_row random nonzeros per row (values ~ N(0,1)/sqrt(nnz))."""
    k = min(nnz_per_row, cols)
    J = np.concatenate([rng.choice(cols, k, replace=False) for _ in range(rows)])
    I = np.repeat(np.arange(rows), k); V = rng.standard_normal(rows * k) / np.sqrt(k)
    return sp.coo_matrix((V, (I, J)), shape=(rows, cols)).tocsr()

def _local_rows(rng, rows, cols, nnz_per_row, halfwidth):
    """Sparse matrix whose row i has nnz_per_row nonzeros inside a window of half-width `halfwidth` around column i*cols/rows.
    G'G and A'A are then banded and the Cholesky factor of P + G'W^-2 G has O(n) nonzeros: the low-fill control case in which
    a sparse direct solver should win."""
    k = min(nnz_per_row, cols, 2 * halfwidth + 1); J = np.empty(rows * k, dtype=np.int64)
    for i in range(rows):
        c = int(round(i * (cols - 1) / max(rows - 1, 1))); lo = max(0, min(c - halfwidth, cols - (2 * halfwidth + 1))); hi = min(cols, lo + 2 * halfwidth + 1)
        J[i * k:(i + 1) * k] = lo + rng.choice(hi - lo, k, replace=False)
    I = np.repeat(np.arange(rows), k); V = rng.standard_normal(rows * k) / np.sqrt(k)
    return sp.coo_matrix((V, (I, J)), shape=(rows, cols)).tocsr()

def make_qp_instance(inst_id, family, n, m, p, kappa_P, density, active_frac, seed, structure=None, nnz_per_row=4, bandwidth=3):
    rng = rng_for(family, inst_id, n, m, p, kappa_P, density, seed)
    sparse = density < 0.5
    structure = structure or ("dense" if not sparse else "random_sparse")
    lam = np.logspace(0, np.log10(kappa_P), n) if n > 1 else np.array([1.0])
    if structure == "banded":
        # P = diag(lam) + symmetric band of half-width `bandwidth`, Gershgorin radius <= 0.1*lam_min  => P > 0 and kappa within ~1.2x of kappa_P
        rng_b = rng_for(family, inst_id, "banded", n, m, p, kappa_P, seed)
        diags = [lam]; offs = [0]
        for k in range(1, bandwidth + 1):
            v = rng_b.uniform(-1, 1, n - k) * (0.1 * lam.min() / (2 * bandwidth)); diags += [v, v]; offs += [k, -k]
        P = sp.diags(diags, offs, format="csr")
        G = _banded_rows(rng_b, m, n, nnz_per_row)
        A = _banded_rows(rng_b, p, n, nnz_per_row) if p > 0 else None
        sparse = True
    elif structure == "banded_local":
        rng_b = rng_for(family, inst_id, "banded_local", n, m, p, kappa_P, seed)
        diags = [lam]; offs = [0]
        for k in range(1, bandwidth + 1):
            v = rng_b.uniform(-1, 1, n - k) * (0.1 * lam.min() / (2 * bandwidth)); diags += [v, v]; offs += [k, -k]
        P = sp.diags(diags, offs, format="csr")
        G = _local_rows(rng_b, m, n, nnz_per_row, 2 * bandwidth)
        A = _local_rows(rng_b, p, n, nnz_per_row, 2 * bandwidth) if p > 0 else None
        sparse = True
    elif not sparse:
        Q, _ = np.linalg.qr(rng.standard_normal((n, n)))
        P = (Q * lam) @ Q.T
        P = 0.5 * (P + P.T)
        G = rng.standard_normal((m, n)) / np.sqrt(n)
        A = rng.standard_normal((p, n)) / np.sqrt(n) if p > 0 else None
    else:
        E = _sparse_random(rng, n, n, density / 2)
        E = 0.5 * (E + E.T)
        row_abs = np.asarray(abs(E).sum(axis=1)).ravel()
        scale = 0.1 * lam.min() / max(row_abs.max(), 1e-300)
        P = (sp.diags(lam) + scale * E).tocsr()
        G = (_sparse_random(rng, m, n, density) / np.sqrt(max(1, density * n))).tocsr()
        A = (_sparse_random(rng, p, n, max(density, 3.0 / n)) / np.sqrt(max(1, density * n))).tocsr() if p > 0 else None
    # A must have full row rank (checked densely when small enough)
    if p > 0 and p * n <= 4_000_000:
        Ad = A.toarray() if sp.issparse(A) else A
        if np.linalg.matrix_rank(Ad) < p:
            raise ValueError(f"{inst_id}: A is rank deficient")
    x_star = rng.standard_normal(n)
    n_active = min(int(math.ceil(active_frac * m)), max(0, n - p - 1))
    active = np.zeros(m, dtype=bool)
    active[rng.choice(m, n_active, replace=False)] = True
    s_star = np.where(active, 0.0, rng.uniform(0.5, 2.0, m))
    z_star = np.where(active, rng.uniform(0.5, 2.0, m), 0.0)
    y_star = rng.standard_normal(p) if p > 0 else np.zeros(0)
    Px = P @ x_star
    q = -(Px + G.T @ z_star + (A.T @ y_star if p > 0 else 0.0))
    h = G @ x_star + s_star
    b = A @ x_star if p > 0 else None
    f_star = 0.5 * x_star @ Px + q @ x_star
    Pd = P.toarray() if sp.issparse(P) else P
    kappa_meas = float(np.linalg.cond(Pd)) if n <= 2000 else float("nan")
    nnz_kkt = (P.nnz if sp.issparse(P) else n * n) + 2 * (G.nnz if sp.issparse(G) else m * n) + 2 * ((A.nnz if sp.issparse(A) else p * n) if p > 0 else 0) + m
    inst = dict(id=inst_id, family=family, kind="coneqp", n=n, m=m, p=p, kkt_dim=n + m + p,
                kappa_P_target=float(kappa_P), kappa_P_measured=kappa_meas, density=float(density), seed=int(seed), structure=structure, nnz_kkt=int(nnz_kkt),
                sparse=sparse, P=P, q=q, G=G, h=h, A=A, b=b, dims={"l": m, "q": [], "s": []},
                x_star=x_star, y_star=y_star, z_star=z_star, s_star=s_star, f_star=float(f_star),
                n_active=int(n_active), known_solution=True)
    return inst

def cvx_data(inst):
    """CVXOPT matrices for an instance (cached)."""
    if "_cvx" not in inst:
        sparse = inst["sparse"]
        d = {"P": cvx_matrix_from(inst["P"], sparse) if inst["P"] is not None else None,
             "q": to_cvx(inst["q"]), "G": cvx_matrix_from(inst["G"], sparse), "h": to_cvx(inst["h"]),
             "A": cvx_matrix_from(inst["A"], sparse) if inst["p"] > 0 else None,
             "b": to_cvx(inst["b"]) if inst["p"] > 0 else None}
        inst["_cvx"] = d
    return inst["_cvx"]

def build_family(name, spec):
    out = []
    for n in spec["n_list"]:
        m = int(round(spec["m_ratio"] * n)); p = int(round(spec["p_ratio"] * n))
        seeds = spec["seeds"] if n < spec.get("seeds_large_from_n", 10 ** 9) else spec.get("seeds_large", spec["seeds"])     # fewer seeds at the largest sizes
        for kappa in spec["kappa_list"]:
            for seed in seeds:
                inst_id = f"{name}_n{n}_k{int(np.log10(kappa))}_s{seed}"
                out.append(make_qp_instance(inst_id, name, n, m, p, kappa, spec["density"], spec["active_frac"], seed, structure=spec.get("structure"),
                                            nnz_per_row=spec.get("nnz_per_row", 4), bandwidth=spec.get("bandwidth", 3)))
    return out

def sanity_instances():
    return [make_qp_instance("tiny_A", "sanity", 2, 2, 0, 1e0, 1.0, 0.5, 0),
            make_qp_instance("tiny_B", "sanity", 3, 4, 1, 1e1, 1.0, 0.5, 1),
            make_qp_instance("tiny_C", "sanity", 4, 4, 0, 1e2, 1.0, 0.5, 2)]

INSTANCES = {}
for inst in sanity_instances():
    INSTANCES[inst["id"]] = inst
for fam, spec in CONFIG["families"].items():
    for inst in build_family(fam, spec):
        INSTANCES[inst["id"]] = inst
summary = pd.DataFrame([{k: inst[k] for k in ("id", "family", "n", "m", "p", "kkt_dim", "kappa_P_target", "kappa_P_measured", "density", "structure", "nnz_kkt", "sparse", "n_active")} for inst in INSTANCES.values()])
print(f"{len(INSTANCES)} instances generated")
print(summary.groupby("family").agg(count=("id", "size"), n_min=("n", "min"), n_max=("n", "max"), kkt_dim_max=("kkt_dim", "max"), nnz_kkt_max=("nnz_kkt", "max"), structure=("structure", "first")).to_string())
summary.to_csv(os.path.join(RESULTS_DIR, "instances.csv"), index=False)

## 6. Maros-Mészáros QPs

A subset of the Maros-Mészáros convex QP test set (Maros and Mészáros, *Optim. Methods Softw.* 11, 1999), read from the `.mat` files of [`qpsolvers/maros_meszaros_qpbenchmark`](https://github.com/qpsolvers/maros_meszaros_qpbenchmark). These files store $\min \tfrac12 x^\top P x + q^\top x + r$ subject to $l \le Ax \le u$, with variable bounds as rows of $A$. Conversion to the `coneqp` form:

* rows with $l = u$ become equality rows, every other finite bound becomes one inequality row, free rows are dropped;
* redundant equality rows are removed by a pivoted QR of $A^\top$ (the number is recorded);
* $P$ is symmetrised and kept sparse.

No optimal value is assumed. The objective of the unmodified `coneqp` run is the reference for `obj_rel_err`, and an instance that the baseline cannot solve is reported as such for every method. Many of these $P$ are only positive semidefinite, so the acceptance of the final iterate uses CVXOPT's relative gap instead of the recomputed dual value.

The files are looked up in `search_dirs` (for example a Kaggle dataset under `/kaggle/input`) and downloaded otherwise. A file that cannot be found makes the instance `skipped`.

In [ ]:
# Maros-Meszaros QPs: locate or download the .mat file, convert to coneqp form, register as family "maros_meszaros".
MM_CFG = CONFIG["maros_meszaros"]; MM_INSTANCES = {}; MM_LOG = []; MM_INF = 1e19

def find_mm_file(name):
    fname = f"{name}.mat"
    for d in MM_CFG["search_dirs"]:
        if not os.path.isdir(d): continue
        p = os.path.join(d, fname)
        if os.path.isfile(p): return p, "local"
        for root, dirs, files in os.walk(d):
            if fname in files: return os.path.join(root, fname), "local"
            if root[len(d):].count(os.sep) >= 3: dirs[:] = []
    cache = os.path.join(CONFIG["output_root"], "maros_meszaros_cache"); os.makedirs(cache, exist_ok=True); p = os.path.join(cache, fname)
    if os.path.isfile(p) and os.path.getsize(p) > 0: return p, "cache"
    if MM_CFG.get("download_if_missing", True):
        import urllib.request
        try:
            urllib.request.urlretrieve(MM_CFG["download_url"].format(name=name), p)
            if os.path.getsize(p) > 0: return p, "download"
        except Exception as e:
            MM_LOG.append(f"{name}: download failed ({type(e).__name__}: {str(e)[:80]})")
    return None, None

def mm_to_instance(name, path):
    import scipy.io
    d = scipy.io.loadmat(path)
    P = sp.csr_matrix(d["P"]).astype(float); A = sp.csr_matrix(d["A"]).astype(float); q = np.asarray(d["q"], dtype=float).ravel()
    l = np.asarray(d["l"], dtype=float).ravel(); u = np.asarray(d["u"], dtype=float).ravel(); r0 = float(np.asarray(d.get("r", 0.0)).ravel()[0]) if "r" in d else 0.0
    n = P.shape[0]; P = (0.5 * (P + P.T)).tocsr(); P.sum_duplicates()
    l = np.where(l <= -MM_INF, -np.inf, l); u = np.where(u >= MM_INF, np.inf, u)          # the .mat files encode "no bound" as +-1e20
    tol_eq = 1e-10 * np.maximum(1.0, np.abs(l)); eq = np.isfinite(l) & np.isfinite(u) & (u - l <= tol_eq)
    lo = np.isfinite(l) & ~eq; up = np.isfinite(u) & ~eq; free = ~np.isfinite(l) & ~np.isfinite(u)
    Aeq = A[eq]; beq = 0.5 * (l[eq] + u[eq]); p = int(eq.sum()); removed = 0
    if p > 0 and p * n <= 4_000_000:
        Ad = Aeq.toarray(); rk = int(np.linalg.matrix_rank(Ad))
        if rk < p:                                       # keep an independent subset of the equality rows (pivoted QR of A')
            _, _, piv = sla.qr(Ad.T, mode="economic", pivoting=True); keep = np.sort(piv[:rk]); Aeq = Aeq[keep]; beq = beq[keep]; removed = p - rk; p = rk
    G = sp.vstack([-A[lo], A[up]], format="csr") if (lo.any() or up.any()) else sp.csr_matrix((0, n))
    h = np.concatenate([-l[lo], u[up]]); m = G.shape[0]
    rank_ok, gram_cond = True, np.nan
    if n <= 2500:                                    # coneqp needs rank([P; A; G]) = n: test the n x n Gram matrix P'P + A'A + G'G (dense, moderate n only)
        Gm = (P.T @ P + G.T @ G + (Aeq.T @ Aeq if p else 0)).toarray(); gram_cond = float(np.linalg.cond(Gm)); rank_ok = bool(np.isfinite(gram_cond) and gram_cond < 1e14)
    nnz_kkt = int(P.nnz + 2 * G.nnz + 2 * (Aeq.nnz if p else 0) + m)
    inst = dict(id=f"mm_{name}", family="maros_meszaros", kind="coneqp", n=int(n), m=int(m), p=int(p), kkt_dim=int(n + m + p), kappa_P_target=np.nan, kappa_P_measured=np.nan,
                density=float(A.nnz / max(1, A.shape[0] * n)), seed=0, structure="real", nnz_kkt=nnz_kkt, sparse=True, P=P, q=q, G=G, h=h, A=(Aeq.tocsr() if p else None), b=(beq if p else None),
                dims={"l": int(m), "q": [], "s": []}, x_star=None, y_star=None, z_star=None, s_star=None, f_star=np.nan, n_active=-1, known_solution=False, solution_source="",
                no_dual_recompute=True, obj_offset=r0, mm_name=name, mm_rows_total=int(A.shape[0]), mm_rows_free=int(free.sum()), mm_eq_rows_removed=int(removed), mm_nnz_P=int(P.nnz), mm_nnz_A=int(A.nnz),
                mm_P_zero=bool(P.nnz == 0), mm_rank_ok=rank_ok, mm_gram_cond=gram_cond)
    return inst

if not MM_CFG.get("enabled", True):
    REC.status("MM_load", "skipped", "disabled in CONFIG")
else:
    t_mm = time.perf_counter(); missing = []
    for name in MM_CFG["instances"]:
        path, src = find_mm_file(name)
        if path is None: missing.append(name); continue
        try:
            inst = mm_to_instance(name, path); inst["source"] = src
            if inst["kkt_dim"] > MM_CFG.get("max_kkt_dim", 10 ** 9): REC.status(f"MM_{name}", "skipped", f"kkt_dim {inst['kkt_dim']} > max_kkt_dim"); continue
            if inst["m"] == 0: REC.status(f"MM_{name}", "skipped", "no inequality rows after conversion"); continue
            if not inst["mm_rank_ok"]: REC.status(f"MM_{name}", "skipped", f"rank([P; A; G]) < n (Gram condition {inst['mm_gram_cond']:.1e}): violates the coneqp assumption; free variables without curvature"); continue
            MM_INSTANCES[inst["id"]] = inst; INSTANCES[inst["id"]] = inst
        except Exception as e:
            REC.failure(f"mm_{name}", "parse", repr(e)[:300]); MM_LOG.append(f"{name}: {repr(e)[:120]}")
    for msg in MM_LOG: print("  [mm]", msg)
    if missing: REC.status("MM_missing", "skipped", f"not found / not downloadable (Internet off?): {missing}")
    if MM_INSTANCES:
        mm_summary = pd.DataFrame([{k: inst[k] for k in ("id", "family", "n", "m", "p", "kkt_dim", "kappa_P_target", "kappa_P_measured", "density", "structure", "nnz_kkt", "sparse", "n_active")} for inst in MM_INSTANCES.values()])
        summary = pd.concat([summary, mm_summary], ignore_index=True); summary.to_csv(os.path.join(RESULTS_DIR, "instances.csv"), index=False)
        mm_detail = pd.DataFrame([{k: inst[k] for k in ("id", "mm_name", "n", "m", "p", "kkt_dim", "nnz_kkt", "mm_nnz_P", "mm_nnz_A", "mm_rows_total", "mm_rows_free", "mm_eq_rows_removed", "mm_P_zero", "mm_gram_cond", "source")} for inst in MM_INSTANCES.values()])
        mm_detail.to_csv(os.path.join(RESULTS_DIR, "maros_meszaros_instances.csv"), index=False)
        print(f"Maros-Meszaros: {len(MM_INSTANCES)} instances loaded in {time.perf_counter() - t_mm:.1f}s"); print(mm_detail.to_string(index=False))
        REC.status("MM_load", "done", f"{len(MM_INSTANCES)} instances ({len(missing)} missing)")
    else:
        REC.status("MM_load", "skipped", f"no instance could be loaded: missing={missing}")

## 7. KKT assembly, conditioning and scaling

`KKTAssembler` forms $\tilde K$ with the block layout and the $W^{-\top}G$ scaling of `cvxopt.misc.kkt_ldl`; for semidefinite cones it uses the same packed storage with $\sqrt2$-scaled off-diagonals (`misc.scale`, `misc.pack`). Every solver therefore sees the matrix that CVXOPT's own `'ldl'` option factors. The cell ends with a check against `misc.kkt_ldl` on `tiny_B`.

$\kappa_2(\tilde K)$ is computed by an SVD at every factorisation of the reference runs up to dimension `cond_limit`. This diagnostic is timed separately and excluded from the method timings; above the limit the field is `NaN`.

Ruiz equilibration $\tilde K_s = D\tilde K D$ (six sweeps of $d_i \leftarrow d_i/\sqrt{\|(D\tilde K D)_{i,:}\|_\infty}$) keeps the matrix symmetric, so MINRES and the quantum solvers can both use it; the solution is $\tilde u = D\tilde u_s$. For a quantum solver $D$ costs only row norms, and since the IPM reads the solution out classically anyway, applying $D$ to the result is a classical rescaling.

`reduced_spd` eliminates $w$ for $p = 0$ and returns $S = P + G^\top W^{-1}W^{-\top}G$ with the maps between the full and the reduced system.

In [ ]:
# KKT assembly (same layout as cvxopt.misc.kkt_ldl), condition numbers, Ruiz scaling, reduced SPD system.
class KKTAssembler:
    """Symmetric scaled KKT matrix  K~ = [P A' G'W^{-1}; A 0 0; W^{-T}G 0 -I]  and its rhs/solution packing."""
    def __init__(self, inst):
        self.inst = inst
        self.n, self.p, self.dims = inst["n"], inst["p"], inst["dims"]
        self.l_only = not (self.dims["q"] or self.dims["s"])
        self.sparse = bool(inst.get("sparse", False)) and self.l_only
        self.mz = self.dims["l"] + sum(self.dims["q"]) + sum(k * (k + 1) // 2 for k in self.dims["s"])
        self.dim = self.n + self.p + self.mz
        self.P = inst["P"]; self.G = inst["G"]; self.A = inst["A"] if self.p > 0 else None
        if not self.l_only:
            cd = cvx_data(inst)
            self.Gc = matrix(cd["G"]) if isinstance(cd["G"], spmatrix) else cd["G"]   # dense cvxopt copy for column scaling

    def build(self, W):
        n, p, N = self.n, self.p, self.dim
        if self.l_only:
            di = to_np(W["di"])
            if self.sparse:
                Gs = (sp.diags(di) @ self.G).tocsr()
                P = self.P if self.P is not None else sp.csr_matrix((n, n))
                blocks = [[P, self.A.T if p else None, Gs.T],
                          [self.A if p else None, None, None],
                          [Gs, None, -sp.identity(self.mz, format="csr")]]
                if p == 0:
                    blocks = [[P, Gs.T], [Gs, -sp.identity(self.mz, format="csr")]]
                return sp.bmat(blocks, format="csc")
            K = np.zeros((N, N))
            if self.P is not None:
                K[:n, :n] = self.P.toarray() if sp.issparse(self.P) else self.P
            if p:
                A = self.A.toarray() if sp.issparse(self.A) else self.A
                K[n:n + p, :n] = A; K[:n, n:n + p] = A.T
            G = self.G.toarray() if sp.issparse(self.G) else self.G
            Gs = di[:, None] * G
            K[n + p:, :n] = Gs; K[:n, n + p:] = Gs.T
            K[n + p:, n + p:] = -np.eye(self.mz)
            return K
        # generic cone path (used for SDP): identical construction to misc.kkt_ldl
        K = np.zeros((N, N))
        if self.P is not None:
            K[:n, :n] = self.P.toarray() if sp.issparse(self.P) else self.P
        if p:
            A = self.A.toarray() if sp.issparse(self.A) else self.A
            K[n:n + p, :n] = A; K[:n, n:n + p] = A.T
        gp = matrix(0.0, (self.mz, 1))
        for k in range(n):
            g = self.Gc[:, k]
            misc.scale(g, W, trans="T", inverse="I")
            misc.pack(g, gp, self.dims)
            col = to_np(gp)
            K[n + p:, k] = col; K[k, n + p:] = col
        K[n + p:, n + p:] = -np.eye(self.mz)
        return K

    def rhs(self, x, y, z, W):
        parts = [to_np(x)]
        if self.p: parts.append(to_np(y))
        if self.l_only:
            parts.append(to_np(W["di"]) * to_np(z))
        else:
            zc = matrix(z); misc.scale(zc, W, trans="T", inverse="I")
            zp = matrix(0.0, (self.mz, 1)); misc.pack(zc, zp, self.dims)
            parts.append(to_np(zp))
        return np.concatenate(parts)

    def write_back(self, u, x, y, z):
        n, p = self.n, self.p
        write_back(x, u[:n])
        if p: write_back(y, u[n:n + p])
        if self.l_only:
            write_back(z, u[n + p:])
        else:
            misc.unpack(to_cvx(u[n + p:]), z, self.dims)

def nt_spread(W):
    """Heuristic proxy for the duality measure mu from the NT scaling: min/max of the scaling factors.
    For the linear cone d_i = sqrt(s_i/z_i): active constraints give d_i ~ sqrt(mu), inactive ones d_i ~ 1/sqrt(mu),
    so min(d)/max(d) ~ mu (up to constants). For SDP blocks the singular values of r_k are used."""
    vals = [to_np(W["d"])] if W["d"].size[0] > 0 else []
    for r in W.get("r", []):
        s = np.linalg.svd(np.array(r), compute_uv=False); vals.append(s ** 2)
    for bta in W.get("beta", []):
        vals.append(np.array([bta]))
    if not vals: return 1.0
    v = np.concatenate(vals)
    return float(v.min() / v.max()) if v.max() > 0 else 1.0

def cond_exact(K, limit=None):
    """2-norm condition number (SVD) for dimension <= limit; NaN otherwise."""
    limit = CONFIG["cond_limit"] if limit is None else limit
    N = K.shape[0]
    if N > limit: return float("nan")
    Kd = K.toarray() if sp.issparse(K) else K
    try:
        return float(np.linalg.cond(Kd))
    except Exception:
        return float("nan")

def ruiz_equilibrate(K, iters=None):
    """Symmetric Ruiz equilibration: returns d such that D K D has rows/cols of ~unit inf-norm."""
    iters = CONFIG["inexact"]["ruiz_iters"] if iters is None else iters
    N = K.shape[0]; d = np.ones(N)
    if sp.issparse(K):
        # sweeps on the CSR arrays; forming D K D with two sparse products per sweep is much slower at large N
        Kc = K.tocsr(); ip = Kc.indptr; cnt = np.diff(ip); a = np.abs(Kc.data); rows = np.repeat(np.arange(N), cnt); cols = Kc.indices; ne = cnt > 0
        for _ in range(iters):
            v = a * d[rows] * d[cols]; rn = np.ones(N)
            if v.size: rn[ne] = np.maximum.reduceat(v, ip[:-1][ne])
            rn[rn == 0] = 1.0; d = d / np.sqrt(rn)
        return d
    for _ in range(iters):
        Ks = apply_scaling(K, d)
        if sp.issparse(Ks):
            rn = np.asarray(abs(Ks).max(axis=1).todense()).ravel()
        else:
            rn = np.abs(Ks).max(axis=1)
        rn[rn == 0] = 1.0
        d = d / np.sqrt(rn)
    return d

def apply_scaling(K, d):
    if sp.issparse(K):
        return (sp.diags(d) @ K @ sp.diags(d)).tocsc()
    return d[:, None] * K * d[None, :]

def reduced_spd(asm, W):
    """For p = 0 and the linear cone: S = P + G' W^{-1} W^{-T} G (SPD) plus maps between the full and reduced systems."""
    assert asm.l_only and asm.p == 0
    di = to_np(W["di"])
    if asm.sparse:
        Gs = (sp.diags(di) @ asm.G).tocsr()
        S = (asm.P + Gs.T @ Gs).tocsc()
    else:
        G = asm.G.toarray() if sp.issparse(asm.G) else asm.G
        Gs = di[:, None] * G
        P = asm.P.toarray() if sp.issparse(asm.P) else asm.P
        S = P + Gs.T @ Gs
    n = asm.n
    def rhs_S(r):                       # r = [bx; di*bz]
        return r[:n] + Gs.T @ r[n:]
    def expand(ux, r):                  # w = W^{-T} G ux - W^{-T} bz
        return np.concatenate([ux, Gs @ ux - r[n:]])
    return S, rhs_S, expand

# quick self-test of the assembler against cvxopt's own LDL solver on a tiny instance
_inst = INSTANCES["tiny_B"]; _asm = KKTAssembler(_inst)
_W = {"d": matrix(np.array([0.5, 2.0, 1.0, 3.0])), "di": matrix(np.array([2.0, 0.5, 1.0, 1 / 3])), "v": [], "beta": [], "r": [], "rti": []}
_K = _asm.build(_W); _cd = cvx_data(_inst)
_ref = misc.kkt_ldl(_cd["G"], _inst["dims"], _cd["A"])(_W, _cd["P"])
_x, _y, _z = to_cvx(np.arange(1, 4.0)), to_cvx(np.array([0.7])), to_cvx(np.array([1.0, -2.0, 0.5, 0.25]))
_r = _asm.rhs(_x, _y, _z, _W); _u = np.linalg.solve(_K, _r)
_ref(_x, _y, _z)
_uref = np.concatenate([to_np(_x), to_np(_y), to_np(_z)])
print("assembler vs cvxopt.misc.kkt_ldl: max |diff| =", float(np.abs(_u - _uref).max()), " (must be ~1e-15)")
assert np.allclose(_u, _uref, atol=1e-10)

## 8. KKT solvers and the solve protocol

Every KKT solve of a method other than the unmodified baseline goes through `KKTSolveEngine.solve`:

1. **Base solve** with the method's solver (table below).
2. **Verify** the direction on the original system with the method's acceptance test (Section 3.4). Krylov methods stop as soon as the test holds; it is evaluated every `check_every` iterations, because checking every iteration costs as much as the iteration itself.
3. **Refine** at most `max_refine` times with the same solver on the residual.
4. **Fall back** to a direct solve of the reduced quasi-definite system (CHOLMOD) if the test still fails and the method allows it. Fallbacks are counted and logged as `classical_fallback`; a run counts as fallback-free only with zero fallbacks.

The reference solver `exact` never refines, never falls back and does not stop a run on a large residual; only a non-finite direction raises `ArithmeticError`, as in CVXOPT itself. Every solve log carries all three test values (`relres`, `backward_err`, `block_ratio_max` and the per-block ratios), the estimate of $\|\tilde K\|_2$ and the timing components, so the acceptance tests can be re-evaluated offline.

| kind | solver |
|---|---|
| `exact` | dense LU or SuperLU of $\tilde K$ |
| `exact_reduced` | CHOLMOD on the reduced system $[[S, A^\top], [A, 0]]$ with a dense Schur complement |
| `noise` | exact solve plus an injected relative error (isotropic or per block) |
| `minres` | SciPy MINRES, optional Ruiz scaling |
| `minres_base` | MINRES written against NumPy or CuPy, optional warm start (baseline kernel) |
| `minres_fused` | MINRES with one host synchronisation per iteration and fused vector updates (latency-optimised kernel) |
| `cg_reduced` | CG on $S$ for $p = 0$ |
| `oss_cg` | CG on the Jacobi-scaled null-space system $Z^\top S Z$ |
| `lsqr_sqrt` | LSQR on the square-root factor $B$ with $B^\top B = S$ |
| `gmres_stalelu` | GMRES preconditioned with the factors of an earlier IPM iteration |

The cell ends with self-tests of both MINRES kernels against a dense solve, on the GPU as well when one is available.

In [ ]:
# KKT solvers and the solve protocol: acceptance test on the original system, refinement, classical fallback.
_KW_TOL_MINRES = "rtol" if "rtol" in inspect.signature(spla.minres).parameters else "tol"
_KW_TOL_CG = "rtol" if "rtol" in inspect.signature(spla.cg).parameters else "tol"

class ExactFactor:
    def __init__(self, K):
        self.sparse = sp.issparse(K)
        self.lu = spla.splu(K.tocsc()) if self.sparse else sla.lu_factor(K, check_finite=False)
    def solve(self, r):
        return self.lu.solve(r) if self.sparse else sla.lu_solve(self.lu, r, check_finite=False)

def norm2_estimate(K, iters=None, rng=None):
    """||K||_2 by power iteration (K symmetric): iters mat-vecs, cheap relative to a solve."""
    iters = CONFIG["inexact"]["norm_est_iters"] if iters is None else iters
    rng = np.random.default_rng(0) if rng is None else rng
    v = rng.standard_normal(K.shape[0]); v /= np.linalg.norm(v); lam = 0.0
    for _ in range(iters):
        w = K @ v; lam = float(np.linalg.norm(w))
        if lam == 0: return 0.0
        v = w / lam
    return lam

class _Converged(Exception):
    pass

def krylov_solve(Kop, r, kind, maxiter, stop_fn, check_every):
    """MINRES / CG with an explicit stopping test stop_fn(xk) -> bool evaluated every `check_every` iterations.
    SciPy's own test is disabled (rtol=1e-30) so the criterion is exactly the one requested; the checking cost is timed (t_check)."""
    cnt = {"it": 0, "t_check": 0.0}; box = {"u": None}
    def cb(xk):
        cnt["it"] += 1
        if cnt["it"] % check_every == 0:
            t0 = time.perf_counter(); ok = stop_fn(xk); cnt["t_check"] += time.perf_counter() - t0
            if ok:
                box["u"] = xk.copy(); raise _Converged()
    try:
        if kind == "minres":
            u, info = spla.minres(Kop, r, maxiter=maxiter, callback=cb, **{_KW_TOL_MINRES: 1e-30})
        else:
            u, info = spla.cg(Kop, r, maxiter=maxiter, callback=cb, **{_KW_TOL_CG: 1e-30})
        info = int(info)
    except _Converged:
        u, info = box["u"], 0
    return u, info, cnt["it"], cnt["t_check"]

SOLVER_KINDS = {}

class ExactKind:
    def __init__(self, spec, K, asm, W, eng): self.f = ExactFactor(K)
    def base_solve(self, r, ctx): return self.f.solve(r), {}
SOLVER_KINDS["exact"] = ExactKind

class NoiseKind:
    """Exact solve plus a random perturbation of prescribed relative size (sensitivity experiment, not a quantum solver).
    model 'iso'  : u += delta*||u|| * g/||g||
    model 'block': u_b += delta*||u_b|| * g_b/||g_b|| per block"""
    def __init__(self, spec, K, asm, W, eng):
        self.f = ExactFactor(K); self.delta = spec["delta"]; self.model = spec.get("noise_model", "iso")
        self.rng = rng_for("noise", spec["name"], asm.inst["id"]); self.blocks = eng.block_slices
    def base_solve(self, r, ctx):
        u = self.f.solve(r)
        if self.model == "iso":
            g = self.rng.standard_normal(u.shape); g /= max(np.linalg.norm(g), 1e-300); u = u + self.delta * np.linalg.norm(u) * g
        else:
            u = u.copy()
            for sl in self.blocks:
                if sl.stop - sl.start == 0: continue
                g = self.rng.standard_normal(sl.stop - sl.start); g /= max(np.linalg.norm(g), 1e-300)
                u[sl] += self.delta * np.linalg.norm(u[sl]) * g
        return u, {"injected_rel_noise": self.delta, "noise_model": self.model}
SOLVER_KINDS["noise"] = NoiseKind

class MinresKind:
    def __init__(self, spec, K, asm, W, eng):
        self.scaled = spec.get("scaling", "ruiz") == "ruiz"
        self.d = ruiz_equilibrate(K) if self.scaled else None
        self.Ks = apply_scaling(K, self.d) if self.scaled else K
        self.maxiter = int(min(CONFIG["inexact"]["minres_maxiter_cap"], CONFIG["inexact"]["minres_maxiter_factor"] * K.shape[0]))
        self.eng = eng
    def base_solve(self, r, ctx):
        rs = self.d * r if self.scaled else r
        to_full = (lambda xs: self.d * xs) if self.scaled else (lambda xs: xs)
        stop = lambda xk: self.eng.criterion_holds(to_full(xk), r, ctx)
        us, info, its, tc = krylov_solve(self.Ks, rs, "minres", self.maxiter, stop, CONFIG["inexact"]["check_every"])
        return to_full(us), {"krylov_iters": its, "krylov_info": info, "t_check": tc}
SOLVER_KINDS["minres"] = MinresKind

class CGReducedKind:
    """CG on the reduced SPD system S = P + G'W^{-1}W^{-T}G (p = 0 only), Ruiz-scaled, expanded to the full system for verification."""
    def __init__(self, spec, K, asm, W, eng):
        self.S, self.rhs_S, self.expand = reduced_spd(asm, W)
        self.scaled = spec.get("scaling", "ruiz") == "ruiz"
        self.d = ruiz_equilibrate(self.S) if self.scaled else None
        self.Ss = apply_scaling(self.S, self.d) if self.scaled else self.S
        self.maxiter = int(min(CONFIG["inexact"]["minres_maxiter_cap"], CONFIG["inexact"]["minres_maxiter_factor"] * self.S.shape[0]))
        self.kappa_S = cond_exact(self.S) if spec.get("_compute_cond") else float("nan"); self.eng = eng
    def base_solve(self, r, ctx):
        rS = self.rhs_S(r); rSs = self.d * rS if self.scaled else rS
        to_full = (lambda xs: self.expand(self.d * xs if self.scaled else xs, r))
        stop = lambda xk: self.eng.criterion_holds(to_full(xk), r, ctx)
        us, info, its, tc = krylov_solve(self.Ss, rSs, "cg", self.maxiter, stop, CONFIG["inexact"]["check_every"])
        return to_full(us), {"krylov_iters": its, "krylov_info": info, "t_check": tc, "kappa_reduced": self.kappa_S}
SOLVER_KINDS["cg_reduced"] = CGReducedKind

def tolerance_for(spec, ctx):
    rule = spec.get("tol_rule", "fixed")
    if rule == "fixed": return float(spec.get("tol", 1e-8))
    if rule == "adaptive_nt":
        c = spec; return float(min(c["tol_max"], max(c["tol_min"], c["eta"] * ctx["nt_proxy"])))
    raise ValueError(rule)

class KKTSolveEngine:
    """One object per kktsolver(W) call: holds K, its norm estimate, the block layout and the acceptance criterion."""
    def __init__(self, spec, K, asm, W):
        self.spec, self.K, self.asm, self.W = spec, K, asm, W
        self.kind = spec["kind"]; self.criterion = spec.get("criterion", "backward")
        n, p, mz = asm.n, asm.p, asm.mz
        self.block_slices = [slice(0, n), slice(n, n + p), slice(n + p, n + p + mz)]
        inst = asm.inst
        floor = float(spec.get("block_abs_floor", CONFIG["inexact"]["block_abs_floor"]))
        self.block_floor = [floor * max(1.0, np.linalg.norm(inst["q"])), floor * max(1.0, np.linalg.norm(inst["b"]) if inst.get("b") is not None and p > 0 else 1.0),
                            floor * max(1.0, np.linalg.norm(inst["h"]))]
        self.nK = norm2_estimate(K)
        self.accept_factor = spec.get("accept_factor", CONFIG["inexact"]["accept_factor"])
        self.max_refine = spec.get("max_refine", CONFIG["inexact"]["max_refine"])
        self.fallback = spec.get("fallback", CONFIG["inexact"]["fallback"])
        if self.kind == "exact": self.max_refine, self.fallback = 0, False
        self.impl = SOLVER_KINDS[self.kind](spec, K, asm, W, self)
        if self.kind == "exact_reduced": self.max_refine, self.fallback = 0, False
        self._exact = self.impl.f if isinstance(self.impl, (ExactKind, NoiseKind)) else None
        self._last_measure = None
    def exact(self):
        if self._exact is None:
            # the fallback factors the reduced quasi-definite system with CHOLMOD, as CVXOPT does; the LU of the full KKT
            # matrix fills in badly on the sparse families (fallback_reduced=False restores it)
            if self.asm.l_only and CONFIG["inexact"].get("fallback_reduced", True):
                try: self._exact = _ReducedFallback(self.K, self.asm, self.W, self)
                except Exception: self._exact = ExactFactor(self.K)
            else: self._exact = ExactFactor(self.K)
        return self._exact
    # acceptance tests, always on the original system
    def measure(self, u, r):
        e = self.K @ u - r
        ne, nr, nu = np.linalg.norm(e), np.linalg.norm(r), np.linalg.norm(u)
        relres = ne / nr if nr > 0 else ne
        backward = ne / (self.nK * nu + nr) if (self.nK * nu + nr) > 0 else ne
        ratios = []
        for sl, fl in zip(self.block_slices, self.block_floor):
            if sl.stop - sl.start == 0: ratios.append(0.0); continue
            ratios.append(float(np.linalg.norm(e[sl]) / max(np.linalg.norm(r[sl]), fl)))
        return {"relres": float(relres), "backward_err": float(backward), "block_ratio_x": ratios[0], "block_ratio_y": ratios[1], "block_ratio_z": ratios[2],
                "block_ratio_max": float(max(ratios)), "abs_resid": float(ne)}
    def threshold(self, ctx):
        if self.criterion == "block": return float(self.spec.get("eta", 0.1))
        if self.criterion in ("relres_floor", "combined"): return 1.0
        return float(self.accept_factor * tolerance_for(self.spec, ctx))
    def criterion_value(self, m, ctx=None):
        if self.criterion == "relres_floor":
            # relres, or a backward error already at the level of a direct solve
            tol = self.accept_factor * tolerance_for(self.spec, ctx or {}); return float(min(m["relres"] / tol, m["backward_err"] / self.spec.get("be_floor", 1e-14)))
        if self.criterion == "combined":
            tol = self.accept_factor * tolerance_for(self.spec, ctx or {}); return float(max(m["block_ratio_max"] / self.spec.get("eta", 0.1), m["backward_err"] / tol))
        return {"relres": m["relres"], "backward": m["backward_err"], "block": m["block_ratio_max"]}[self.criterion]
    def criterion_holds(self, u, r, ctx):
        if not np.all(np.isfinite(u)): return False
        m = self.measure(u, r); self._last_measure = m
        return self.criterion_value(m, ctx) <= self.threshold(ctx)
    # base solve -> verify -> refine -> fallback
    def solve(self, r, ctx):
        rtol = tolerance_for(self.spec, ctx) if self.criterion not in ("block",) else float(self.spec.get("eta", 0.1))
        accept = self.threshold(ctx)
        if self.kind == "noise": accept = float("inf")
        if self.kind == "exact_reduced": accept = 1e-6
        info_all = {}
        with Timer() as t_solve:
            u, info = self.impl.base_solve(r, ctx)
        info_all.update(info)
        with Timer() as t_verify:
            finite = bool(np.all(np.isfinite(u)))
            m0 = self.measure(u, r) if finite else {"relres": np.inf, "backward_err": np.inf, "block_ratio_x": np.inf, "block_ratio_y": np.inf, "block_ratio_z": np.inf, "block_ratio_max": np.inf, "abs_resid": np.inf}
            val0 = self.criterion_value(m0, ctx)
        m, val, n_ref, t_ref = m0, val0, 0, Timer()
        if finite and val > accept and self.max_refine > 0:
            with t_ref:
                while val > accept and n_ref < self.max_refine:
                    du, info2 = self.impl.base_solve(r - self.K @ u, dict(ctx, is_refine=True))
                    for k2, v2 in info2.items():
                        if k2 in ("krylov_iters", "t_check"): info_all[k2] = info_all.get(k2, 0) + v2
                    if not np.all(np.isfinite(du)): break
                    u = u + du; n_ref += 1; m = self.measure(u, r); val = self.criterion_value(m, ctx)
        used_fallback, t_fb, solve_failed = False, Timer(), False
        if (not np.all(np.isfinite(u))) or val > accept:
            if self.fallback:
                with t_fb:
                    u = self.exact().solve(r); used_fallback = True; info_all["fallback_backend"] = getattr(self._exact, "backend", "lu_full")
                    if np.all(np.isfinite(u)): m = self.measure(u, r); val = self.criterion_value(m, ctx)
                    else: val = np.inf
        if not np.all(np.isfinite(u)): solve_failed = True      # only non-finite directions abort a run (CVXOPT behaviour)
        log = {"solve_failed": solve_failed, "criterion": self.criterion, "target_rtol": rtol, "accept_rtol": accept, "crit_initial": val0, "crit_final": val,
               "rel_resid_initial": m0["relres"], "rel_resid_final": m["relres"], "backward_err_initial": m0["backward_err"], "backward_err_final": m["backward_err"],
               "block_ratio_max_initial": m0["block_ratio_max"], "block_ratio_max_final": m["block_ratio_max"],
               "block_ratio_x": m["block_ratio_x"], "block_ratio_y": m["block_ratio_y"], "block_ratio_z": m["block_ratio_z"], "abs_resid_final": m["abs_resid"],
               "n_refine": n_ref, "fallback": used_fallback, "accepted": bool(val <= accept), "method_used": "classical_fallback" if used_fallback else self.kind,
               "t_solve": t_solve.t - info_all.get("t_check", 0.0), "t_verify": t_verify.t, "t_refine": t_ref.t, "t_fallback": t_fb.t, "t_check": info_all.get("t_check", 0.0),
               "rhs_norm": float(np.linalg.norm(r)), "rhs_norm_x": float(np.linalg.norm(r[self.block_slices[0]])), "rhs_norm_y": float(np.linalg.norm(r[self.block_slices[1]])),
               "rhs_norm_z": float(np.linalg.norm(r[self.block_slices[2]])), "kkt_dim": int(self.K.shape[0]), "nK_est": float(self.nK)}
        log.update(info_all)
        return u, log



# MINRES on NumPy or CuPy arrays, LSQR on the square-root system, reduced sparse direct solve.
def minres_base_kernel(matvec, b, xp, x0=None, maxiter=1000, stop_fn=None, check_every=8):
    """Paige-Saunders MINRES written against the array module xp (numpy or cupy).
    Stops when stop_fn(x) is True (checked every `check_every` iterations and when the recurrence residual estimate is tiny).
    Returns (x, info, iterations, t_check); info = 0 converged by stop_fn, 1 maxiter, 2 breakdown/exact."""
    t_check = 0.0
    x = xp.zeros_like(b) if x0 is None else x0.copy()
    r = b if x0 is None else b - matvec(x)
    beta1 = float(xp.linalg.norm(r))
    def check(xk):
        nonlocal t_check
        t0 = time.perf_counter(); ok = bool(stop_fn(xk)) if stop_fn is not None else False; t_check += time.perf_counter() - t0
        return ok
    if beta1 == 0.0: return x, 0, 0, t_check
    if x0 is not None and check(x): return x, 0, 0, t_check
    v_old = xp.zeros_like(b); v = r / beta1; beta = beta1
    w_old = xp.zeros_like(b); w = xp.zeros_like(b)
    eta = beta1; g0 = g1 = 1.0; s0 = s1 = 0.0; bnorm = float(xp.linalg.norm(b)) or 1.0
    for it in range(1, maxiter + 1):
        Av = matvec(v); alpha = float(xp.vdot(v, Av).real)
        v_new = Av - alpha * v - beta * v_old
        beta_new = float(xp.linalg.norm(v_new))
        delta = g1 * alpha - g0 * s1 * beta
        rho1 = math.hypot(delta, beta_new); rho2 = s1 * alpha + g0 * g1 * beta; rho3 = s0 * beta
        if rho1 == 0.0: return x, 2, it, t_check
        g_new = delta / rho1; s_new = beta_new / rho1
        w_new = (v - rho3 * w_old - rho2 * w) / rho1
        x = x + (g_new * eta) * w_new
        eta = -s_new * eta
        v_old, v = v, (v_new / beta_new if beta_new > 0 else v_new)
        w_old, w = w, w_new; beta = beta_new; g0, g1 = g1, g_new; s0, s1 = s1, s_new
        if (it % check_every == 0) or abs(eta) <= 1e-15 * bnorm or beta_new == 0.0:
            if check(x): return x, 0, it, t_check
            if beta_new == 0.0: return x, 2, it, t_check
    return x, 1, maxiter, t_check

class MinresBaseKind:
    """Ruiz-scaled MINRES with the backend-agnostic kernel; backend 'cpu' (numpy) or 'gpu' (cupy). Optional warm start:
    the combined-direction solve of an IPM iteration starts from the affine direction computed with the same matrix."""
    def __init__(self, spec, K, asm, W, eng):
        self.gpu = spec.get("backend", "cpu") == "gpu"
        if self.gpu and not GPU_OK: raise RuntimeError("GPU backend requested but no usable GPU")
        self.xp = DEVICE_XP if self.gpu else np
        self.eng = eng; self.ws = bool(spec.get("warm_start", False)); self.last = None
        d = ruiz_equilibrate(K) if spec.get("scaling", "ruiz") == "ruiz" else np.ones(K.shape[0])
        Ks = apply_scaling(K, d)
        t0 = time.perf_counter()
        self.K = to_device(K) if self.gpu else K; self.Ks = to_device(Ks) if self.gpu else Ks; self.d = self.xp.asarray(d)
        self.t_transfer = time.perf_counter() - t0
        self.maxiter = int(min(CONFIG["inexact"]["minres_maxiter_cap"], CONFIG["inexact"]["minres_maxiter_factor"] * K.shape[0]))
        self.floors = eng.block_floor; self.slices = eng.block_slices
    def _holds(self, u, r, ctx):
        xp = self.xp; e = self.K @ u - r
        ne, nr, nu = float(xp.linalg.norm(e)), float(xp.linalg.norm(r)), float(xp.linalg.norm(u))
        if not np.isfinite(ne): return False
        ratios = [0.0 if sl.stop == sl.start else float(xp.linalg.norm(e[sl])) / max(float(xp.linalg.norm(r[sl])), fl) for sl, fl in zip(self.slices, self.floors)]
        m = {"relres": ne / nr if nr > 0 else ne, "backward_err": ne / (self.eng.nK * nu + nr) if (self.eng.nK * nu + nr) > 0 else ne, "block_ratio_max": max(ratios)}
        return self.eng.criterion_value(m, ctx) <= self.eng.threshold(ctx)
    def base_solve(self, r, ctx):
        xp = self.xp; r_d = xp.asarray(r); rs = self.d * r_d
        warm = self.ws and self.last is not None and not ctx.get("is_refine"); x0 = self.last if warm else None
        # Warm start as a correction solve K delta = r - K u0, with the acceptance test applied to that system. Testing u0 + delta
        # against r is unsafe: the affine direction u0 can pass the block test for the combined right-hand side at iteration 0,
        # the corrector term is then dropped and the IPM diverges (seen on 2 of 35 instances).
        r_t = (r_d - self.K @ x0) if warm else r_d
        xs, info, its, tc = minres_base_kernel(lambda v: self.Ks @ v, self.d * r_t, xp, x0=None, maxiter=self.maxiter,
                                      stop_fn=lambda xk: self._holds(self.d * xk, r_t, ctx), check_every=CONFIG["inexact"]["check_every"])
        u_d = self.d * xs + (x0 if warm else 0.0)
        if not ctx.get("is_refine"): self.last = u_d
        if self.gpu: dev_sync()
        return to_host(u_d) if self.gpu else u_d, {"krylov_iters": its, "krylov_info": info, "t_check": tc, "warm_started": bool(warm), "backend": "gpu" if self.gpu else "cpu", "t_transfer": self.t_transfer}
SOLVER_KINDS["minres_base"] = MinresBaseKind

class LSQRSqrtKind:
    """Square-root target system for P > 0 and no equalities: S = P + Ghat'Ghat = B'B with B = [L'; Ghat], P = L L'.
    LSQR on B sees kappa(B) = sqrt(kappa(S))."""
    def __init__(self, spec, K, asm, W, eng):
        assert asm.l_only and asm.p == 0
        inst = asm.inst; n = asm.n
        if "_Lchol" not in inst:
            Pd = inst["P"].toarray() if sp.issparse(inst["P"]) else inst["P"]; inst["_Lchol"] = np.linalg.cholesky(Pd)
        self.L = inst["_Lchol"]; di = to_np(W["di"]); G = inst["G"]
        self.Gh = (sp.diags(di) @ G).toarray() if sp.issparse(G) else di[:, None] * G
        B = np.vstack([self.L.T, self.Gh]); self.c = 1.0 / np.maximum(np.linalg.norm(B, axis=0), 1e-300); self.Bs = B * self.c[None, :]
        self.n = n; self.maxiter = int(min(CONFIG["inexact"]["minres_maxiter_cap"], 4 * n)); self.tol = float(spec.get("lsqr_tol", 1e-13))
    def base_solve(self, r, ctx):
        n = self.n; bx, bz = r[:n], r[n:]
        t = np.concatenate([sla.solve_triangular(self.L, bx, lower=True), bz])
        y, istop, itn = spla.lsqr(self.Bs, t, atol=self.tol, btol=self.tol, iter_lim=self.maxiter)[:3]
        ux = self.c * y
        return np.concatenate([ux, self.Gh @ ux - bz]), {"krylov_iters": int(itn), "krylov_info": int(istop)}
SOLVER_KINDS["lsqr_sqrt"] = LSQRSqrtKind

class ReducedCholFactor:
    """Sparse factorisation of the reduced quasi-definite system M = [[S, A'], [A, 0]] (S symmetric positive definite) with
    CHOLMOD (shipped with CVXOPT) and a dense p x p Schur complement C = A S^-1 A'. At n = 3 200 this takes 0.44 s per
    factorisation against 4.7-7.1 s for SuperLU. Every solve applies one step of iterative refinement on M. Uses SuperLU
    (symmetric-mode ordering) if CHOLMOD reports that S is not positive definite."""
    def __init__(self, S, A, M):
        from cvxopt import cholmod as _chol, spmatrix as _spm
        self._chol = _chol; self.M = M; self.n = S.shape[0]; self.p = 0 if A is None else A.shape[0]; self.backend = "cholmod"
        try:
            C = sp.tril(S, format="coo"); Sc = _spm(C.data.tolist(), C.row.tolist(), C.col.tolist(), S.shape)
            self.F = _chol.symbolic(Sc); _chol.numeric(Sc, self.F)
            if self.p:
                self.A = A.tocsr() if sp.issparse(A) else sp.csr_matrix(A)
                Z = matrix(np.asfortranarray(self.A.T.toarray())); _chol.solve(self.F, Z); self.SiAt = np.array(Z)
                Cs = self.A @ self.SiAt; self.Cf = sla.cho_factor(0.5 * (Cs + Cs.T), check_finite=False)
        except Exception as e:
            self.backend = f"superlu ({type(e).__name__})"; self.lu = spla.splu(sp.csc_matrix(M), permc_spec="MMD_AT_PLUS_A")
    def _S_solve(self, v):
        B = matrix(np.ascontiguousarray(v, dtype=float).reshape(-1, 1)); self._chol.solve(self.F, B); return np.array(B).ravel()
    def _solve_once(self, b):
        if self.backend != "cholmod": return self.lu.solve(b)
        n, p = self.n, self.p
        if not p: return self._S_solve(b)
        t = self._S_solve(b[:n]); uy = sla.cho_solve(self.Cf, self.A @ t - b[n:], check_finite=False)
        return np.concatenate([t - self.SiAt @ uy, uy])
    def solve(self, b):
        x = self._solve_once(b); return x + self._solve_once(b - self.M @ x)

def reduced_factor(S, A, M, sparse):
    return ReducedCholFactor(S, A, M) if (sparse and CONFIG["inexact"].get("use_cholmod", True)) else ExactFactor(M)

class ReducedExactKind:
    """Sparse direct reference: factor the reduced quasi-definite system [[S, A'], [A, 0]], S = P + Ghat'Ghat, instead of the
    full KKT matrix, whose sparse LU fills in badly (970 s per run at KKT dimension 9 920)."""
    def __init__(self, spec, K, asm, W, eng):
        assert asm.l_only
        inst = asm.inst; n, p = asm.n, asm.p; di = to_np(W["di"]); G = inst["G"]; P = inst["P"]
        if asm.sparse:
            self.Gh = (sp.diags(di) @ G).tocsr(); S = (P + self.Gh.T @ self.Gh)
            M = sp.bmat([[S, inst["A"].T], [inst["A"], None]], format="csc") if p else S.tocsc()
        else:
            Gd = G.toarray() if sp.issparse(G) else G; self.Gh = di[:, None] * Gd; S = (P.toarray() if sp.issparse(P) else P) + self.Gh.T @ self.Gh
            A = inst["A"].toarray() if (p and sp.issparse(inst["A"])) else inst["A"]
            M = np.block([[S, A.T], [A, np.zeros((p, p))]]) if p else S
        self.f = reduced_factor(S, inst["A"] if p else None, M, asm.sparse); self.n, self.p = n, p
    def base_solve(self, r, ctx):
        n, p = self.n, self.p; bx, by, bz = r[:n], r[n:n + p], r[n + p:]
        sol = self.f.solve(np.concatenate([bx + self.Gh.T @ bz, by])); ux = sol[:n]
        return np.concatenate([ux, sol[n:], self.Gh @ ux - bz]), {}
SOLVER_KINDS["exact_reduced"] = ReducedExactKind

class _ReducedFallback:
    """ReducedExactKind behind a .solve(r) interface: the classical fallback of every inexact method."""
    def __init__(self, K, asm, W, eng):
        self.impl = ReducedExactKind({"name": "fallback_reduced", "kind": "exact_reduced"}, K, asm, W, eng); self.backend = "reduced_" + str(getattr(self.impl.f, "backend", "lu"))
    def solve(self, r): return self.impl.base_solve(r, {})[0]


# Null-space target system. u_x = x_p + Z v with A x_p = r_y and Z an orthonormal basis of null(A);
# (Z'SZ) v = Z'(r_x + Gh' r_z - S x_p) with S = P + Gh'Gh SPD; u_z = Gh u_x - r_z; u_y = (AA')^-1 A (r_x - P u_x - Gh' u_z).
# The equality block is exact and the acceptance test is applied to the full KKT system. Z is dense (QR of A', once per
# instance), so for these methods the iteration counts and kappa are the quantities of interest, not the time.
def oss_basis(inst):
    if "_oss" not in inst:
        n, p = inst["n"], inst["p"]
        if p == 0: inst["_oss"] = {"Z": None, "Ad": None, "AAt": None}
        else:
            Ad = inst["A"].toarray() if sp.issparse(inst["A"]) else np.asarray(inst["A"], dtype=float); Q, _ = np.linalg.qr(Ad.T, mode="complete")
            inst["_oss"] = {"Z": np.ascontiguousarray(Q[:, p:]), "Ad": Ad, "AAt": sla.cho_factor(Ad @ Ad.T, check_finite=False)}
    return inst["_oss"]

class OSSKind:
    """KKT-solver kind 'oss_cg': CG on the Jacobi-scaled null-space SPD system, combined acceptance on the full KKT system."""
    def __init__(self, spec, K, asm, W, eng):
        assert asm.l_only, "null-space solver: linear cone only"
        inst = asm.inst; n, p = asm.n, asm.p
        if n > int(spec.get("max_n", CONFIG["inexact"].get("oss_max_n", 3200))): raise RuntimeError(f"oss: n = {n} above oss_max_n (dense null-space basis)")
        di = to_np(W["di"]); G = inst["G"]; P = inst["P"]
        self.Gh = (sp.diags(di) @ G).tocsr() if sp.issparse(G) else di[:, None] * np.asarray(G)
        self.S = (P + self.Gh.T @ self.Gh).tocsr() if sp.issparse(P) else (np.asarray(P) + self.Gh.T @ self.Gh)
        self.P = P; bs = oss_basis(inst); self.Z, self.Ad, self.AAt = bs["Z"], bs["Ad"], bs["AAt"]; self.n, self.p, self.eng = n, p, eng; self.dim = n - p
        SZ = self.S @ self.Z if p else None
        diag = np.einsum("ij,ij->j", self.Z, SZ) if p else np.asarray(self.S.diagonal() if sp.issparse(self.S) else np.diag(self.S)).ravel()
        self.dj = 1.0 / np.sqrt(np.maximum(diag, 1e-300))                           # Jacobi scaling of the reduced system
        self.maxiter = int(min(spec.get("minres_maxiter_cap", CONFIG["inexact"]["minres_maxiter_cap"]), CONFIG["inexact"]["minres_maxiter_factor"] * K.shape[0]))    # same cap as the augmented-system methods
        self.kappa_reduced = float("nan"); self.reduced_dim = self.dim
    def matrix(self):
        """Explicit Jacobi-scaled reduced matrix D Z'SZ D (dense; used for kappa and by the quantum OSS solver on small instances)."""
        M = (self.Z.T @ (self.S @ self.Z)) if self.p else (self.S.toarray() if sp.issparse(self.S) else np.asarray(self.S))
        return self.dj[:, None] * M * self.dj[None, :]
    def compute_kappa(self):
        if self.dim <= CONFIG["cond_limit"]: self.kappa_reduced = float(np.linalg.cond(self.matrix()))
        return self.kappa_reduced
    def matvec(self, v):
        w = self.dj * v; x = self.Z @ w if self.p else w; y = self.S @ x
        return self.dj * (self.Z.T @ y if self.p else y)
    def reduced_rhs(self, r):
        n, p = self.n, self.p; rx, ry, rz = r[:n], r[n:n + p], r[n + p:]
        xp_ = self.Ad.T @ sla.cho_solve(self.AAt, ry, check_finite=False) if p else np.zeros(n)   # particular solution of A x = r_y
        b = rx + self.Gh.T @ rz - (self.S @ xp_ if p else 0.0)
        return xp_, self.dj * (self.Z.T @ b if p else b)
    def full(self, v, r, xp_):
        n, p = self.n, self.p; rx, rz = r[:n], r[n + p:]; w = self.dj * v; ux = xp_ + (self.Z @ w if p else w); uz = self.Gh @ ux - rz
        uy = sla.cho_solve(self.AAt, self.Ad @ (rx - self.P @ ux - self.Gh.T @ uz), check_finite=False) if p else np.zeros(0)
        return np.concatenate([ux, uy, uz])
    def base_solve(self, r, ctx):
        xp_, b = self.reduced_rhs(r); op = spla.LinearOperator((self.dim, self.dim), matvec=self.matvec, dtype=float)
        stop = lambda vk: self.eng.criterion_holds(self.full(vk, r, xp_), r, ctx)
        v, info, its, tc = krylov_solve(op, b, "cg", self.maxiter, stop, CONFIG["inexact"]["check_every"])
        return self.full(v, r, xp_), {"krylov_iters": its, "krylov_info": info, "t_check": tc, "kappa_reduced": self.kappa_reduced, "reduced_dim": self.dim}
SOLVER_KINDS["oss_cg"] = OSSKind

class StaleLUKind:
    """Lazy refactorisation. The reduced system M_k = [[P + G'W_k^-2 G, A'], [A, 0]] is factorised only when needed; in
    between, the factors of an earlier IPM iteration are a right preconditioner for GMRES on the current M_k
    (M_k P^-1 y = b, u = P^-1 y, so the monitored residual is the true one). The matrix is refactorised when GMRES misses
    its tolerance within `stale_maxiter` iterations (the solve is then finished with the new factors) or when the previous
    iteration needed more than `stale_refresh_iters` iterations. Every direction still has to pass the acceptance test on
    the full KKT system. The state lives on the per-run assembler object."""
    def __init__(self, spec, K, asm, W, eng):
        assert asm.l_only
        inst = asm.inst; n, p = asm.n, asm.p; di = to_np(W["di"]); G = inst["G"]; P = inst["P"]
        if asm.sparse:
            self.Gh = (sp.diags(di) @ G).tocsr(); S = (P + self.Gh.T @ self.Gh)
            self.M = sp.bmat([[S, inst["A"].T], [inst["A"], None]], format="csc") if p else S.tocsc()
        else:
            Gd = G.toarray() if sp.issparse(G) else G; self.Gh = di[:, None] * Gd; S = (P.toarray() if sp.issparse(P) else P) + self.Gh.T @ self.Gh
            A = inst["A"].toarray() if (p and sp.issparse(inst["A"])) else inst["A"]
            self.M = np.block([[S, A.T], [A, np.zeros((p, p))]]) if p else S
        self.n, self.p, self.asm = n, p, asm; self.S = S; self.A = inst["A"] if p else None
        self.maxit = int(spec.get("stale_maxiter", 40)); self.refresh_at = int(spec.get("stale_refresh_iters", 15)); self.rtol = float(spec.get("stale_rtol", 1e-12))
        st = getattr(asm, "_stale", None)
        self.did_factor = st is None or st.get("refresh", True)
        if self.did_factor: self._factor_now()
        self.age = asm._stale["age"]; asm._stale["age"] += 1
    def _factor_now(self):
        self.asm._stale = {"f": reduced_factor(self.S, self.A, self.M, self.asm.sparse), "age": 0, "refresh": False}; self.fresh = True
    def base_solve(self, r, ctx):
        n, p = self.n, self.p; bx, by, bz = r[:n], r[n:n + p], r[n + p:]
        b = np.concatenate([bx + self.Gh.T @ bz, by]); f = self.asm._stale["f"]; its = 0; refactored = False
        if getattr(self, "fresh", False):
            sol = f.solve(b)
        else:
            cnt = [0]
            def _cb(_): cnt[0] += 1
            psolve = getattr(f, "_solve_once", f.solve)          # preconditioner application: no refinement step needed
            op = spla.LinearOperator(self.M.shape, matvec=lambda v: self.M @ psolve(v), dtype=float)
            y, info = spla.gmres(op, b, rtol=self.rtol, atol=0.0, restart=self.maxit, maxiter=1, callback=_cb, callback_type="pr_norm")
            its = cnt[0]; sol = psolve(y)
            bad = (info != 0) or (not np.all(np.isfinite(sol))) or np.linalg.norm(self.M @ sol - b) > 1e3 * self.rtol * max(np.linalg.norm(b), 1e-300)
            if bad:
                self._factor_now(); sol = self.asm._stale["f"].solve(b); refactored = True
            elif its > self.refresh_at:
                self.asm._stale["refresh"] = True                 # next IPM iteration starts with a fresh factorisation
        ux = sol[:n]
        return np.concatenate([ux, sol[n:], self.Gh @ ux - bz]), {"krylov_iters": its, "refactored_now": refactored, "stale_age": 0 if getattr(self, "fresh", False) else self.age}
SOLVER_KINDS["gmres_stalelu"] = StaleLUKind


# Latency-optimised MINRES: one host synchronisation per iteration, fused vector updates (cupy.fuse, checked against the
# plain code), explicit checks spaced as max(check_every, it // 16) (at most ~6 % extra iterations), one synchronisation per
# check, and a device-resident sparsity pattern (only K.data and the Ruiz vector are uploaded per factorisation).
_FUSED_CACHE = {}
def _lanczos_step_plain(Av, v, v_old, alpha, beta): return Av - alpha * v - beta * v_old
def _wx_step_plain(v, w_old, w, x, rho3, rho2, inv_rho1, ge):
    w_new = (v - rho3 * w_old - rho2 * w) * inv_rho1
    return w_new, x + ge * w_new
def fused_ops(xp):
    key = getattr(xp, "__name__", str(xp))
    if key in _FUSED_CACHE: return _FUSED_CACHE[key]
    out = (_lanczos_step_plain, _wx_step_plain, "plain")
    if xp is not np and CONFIG["inexact"].get("gpu_fuse", True) and hasattr(xp, "fuse"):
        try:
            f1, f2 = xp.fuse(_lanczos_step_plain), xp.fuse(_wx_step_plain)
            a, b_, c, x_ = (xp.asarray(np.random.default_rng(k_).standard_normal(257)) for k_ in range(4)); al = xp.vdot(a, b_)
            t1, t2 = f1(a, b_, c, al, 0.3), _lanczos_step_plain(a, b_, c, al, 0.3)
            u1, u2 = f2(a, b_, c, x_, 0.1, 0.2, 0.7, 1.3), _wx_step_plain(a, b_, c, x_, 0.1, 0.2, 0.7, 1.3)
            err = max(float(xp.max(xp.abs(t1 - t2))), float(xp.max(xp.abs(u1[0] - u2[0]))), float(xp.max(xp.abs(u1[1] - u2[1]))))
            if not err <= 1e-12: raise ValueError(f"fused kernels differ from plain code by {err:.1e}")
            out = (f1, f2, "cupy.fuse")
        except Exception as e:
            out = (_lanczos_step_plain, _wx_step_plain, f"plain (fuse unavailable: {type(e).__name__}: {e})"[:120])
    _FUSED_CACHE[key] = out
    return out

def minres_fused_kernel(matvec, b, xp, maxiter=1000, stop_fn=None, check_every=8, adaptive_div=16):
    """Paige-Saunders MINRES with the same arithmetic as minres_base_kernel, arranged for few host synchronisations.
    Returns (x, info, iterations, t_check, n_checks); info = 0 converged by stop_fn, 1 maxiter, 2 breakdown."""
    lanczos_step, wx_step, _ = fused_ops(xp); t_check = 0.0; n_checks = 0
    beta1 = float(xp.linalg.norm(b)); x = xp.zeros_like(b)
    if beta1 == 0.0: return x, 0, 0, t_check, n_checks
    v_old = xp.zeros_like(b); v = b * (1.0 / beta1); beta = beta1; w_old = xp.zeros_like(b); w = xp.zeros_like(b)
    eta = beta1; g0 = g1 = 1.0; s0 = s1 = 0.0; next_check = check_every
    for it in range(1, maxiter + 1):
        Av = matvec(v); alpha_d = xp.vdot(v, Av)
        v_new = lanczos_step(Av, v, v_old, alpha_d, beta)
        if xp is np: alpha, beta_new = float(alpha_d), float(np.linalg.norm(v_new))
        else:
            ab = xp.stack((alpha_d, xp.linalg.norm(v_new))); ab = ab.get() if hasattr(ab, "get") else ab  # the only synchronisation of the iteration
            alpha, beta_new = float(ab[0]), float(ab[1])
        delta = g1 * alpha - g0 * s1 * beta
        rho1 = math.hypot(delta, beta_new); rho2 = s1 * alpha + g0 * g1 * beta; rho3 = s0 * beta
        if rho1 == 0.0: return x, 2, it, t_check, n_checks
        g_new = delta / rho1; s_new = beta_new / rho1
        w_new, x = wx_step(v, w_old, w, x, rho3, rho2, 1.0 / rho1, g_new * eta)
        eta = -s_new * eta
        v_old, v = v, (v_new * (1.0 / beta_new) if beta_new > 0 else v_new)
        w_old, w = w, w_new; beta = beta_new; g0, g1 = g1, g_new; s0, s1 = s1, s_new
        if it >= next_check or abs(eta) <= 1e-15 * beta1 or beta_new == 0.0:
            t0 = time.perf_counter(); ok = bool(stop_fn(x)) if stop_fn is not None else False; t_check += time.perf_counter() - t0; n_checks += 1
            if ok: return x, 0, it, t_check, n_checks
            if beta_new == 0.0: return x, 2, it, t_check, n_checks
            next_check = it + max(check_every, it // max(int(adaptive_div), 1))
    return x, 1, maxiter, t_check, n_checks

class MinresFusedKind:
    """Ruiz-scaled MINRES with the latency-optimised kernel.
    backend = 'cpu' | 'gpu' | 'auto' ('auto' uses the GPU from CONFIG['inexact']['gpu_auto_min_dim'] on, else the CPU)."""
    def __init__(self, spec, K, asm, W, eng):
        be = spec.get("backend", "cpu"); N = K.shape[0]
        if be == "auto": be = "gpu" if (GPU_OK and N >= CONFIG["inexact"]["gpu_auto_min_dim"]) else "cpu"
        self.gpu = be == "gpu"
        if self.gpu and not GPU_OK: raise RuntimeError("GPU backend requested but no usable GPU")
        self.xp = DEVICE_XP if self.gpu else np; xp = self.xp; self.eng = eng
        d = ruiz_equilibrate(K) if spec.get("scaling", "ruiz") == "ruiz" else np.ones(N)
        t0 = time.perf_counter()
        if sp.issparse(K):
            Kc = K.tocsr(); Kc.sort_indices()
            if self.gpu:
                pat = getattr(asm, "_dev_pattern", None)
                if pat is None or pat["nnz"] != Kc.nnz or not (np.array_equal(pat["indptr_h"], Kc.indptr) and np.array_equal(pat["indices_h"], Kc.indices)):
                    ip = Kc.indptr.astype(np.int32); ix = Kc.indices.astype(np.int32)
                    pat = {"nnz": Kc.nnz, "indptr_h": Kc.indptr.copy(), "indices_h": Kc.indices.copy(), "indptr": xp.asarray(ip), "indices": xp.asarray(ix),
                           "rows": xp.asarray(np.repeat(np.arange(N, dtype=np.int32), np.diff(Kc.indptr))), "uploads": 0}
                    asm._dev_pattern = pat
                pat["uploads"] += 1; data_d = xp.asarray(Kc.data); d_d = xp.asarray(d)
                self.K = DEVICE_SPARSE.csr_matrix((data_d, pat["indices"], pat["indptr"]), shape=Kc.shape)
                self.Ks = DEVICE_SPARSE.csr_matrix((data_d * d_d[pat["rows"]] * d_d[pat["indices"]], pat["indices"], pat["indptr"]), shape=Kc.shape)
                self.d = d_d
            else:
                rows = np.repeat(np.arange(N), np.diff(Kc.indptr))
                self.K = Kc; self.Ks = sp.csr_matrix((Kc.data * d[rows] * d[Kc.indices], Kc.indices, Kc.indptr), shape=Kc.shape); self.d = d
        else:
            Ks = d[:, None] * K * d[None, :]
            self.K = xp.asarray(K) if self.gpu else K; self.Ks = xp.asarray(Ks) if self.gpu else Ks; self.d = xp.asarray(d)
        if self.gpu: dev_sync()
        self.t_transfer = time.perf_counter() - t0
        self.maxiter = int(min(spec.get("minres_maxiter_cap", CONFIG["inexact"]["minres_maxiter_cap"]), CONFIG["inexact"]["minres_maxiter_factor"] * N))
        self.floors = eng.block_floor; self.slices = [sl for sl in eng.block_slices if sl.stop > sl.start]; self.fl = [fl for sl, fl in zip(eng.block_slices, eng.block_floor) if sl.stop > sl.start]
        self.fuse_mode = fused_ops(xp)[2]
    def _holds(self, u, r, ctx):
        xp = self.xp; e = self.K @ u - r; nrm = xp.linalg.norm
        vals = xp.stack([nrm(e), nrm(r), nrm(u)] + [nrm(e[sl]) for sl in self.slices] + [nrm(r[sl]) for sl in self.slices])
        vals = np.asarray(vals.get() if hasattr(vals, "get") else vals, dtype=float)                       # one synchronisation per check
        ne, nr, nu = vals[:3]; nb = len(self.slices)
        if not np.isfinite(ne): return False
        ratios = [vals[3 + j] / max(vals[3 + nb + j], self.fl[j]) for j in range(nb)]
        den = self.eng.nK * nu + nr
        m = {"relres": ne / nr if nr > 0 else ne, "backward_err": ne / den if den > 0 else ne, "block_ratio_max": max(ratios) if ratios else 0.0}
        return self.eng.criterion_value(m, ctx) <= self.eng.threshold(ctx)
    def base_solve(self, r, ctx):
        xp = self.xp; r_d = xp.asarray(r)
        xs, info, its, tc, nchk = minres_fused_kernel(lambda v: self.Ks @ v, self.d * r_d, xp, maxiter=self.maxiter, stop_fn=lambda xk: self._holds(self.d * xk, r_d, ctx),
                                            check_every=CONFIG["inexact"]["check_every"], adaptive_div=CONFIG["inexact"]["check_adaptive_div"])
        u_d = self.d * xs
        if self.gpu: dev_sync()
        return (to_host(u_d) if self.gpu else u_d), {"krylov_iters": its, "krylov_info": info, "t_check": tc, "n_checks": nchk, "backend": "gpu" if self.gpu else "cpu",
                                                     "kernel": "fused", "fuse_mode": self.fuse_mode, "t_transfer": self.t_transfer}
SOLVER_KINDS["minres_fused"] = MinresFusedKind

# self-test of the MINRES kernel against a dense solve on a symmetric indefinite system
_r = rng_for("minres_selftest"); _Q, _ = np.linalg.qr(_r.standard_normal((60, 60))); _ev = np.concatenate([np.linspace(1, 50, 40), -np.linspace(1, 30, 20)])
_Am = (_Q * _ev) @ _Q.T; _bv = _r.standard_normal(60); _xe = np.linalg.solve(_Am, _bv)
_xm, _info, _its, _ = minres_base_kernel(lambda v: _Am @ v, _bv, np, maxiter=400, stop_fn=lambda x: np.linalg.norm(_Am @ x - _bv) <= 1e-11 * np.linalg.norm(_bv), check_every=4)
assert np.linalg.norm(_xm - _xe) <= 1e-8 * np.linalg.norm(_xe), "minres_base_kernel failed its self-test"
_xw, _, _its_w, _ = minres_base_kernel(lambda v: _Am @ v, _bv, np, x0=_xe * (1 + 1e-3), maxiter=400, stop_fn=lambda x: np.linalg.norm(_Am @ x - _bv) <= 1e-11 * np.linalg.norm(_bv), check_every=4)
print(f"minres_base_kernel self-test passed: {_its} iterations cold, {_its_w} warm-started; rel. error {np.linalg.norm(_xm - _xe) / np.linalg.norm(_xe):.1e}")
if GPU_OK:
    _xg, _, _itg, _ = minres_base_kernel(lambda v: DEVICE_XP.asarray(_Am) @ v, DEVICE_XP.asarray(_bv), DEVICE_XP, maxiter=400, stop_fn=lambda x: float(DEVICE_XP.linalg.norm(DEVICE_XP.asarray(_Am) @ x - DEVICE_XP.asarray(_bv))) <= 1e-11 * np.linalg.norm(_bv), check_every=4)
    assert np.linalg.norm(to_host(_xg) - _xe) <= 1e-8 * np.linalg.norm(_xe); print(f"  device MINRES matches ({_itg} iterations)")
_x5, _i5, _it5, _, _nc5 = minres_fused_kernel(lambda v: _Am @ v, _bv, np, maxiter=400, stop_fn=lambda x: np.linalg.norm(_Am @ x - _bv) <= 1e-11 * np.linalg.norm(_bv), check_every=4)
assert np.linalg.norm(_x5 - _xe) <= 1e-8 * np.linalg.norm(_xe), "minres_fused_kernel failed its self-test"
print(f"minres_fused_kernel self-test passed: {_it5} iterations, {_nc5} explicit checks (minres_base_kernel: {_its} iterations, {_its // 4} checks)")
if GPU_OK:
    _Ad, _bd = DEVICE_XP.asarray(_Am), DEVICE_XP.asarray(_bv)
    _xg5, _, _itg5, _, _ = minres_fused_kernel(lambda v: _Ad @ v, _bd, DEVICE_XP, maxiter=400, stop_fn=lambda x: float(DEVICE_XP.linalg.norm(_Ad @ x - _bd)) <= 1e-11 * np.linalg.norm(_bv), check_every=4)
    assert np.linalg.norm(to_host(_xg5) - _xe) <= 1e-8 * np.linalg.norm(_xe), "device minres_fused_kernel failed its self-test"
    GPU_STATUS["fuse_mode"] = fused_ops(DEVICE_XP)[2]; print(f"  device minres_fused_kernel matches ({_itg5} iterations); vector kernels: {GPU_STATUS['fuse_mode']}")
print("solver kinds:", list(SOLVER_KINDS))

## 9. Instrumented `kktsolver` and the benchmark runner

`make_kktsolver` wraps a solver in the callback contract of Section 3.2. For every factorisation it records the assembly time, $\kappa_2(\tilde K)$ (reference runs only) and the spread of the NT scaling; for every solve, the test values before and after refinement, refinements, fallback, Krylov iterations and the timing components.

`run_instance` calls `solvers.coneqp` (or `conelp`) with the same algorithm parameters for every method, parses the per-iteration table that CVXOPT prints, and recomputes the feasibility and optimality measures from the returned iterate. This includes a valid duality gap: the Lagrange dual function evaluated at $z_+ = \max(z, 0)$ is a lower bound on the optimal value when $P \succ 0$, so `relgap_valid` does not depend on the solver's own report.

A run is classified at three levels:

* `accepted`: the final iterate passes the thresholds in `CONFIG["accept"]`;
* `pure_method_success`: accepted with no classical fallback;
* `strict_method_success`: in addition, every direction passed the acceptance test (only relevant when fallback is disabled).

Times are wall-clock seconds (`time.perf_counter`). `wall_total` is the whole `coneqp` call and `wall_method = wall_total - t_diag` excludes the condition-number diagnostic. `t_assemble`, `t_factor`, `t_solve`, `t_verify`, `t_refine` and `t_fallback` are summed over all callback calls, so `wall_method` minus their sum is the time spent in CVXOPT's own iteration.

`method_specs` defines every classical method; the quantum methods are added in Sections 16 and 20.

In [ ]:
# Instrumented kktsolver, the benchmark runner, recomputed metrics and the acceptance of a run.
CAPTURED = {}

def make_kktsolver(inst, spec, run_state):
    asm = KKTAssembler(inst); run_state["asm"] = asm
    compute_cond = run_state.get("compute_cond", False) and asm.dim <= CONFIG["cond_limit"]
    capture = run_state.get("capture", False) and asm.dim <= CONFIG["capture"]["max_dim"]
    def kktsolver(W):
        fi = run_state["factor_calls"]; run_state["factor_calls"] += 1
        with Timer() as t_asm:
            K = asm.build(W)
        proxy = nt_spread(W)
        kappa, kappa_scaled, t_diag = float("nan"), float("nan"), 0.0
        if compute_cond:
            with Timer() as td:
                kappa = cond_exact(K)
                if spec.get("scaling", "ruiz") == "ruiz" or run_state.get("cond_scaled"):
                    kappa_scaled = cond_exact(apply_scaling(K, ruiz_equilibrate(K)))
            t_diag = td.t
        kappa_S, kappa_B = float("nan"), float("nan")
        if compute_cond and asm.l_only and asm.p == 0 and inst.get("P") is not None:          # kappa of the reduced system S and of its square root B
            with Timer() as td2:
                try:
                    S_, _, _ = reduced_spd(asm, W); Sd = S_.toarray() if sp.issparse(S_) else S_
                    kappa_S = float(np.linalg.cond(Sd)); kappa_B = float(np.sqrt(kappa_S))      # kappa(B) = sqrt(kappa(S)) for S = B'B
                except Exception:
                    pass
            t_diag += td2.t
        switched = bool(spec.get("switch_policy")) and run_state.get("switched", False)
        eff_spec = dict(spec, kind="exact", name=spec["name"]) if switched else spec
        with Timer() as t_fac:
            engine = KKTSolveEngine(eff_spec, K, asm, W)
        kappa_reduced = float("nan")
        if compute_cond and hasattr(engine.impl, "compute_kappa"):        # kappa of the method's own target system (null-space methods)
            with Timer() as td3:
                try: kappa_reduced = float(engine.impl.compute_kappa())
                except Exception: pass
            t_diag += td3.t
        frec = {"factor_index": fi, "t_assemble": t_asm.t, "t_factor": t_fac.t, "t_diag": t_diag, "kappa": kappa, "kappa_scaled": kappa_scaled, "kappa_S": kappa_S, "kappa_B": kappa_B, "kappa_reduced": kappa_reduced,
                "nt_proxy": proxy, "kkt_dim": asm.dim, "nK_est": engine.nK, "did_factor": bool(getattr(engine.impl, "did_factor", True))}
        run_state["factors"].append(frec)
        if capture:
            Kd = K.toarray() if sp.issparse(K) else K.copy()
            run_state["captured"].append({"factor_index": fi, "K": Kd, "kappa": kappa, "nt_proxy": proxy, "di": to_np(W["di"]).copy() if W["di"].size[0] else np.zeros(0), "solves": []})
        def solve(x, y, z):
            si = run_state["solve_calls"]; run_state["solve_calls"] += 1
            r = asm.rhs(x, y, z, W)
            u, log = engine.solve(r, {"factor_index": fi, "solve_index": si, "nt_proxy": proxy})
            log.update({"factor_index": fi, "solve_index": si, "kappa": kappa, "nt_proxy": proxy, "after_switch": switched})
            if switched: log["method_used"] = "exact_after_switch"
            if spec.get("switch_policy") and not switched and (log["fallback"] or not log["accepted"]):
                run_state["switched"] = True                     # first failed acceptance -> exact solver for the rest of the run
            run_state["solves"].append(log)
            if log["solve_failed"]:
                raise ArithmeticError("non-finite KKT direction")
            asm.write_back(u, x, y, z)
            if capture:
                run_state["captured"][-1]["solves"].append({"rhs": r.copy(), "u": u.copy(), "solve_index": si})
        return solve
    return kktsolver

_PROG_RE = re.compile(r"^\s*(\d+):\s+(.*)$")
_FLOAT_RE = re.compile(r"[-+]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][-+]?\d+)?")
def parse_progress(text):
    rows = []
    for line in text.splitlines():
        m = _PROG_RE.match(line)
        if not m: continue
        vals = [float(v) for v in _FLOAT_RE.findall(m.group(2))]
        if len(vals) < 5: continue
        rows.append({"iter": int(m.group(1)), "pcost": vals[0], "dcost": vals[1], "gap": vals[2], "pres": vals[3], "dres": vals[4], "k_over_t": vals[5] if len(vals) > 5 else float("nan")})
    return rows

def recompute_metrics_qp(inst, sol):
    P, q, G, h, A, b = inst["P"], inst["q"], inst["G"], inst["h"], inst["A"], inst["b"]
    x, s, z = to_np(sol["x"]), to_np(sol["s"]), to_np(sol["z"])
    y = to_np(sol["y"]) if inst["p"] > 0 else np.zeros(0)
    Px = P @ x
    pobj = 0.5 * x @ Px + q @ x
    r_dual = Px + q + G.T @ z + (A.T @ y if inst["p"] > 0 else 0.0)
    r_eq = (A @ x - b) if inst["p"] > 0 else np.zeros(0)
    r_in = G @ x + s - h
    resy0 = max(1.0, np.linalg.norm(b)) if inst["p"] > 0 else 1.0
    resz0 = max(1.0, np.linalg.norm(h))
    pres = max(np.linalg.norm(r_eq) / resy0, np.linalg.norm(r_in) / resz0)
    dres = np.linalg.norm(r_dual) / max(1.0, np.linalg.norm(q))
    cone_viol = max(0.0, -s.min() if s.size else 0.0, -z.min() if z.size else 0.0)
    constr_viol = max(0.0, float((G @ x - h).max())) / resz0 if inst["m"] > 0 else 0.0
    gap = float(s @ z)
    zp = np.maximum(z, 0.0); v = q + G.T @ zp + (A.T @ y if inst["p"] > 0 else 0.0)
    try:
        if inst.get("no_dual_recompute"): raise ValueError("dual value not recomputed for this instance (P may be singular)")
        Pd = P.toarray() if sp.issparse(P) else P
        if Pd.shape[0] <= 3000:
            c, low = sla.cho_factor(Pd); dual_val = -0.5 * v @ sla.cho_solve((c, low), v) - h @ zp - (b @ y if inst["p"] > 0 else 0.0)
        else:
            f = spla.splu(sp.csc_matrix(P)); dual_val = -0.5 * v @ f.solve(v) - h @ zp - (b @ y if inst["p"] > 0 else 0.0)
        valid_gap = pobj - dual_val
    except Exception:
        dual_val, valid_gap = float("nan"), float("nan")
    out = {"pobj": float(pobj), "dobj_valid": float(dual_val), "gap_rec": gap, "relgap_valid": float(abs(valid_gap) / max(1.0, abs(pobj))),
           "pres_rec": float(pres), "dres_rec": float(dres), "cone_viol": float(cone_viol), "constraint_viol": float(constr_viol), "mu_final": gap / max(1, inst["m"])}
    if inst.get("known_solution"):
        out["obj_rel_err"] = float(abs(pobj - inst["f_star"]) / max(1.0, abs(inst["f_star"]))); out["x_rel_err"] = float(np.linalg.norm(x - inst["x_star"]) / max(1e-12, np.linalg.norm(inst["x_star"])))
    else:
        out["obj_rel_err"] = float("nan"); out["x_rel_err"] = float("nan")
    return out
METRIC_FUNCS = {"coneqp": recompute_metrics_qp}

def acceptance(rec, overrides=None):
    a = dict(CONFIG["accept"], **(overrides or {})); fails = []           # per-run overrides: tolerance sweep E4b
    if not (rec.get("pres_rec", np.inf) <= a["pres"]): fails.append("pres")
    if not (rec.get("dres_rec", np.inf) <= a["dres"]): fails.append("dres")
    rg = rec.get("relgap_valid", np.nan)
    if np.isnan(rg): rg = rec.get("relgap_cvx", np.nan)
    if np.isnan(rg) and np.isfinite(rec.get("gap_rec", np.nan)): rg = abs(rec["gap_rec"]) / max(1.0, abs(rec.get("pobj", 0.0)))     # CVXOPT reports no relative gap when pcost ~ 0
    if not (rg <= a["relgap"]): fails.append("relgap")
    if not (rec.get("cone_viol", np.inf) <= a["cone_viol"]): fails.append("cone")
    oe = rec.get("obj_rel_err", np.nan)
    if not np.isnan(oe) and not (oe <= a["obj_rel_err"]): fails.append("obj")
    if rec.get("error"): fails.append("error")
    return (len(fails) == 0), ",".join(fails)

def solver_options_for(inst, spec):
    c = dict(CONFIG["cvxopt"], **spec.get("cvxopt_overrides", {}))          # per-run tolerance overrides (E4b)
    return {"show_progress": True, "maxiters": spec.get("maxiters", c["maxiters"]), "abstol": c["abstol"], "reltol": c["reltol"], "feastol": c["feastol"],
            "refinement": c["refinement_qp"] if not inst["dims"]["s"] and not inst["dims"]["q"] else c["refinement_sdp"]}

def run_instance(inst, spec, repeat=0, compute_cond=False, capture=False, cond_scaled=False, experiment=""):
    run_id = f"{inst['id']}__{spec['name']}__r{repeat}"
    run_state = {"factor_calls": 0, "solve_calls": 0, "factors": [], "solves": [], "captured": [], "compute_cond": compute_cond, "capture": capture, "cond_scaled": cond_scaled}
    kkt = None if spec["kind"] == "baseline" else make_kktsolver(inst, spec, run_state)
    opts = solver_options_for(inst, spec)
    solvers.options.clear(); solvers.options.update(opts)
    cd = cvx_data(inst); buf = io.StringIO(); err = None; sol = None; singular = False
    t0 = time.perf_counter(); pm = PeakMemory()
    try:
        with contextlib.redirect_stdout(buf), pm:
            if inst["kind"] == "coneqp":
                sol = solvers.coneqp(cd["P"], cd["q"], cd["G"], cd["h"], inst["dims"], cd["A"], cd["b"], kktsolver=kkt)
            else:
                sol = solvers.conelp(cd["q"], cd["G"], cd["h"], inst["dims"], cd["A"], cd["b"], kktsolver=kkt)
    except ArithmeticError as e:
        err = (f"cvxopt ArithmeticError (its own factorisation failed): {e}" if kkt is None else f"terminated: non-finite KKT direction ({e})")[:300]; singular = True
    except Exception as e:
        err = f"{type(e).__name__}: {e}"[:300]
    wall = time.perf_counter() - t0
    solvers.options.clear()
    prog = parse_progress(buf.getvalue())
    status = sol["status"] if sol else (("cvxopt_arithmetic_error" if kkt is None else "unknown_singular_kkt") if singular else "error")
    rec = {"run_id": run_id, "code_version": CODE_VERSION, "config_hash": CONFIG_HASH, "experiment": experiment, "family": inst["family"], "instance_id": inst["id"], "kind": inst["kind"],
           "structure": inst.get("structure", ""), "n": inst["n"], "m": inst["m"], "p": inst["p"], "kkt_dim": inst["kkt_dim"], "nnz_kkt": inst.get("nnz_kkt", np.nan),
           "kappa_P_target": inst.get("kappa_P_target"), "kappa_P_measured": inst.get("kappa_P_measured"), "density": inst.get("density"), "seed": inst.get("seed"),
           "method": spec["name"], "method_kind": spec["kind"], "criterion": spec.get("criterion", "" if spec["kind"] in ("baseline", "exact") else "backward"),
           "tol_rule": spec.get("tol_rule", ""), "target_tol": spec.get("tol", spec.get("eta", np.nan)), "scaling": spec.get("scaling", "none" if spec["kind"] in ("baseline", "exact", "noise") else "ruiz"),
           "regularization": spec.get("reg", 0.0), "fallback_enabled": bool(spec.get("fallback", CONFIG["inexact"]["fallback"])), "repeat": repeat,
           "refinement_option": opts["refinement"], "maxiters_option": opts["maxiters"], "status": status, "ipm_tol_level": float(spec.get("tol_level", CONFIG["cvxopt"]["feastol"])),
           "feastol_option": opts["feastol"], "sweep_samples": spec.get("sweep_samples", np.nan), "sweep_seed": spec.get("sweep_seed", np.nan), "solution_source": inst.get("solution_source", "generator"),
           "iterations": int(sol["iterations"]) if sol else (prog[-1]["iter"] if prog else -1), "error": err or "", "wall_total": wall, **pm.fields()}
    if sol is not None:
        rec.update(METRIC_FUNCS[inst["kind"]](inst, sol))
        rec.update({"gap_cvx": safe_float(sol.get("gap")), "relgap_cvx": safe_float(sol.get("relative gap")), "pres_cvx": safe_float(sol.get("primal infeasibility")), "dres_cvx": safe_float(sol.get("dual infeasibility"))})
    F, S = run_state["factors"], run_state["solves"]
    for k in ["t_assemble", "t_factor", "t_diag"]: rec[k] = float(sum(f[k] for f in F))
    for k in ["t_solve", "t_verify", "t_refine", "t_fallback", "t_check"]: rec[k] = float(sum(s.get(k, 0.0) for s in S))
    rec["wall_method"] = wall - rec["t_diag"]
    rec["n_factor_calls"] = len(F); rec["n_solve_calls"] = len(S)
    rec["n_refine_total"] = int(sum(s["n_refine"] for s in S)); rec["n_fallback_total"] = int(sum(s["fallback"] for s in S))
    rec["n_solves_accepted_without_fallback"] = int(sum((s["accepted"] and not s["fallback"]) for s in S)); rec["n_solves_failed"] = int(sum(s.get("solve_failed", False) for s in S))
    rec["n_solves_accepted_direct"] = int(sum((s["accepted"] and not s["fallback"] and s["n_refine"] == 0) for s in S))
    rec["n_solves_accepted_refined"] = int(sum((s["accepted"] and not s["fallback"] and s["n_refine"] > 0) for s in S))
    rec["n_solves_unaccepted_used"] = int(sum(((not s["accepted"]) and not s["fallback"] and not s.get("solve_failed", False)) for s in S))   # only possible when fallback is disabled
    rec["max_rel_resid_final"] = float(max([s["rel_resid_final"] for s in S], default=np.nan)); rec["max_backward_err_final"] = float(max([s.get("backward_err_final", np.nan) for s in S], default=np.nan))
    rec["max_block_ratio_final"] = float(max([s.get("block_ratio_max_final", np.nan) for s in S], default=np.nan))
    rec["krylov_iters_total"] = int(sum(s.get("krylov_iters", 0) for s in S))
    n_after = int(sum(bool(s.get("after_switch", False)) for s in S)); n_own = len(S) - n_after - rec["n_fallback_total"]
    rec["n_solves_after_switch"] = n_after; rec["n_solves_by_method"] = int(max(n_own, 0))
    rec["utilisation"] = float(max(n_own, 0) / len(S)) if S else np.nan        # fraction of KKT solves delivered by the method under test (Amdahl bound on any kernel speed-up)
    rec["n_factorizations"] = int(sum(f.get("did_factor", True) for f in F) + sum(bool(s.get("refactored_now", False)) for s in S)) if spec["kind"] != "baseline" else int(rec["iterations"]) + 1
    rec["warm_started_solves"] = int(sum(bool(s.get("warm_started", False)) for s in S)); rec["backend"] = spec.get("backend", "cpu")
    rec["backend_used"] = ",".join(sorted({str(s_.get("backend")) for s_ in S if s_.get("backend") is not None})) or "cpu"; rec["n_checks_total"] = int(sum(s_.get("n_checks", 0) or 0 for s_ in S))
    rec["fuse_mode"] = next((s_.get("fuse_mode") for s_ in S if s_.get("fuse_mode")), ""); rec["t_transfer"] = float(sum({s_["factor_index"]: (s_.get("t_transfer") or 0.0) for s_ in S}.values()))
    kap = [f["kappa"] for f in F if np.isfinite(f["kappa"])]
    rec["kappa_first"] = kap[0] if kap else np.nan; rec["kappa_last"] = kap[-1] if kap else np.nan; rec["kappa_max"] = max(kap) if kap else np.nan
    kaps = [f["kappa_scaled"] for f in F if np.isfinite(f["kappa_scaled"])]; rec["kappa_scaled_max"] = max(kaps) if kaps else np.nan
    kapr = [f.get("kappa_reduced", np.nan) for f in F if np.isfinite(f.get("kappa_reduced", np.nan))]; rec["kappa_reduced_max"] = max(kapr) if kapr else np.nan
    rec["n_fallback_reduced"] = int(sum(1 for s in S if s.get("fallback_backend"))); rec["reduced_dim"] = next((s.get("reduced_dim") for s in S if s.get("reduced_dim") is not None), np.nan)
    rec["accepted"], rec["accept_fail_reasons"] = acceptance(rec, spec.get("accept_overrides"))
    rec["pure_method_success"] = bool(rec["accepted"] and rec["n_fallback_total"] == 0)                      # no classical fallback
    rec["strict_method_success"] = bool(rec["accepted"] and rec["n_fallback_total"] == 0 and rec["n_solves_unaccepted_used"] == 0)   # ... and every direction passed the acceptance test
    REC.add("benchmark_results", rec)
    fmap = {f["factor_index"]: f for f in F}; smap = {}
    for s in S: smap.setdefault(s["factor_index"], []).append(s)
    for row in prog:
        f = fmap.get(row["iter"] + 1, {}); ss = smap.get(row["iter"] + 1, [])
        REC.add("iteration_logs", {"run_id": run_id, "instance_id": inst["id"], "method": spec["name"], "repeat": repeat, **row, "kappa": f.get("kappa", np.nan), "kappa_scaled": f.get("kappa_scaled", np.nan), "kappa_S": f.get("kappa_S", np.nan), "kappa_B": f.get("kappa_B", np.nan), "kappa_reduced": f.get("kappa_reduced", np.nan),
                                   "nt_proxy": f.get("nt_proxy", np.nan), "n_solves": len(ss), "n_fallback": int(sum(s["fallback"] for s in ss)), "n_refine": int(sum(s["n_refine"] for s in ss)),
                                   "max_rel_resid_final": max([s["rel_resid_final"] for s in ss], default=np.nan), "max_backward_err_final": max([s.get("backward_err_final", np.nan) for s in ss], default=np.nan),
                                   "max_block_ratio_final": max([s.get("block_ratio_max_final", np.nan) for s in ss], default=np.nan)})
    for s in S:
        REC.add("linear_solve_logs", {"run_id": run_id, "instance_id": inst["id"], "method": spec["name"], "method_kind": spec["kind"], "repeat": repeat, **s})
    if err or not rec["accepted"]:
        REC.failure(run_id, "optimization", err or f"acceptance failed: {rec['accept_fail_reasons']}", method=spec["name"], instance_id=inst["id"], status=rec["status"], n_fallback=rec["n_fallback_total"])
    if capture and run_state["captured"]:
        CAPTURED[inst["id"]] = run_state["captured"]
    REC.flush()
    return rec, sol, run_state

def method_specs():
    cfg = CONFIG["inexact"]; specs = [{"name": "cvxopt_default", "kind": "baseline"}, {"name": "custom_exact", "kind": "exact"}]
    # relative-residual test, with and without Ruiz scaling
    specs.append({"name": "minres_relres_1e-06_noscale", "kind": "minres", "criterion": "relres", "tol_rule": "fixed", "tol": cfg["relres_tol"], "scaling": "none"})
    specs.append({"name": "minres_relres_1e-06", "kind": "minres", "criterion": "relres", "tol_rule": "fixed", "tol": cfg["relres_tol"], "scaling": "ruiz"})
    # backward-error test: tolerance sweep and an unscaled ablation
    for t in cfg["be_tols"]:
        specs.append({"name": f"minres_be_{t:.0e}", "kind": "minres", "criterion": "backward", "tol_rule": "fixed", "tol": t, "scaling": "ruiz"})
    specs.append({"name": "minres_be_1e-10_noscale", "kind": "minres", "criterion": "backward", "tol_rule": "fixed", "tol": 1e-10, "scaling": "none"})
    # relative residual or a backward error at direct-solver level
    specs.append({"name": "minres_rrfloor_1e-06", "kind": "minres", "criterion": "relres_floor", "tol_rule": "fixed", "tol": cfg["relres_tol"], "be_floor": cfg.get("be_floor", 1e-14), "scaling": "ruiz"})
    specs.append({"name": "minres_rrfloor_1e-06_noscale", "kind": "minres", "criterion": "relres_floor", "tol_rule": "fixed", "tol": cfg["relres_tol"], "be_floor": cfg.get("be_floor", 1e-14), "scaling": "none"})
    # combined test: block forcing and backward error
    specs.append({"name": "minres_comb_0.1_1e-10", "kind": "minres", "criterion": "combined", "eta": 0.1, "tol_rule": "fixed", "tol": 1e-10, "scaling": "ruiz"})
    # block forcing: eta sweep and an unscaled ablation
    for eta in cfg["block_etas"]:
        specs.append({"name": f"minres_block_{eta:g}", "kind": "minres", "criterion": "block", "eta": eta, "scaling": "ruiz"})
    specs.append({"name": "minres_block_0.1_noscale", "kind": "minres", "criterion": "block", "eta": 0.1, "scaling": "none"})
    # backward-error tolerance tied to the NT-scaling spread
    specs.append({"name": "minres_adaptive_nt_be", "kind": "minres", "criterion": "backward", "tol_rule": "adaptive_nt", "scaling": "ruiz", **cfg["adaptive_nt"]})
    specs.append({"name": "cg_reduced_be_1e-10", "kind": "cg_reduced", "criterion": "backward", "tol_rule": "fixed", "tol": 1e-10, "scaling": "ruiz", "requires_p0": True})
    # methods with the combined test
    comb = {"criterion": "combined", "eta": 0.1, "tol_rule": "fixed", "tol": 1e-10, "scaling": "ruiz"}
    specs.append({"name": "custom_exact_reduced", "kind": "exact_reduced"})
    specs.append({"name": "gmres_stalelu_comb", "kind": "gmres_stalelu", **comb})
    specs.append({"name": "minres_base_comb_cpu", "kind": "minres_base", "backend": "cpu", "warm_start": False, **comb})
    specs.append({"name": "minres_base_comb_ws_cpu", "kind": "minres_base", "backend": "cpu", "warm_start": True, **comb})
    specs.append({"name": "minres_base_comb_ws_switch_cpu", "kind": "minres_base", "backend": "cpu", "warm_start": True, "switch_policy": True, **comb})
    specs.append({"name": "minres_fused_comb_cpu", "kind": "minres_fused", "backend": "cpu", **comb})
    # the y-block test binds late in the run (r_y ~ 1e-15, absolute target 0.1 * floor): ablation with floor 1e-8 instead of 1e-9
    specs.append({"name": "minres_fused_comb_floor8_cpu", "kind": "minres_fused", "backend": "cpu", "block_abs_floor": 1e-8, **comb})
    specs.append({"name": "minres_fused_comb_switch_cpu", "kind": "minres_fused", "backend": "cpu", "switch_policy": True, **comb})    # exact solver after the first rejected direction
    specs.append({"name": "oss_cg_comb", "kind": "oss_cg", "max_n": cfg.get("oss_max_n", 3200), **comb})                        # CG on the null-space system
    specs.append({"name": "minres_fused_comb_gpu", "kind": "minres_fused", "backend": "gpu", "requires_gpu": True, **comb})
    specs.append({"name": "minres_fused_comb_gpu_mm", "kind": "minres_fused", "backend": "gpu", "requires_gpu": True, "min_kkt_dim": 2000, **comb})   # Maros-Meszaros runs: GPU only from KKT dimension 2000
    specs.append({"name": "minres_fused_comb_auto", "kind": "minres_fused", "backend": "auto", "requires_gpu": True, **comb})       # GPU from gpu_auto_min_dim on, CPU below
    specs.append({"name": "minres_base_comb_gpu", "kind": "minres_base", "backend": "gpu", "warm_start": False, "requires_gpu": True, **comb})
    specs.append({"name": "minres_base_comb_ws_gpu", "kind": "minres_base", "backend": "gpu", "warm_start": True, "requires_gpu": True, **comb})
    specs.append({"name": "lsqr_sqrt_comb", "kind": "lsqr_sqrt", "requires_p0": True, "requires_dense_P": True, **comb})
    for d in cfg["noise_iso"]:
        specs.append({"name": f"exact_noise_iso_{d:.0e}", "kind": "noise", "noise_model": "iso", "delta": d, "fallback": False, "max_refine": 0, "criterion": "relres"})
    for d in cfg["noise_block"]:
        specs.append({"name": f"exact_noise_block_{d:.0e}", "kind": "noise", "noise_model": "block", "delta": d, "fallback": False, "max_refine": 0, "criterion": "block"})
    return {s["name"]: s for s in specs}

SPECS = method_specs()
print(f"{len(SPECS)} methods:", ", ".join(SPECS))

## 10. Classical experiments

* **E1, contract check.** `custom_exact` must reproduce the iterations and objective of the unmodified solver on the tiny instances. Their KKT systems are captured here for the quantum experiments.
* **E2, acceptance-test ablation** on `dense_kappa` and `dense_noeq`: the unmodified solver, the exact references, the relative-residual, backward-error, block and combined tests with and without Ruiz scaling, CG on the reduced system, LSQR on the square-root system, lazy refactorisation, and both MINRES kernels. `full_ablation` (on in `medium` and `large`) runs every method defined in Section 9; otherwise an essential subset.
* **E3, scaling** on `sparse_banded`, `sparse_lowfill` and `sparse_scale`, with several seeds per size (fewer at the largest sizes), the null-space CG up to $n = 3\,200$ and peak memory for every run.
* **E4, precision of a direction**: exact solves with an injected isotropic error ($\delta = 10^{-8} \dots 10^{-6}$) or block-relative error ($\delta = 10^{-1} \dots 10^{-3}$).

Time budgets are per method. When a method exceeds `max_run_seconds` on one instance, it is skipped on the larger instances of the same family, and a power law through the last two sizes skips a run that would exceed the budget before it starts. The other methods are not affected. A running CVXOPT call cannot be interrupted, so the budget acts between runs.

In [ ]:
# Classical experiments: E1 contract check, E2 acceptance-test ablation, E3 scaling, E4 direction-error sensitivity.
def run_methods(insts, method_names, experiment, repeats=None, compute_cond_for=("custom_exact",), capture_ids=(), budget=None, predictive=None):
    """Runs methods x instances. Budgets are per method: a method that exceeds `budget` seconds on an instance is skipped only for
    larger instances of the same family; other methods are unaffected. The exact LU reference is skipped above limits.exact_max_dim.
    A running CVXOPT call cannot be interrupted safely (no subprocess isolation is used), so budgets act between runs and every run is flushed."""
    repeats = CONFIG["repeats"] if repeats is None else repeats
    budget = CONFIG["limits"]["max_run_seconds"] if budget is None else budget
    exact_max = CONFIG["limits"].get("exact_max_dim", 10 ** 9)
    gpu_only = [n_ for n_ in method_names if SPECS.get(n_, {}).get("requires_gpu")]
    if gpu_only and not GPU_OK:
        REC.status(f"{experiment}_gpu_methods", "skipped", f"{gpu_only}: device={CONFIG['device']}, GPU usable={GPU_OK} {GPU_STATUS.get('reason', '')}".strip())
        method_names = [n_ for n_ in method_names if n_ not in gpu_only]
    t_exp = time.perf_counter(); n_runs = 0; skipped = []; slow = {}; hist = {}          # (family, method) -> kkt_dim at which it exceeded the budget
    for inst in sorted(insts, key=lambda i: (i["family"], i["kkt_dim"])):
        for name in method_names:
            if name not in SPECS:
                REC.status(f"{experiment}_{name}", "skipped", "method not defined in this profile"); continue
            spec = SPECS[name]
            if spec.get("requires_p0") and inst["p"] > 0: continue
            if spec.get("requires_dense_P") and inst["n"] > 2000: continue
            if spec.get("max_n") and inst["n"] > spec["max_n"]: continue                       # dense null-space basis only up to oss_max_n
            if spec.get("min_kkt_dim") and inst["kkt_dim"] < spec["min_kkt_dim"]: continue
            if spec["kind"] == "exact" and inst["kkt_dim"] > exact_max:
                skipped.append((inst["id"], name, f"exact LU skipped above exact_max_dim={exact_max}")); continue
            if (inst["family"], name) in slow and inst["kkt_dim"] > slow[(inst["family"], name)]:
                skipped.append((inst["id"], name, f"method exceeded {budget}s at kkt_dim {slow[(inst['family'], name)]}")); continue
            hk = hist.get((inst["family"], name), [])
            if (CONFIG["limits"].get("predictive_budget", True) if predictive is None else predictive) and hk and inst["kkt_dim"] > hk[-1][0] and hk[-1][1] > 1.0:
                # power law through the last two sizes (exponent clipped to [1, 3.5]; 2.0 with one point): skip before spending the time
                ex = 2.0
                if len(hk) >= 2 and hk[-2][1] > 0 and hk[-1][0] > hk[-2][0]:
                    ex = float(np.clip(np.log(hk[-1][1] / hk[-2][1]) / np.log(hk[-1][0] / hk[-2][0]), 1.0, 3.5))
                pred = hk[-1][1] * (inst["kkt_dim"] / hk[-1][0]) ** ex
                if pred > CONFIG["limits"].get("budget_margin", 1.5) * budget:
                    skipped.append((inst["id"], name, f"predicted {pred:.0f}s (= {hk[-1][1]:.0f}s at kkt_dim {hk[-1][0]} x size^{ex:.2f}) > {CONFIG['limits'].get('budget_margin', 1.5)} x {budget}s budget")); continue
            for rep in range(repeats):
                rec, _, _ = run_instance(inst, spec, repeat=rep, compute_cond=(name in compute_cond_for and rep == 0),
                                         capture=(inst["id"] in capture_ids and name == "custom_exact" and rep == 0), cond_scaled=(name in compute_cond_for and rep == 0), experiment=experiment)
                n_runs += 1
                if rep == 0:
                    hl = hist.setdefault((inst["family"], name), [])
                    if hl and hl[-1][0] == inst["kkt_dim"]: hl[-1] = (inst["kkt_dim"], max(hl[-1][1], rec["wall_total"]))     # several seeds per size: keep the slowest
                    else: hl.append((inst["kkt_dim"], rec["wall_total"]))
                if rec["wall_total"] > budget: slow[(inst["family"], name)] = min(slow.get((inst["family"], name), 10 ** 9), inst["kkt_dim"])
    dt = time.perf_counter() - t_exp
    for inst_id, name, why in skipped:
        REC.status(f"{experiment}_{inst_id}_{name}", "skipped", why)
    print(f"[{experiment}] {n_runs} runs in {dt:.1f}s" + (f"; {len(skipped)} (instance, method) pairs skipped (per-method budget / size guard)" if skipped else ""))
    return dt

tiny_ids = CONFIG["capture"]["instances"]; tiny = [INSTANCES[i] for i in tiny_ids]
run_methods(tiny, ["cvxopt_default", "custom_exact"], "E1_contract_tiny", repeats=1, capture_ids=tiny_ids)
df = REC.df("benchmark_results")
for inst in tiny:
    sub = df[(df.instance_id == inst["id"]) & (df.experiment == "E1_contract_tiny")]
    base, cust = sub[sub.method == "cvxopt_default"].iloc[0], sub[sub.method == "custom_exact"].iloc[0]
    ok = (base["iterations"] == cust["iterations"]) and abs(base["pobj"] - cust["pobj"]) <= 1e-8 * max(1, abs(base["pobj"]))
    print(f"  {inst['id']}: baseline iters={base['iterations']} custom iters={cust['iterations']} |pobj diff|={abs(base['pobj']-cust['pobj']):.2e} obj_rel_err={base['obj_rel_err']:.2e} -> {'OK' if ok else 'MISMATCH'}")
    assert ok, "custom exact solver does not reproduce the baseline"
REC.status("E1_contract_tiny", "done"); print("captured KKT systems:", {k: len(v) for k, v in CAPTURED.items()})

main_insts = [i for i in INSTANCES.values() if i["family"] in ("dense_kappa", "dense_noeq")]
ESSENTIAL = ["cvxopt_default", "custom_exact", "custom_exact_reduced", "gmres_stalelu_comb", "minres_relres_1e-06", "minres_be_1e-12", "minres_comb_0.1_1e-10",
             "minres_base_comb_cpu", "minres_base_comb_ws_cpu", "minres_base_comb_ws_switch_cpu", "minres_base_comb_gpu", "minres_base_comb_ws_gpu",
             "minres_fused_comb_cpu", "minres_fused_comb_floor8_cpu", "minres_fused_comb_gpu", "minres_fused_comb_auto", "lsqr_sqrt_comb", "cg_reduced_be_1e-10", "oss_cg_comb"]
main_methods = [n for n in SPECS if not n.startswith("exact_noise")] if CONFIG["inexact"]["full_ablation"] else [n for n in ESSENTIAL if n in SPECS]
t_main = run_methods(main_insts, main_methods, "E2_main", compute_cond_for=("custom_exact", "oss_cg_comb"))
REC.status("E2_main", "done", f"{len(main_insts)} instances x {len(main_methods)} methods x {CONFIG['repeats']} repeats in {t_main:.0f}s")

scale_insts = [i for i in INSTANCES.values() if i["family"] in ("sparse_banded", "sparse_scale", "sparse_lowfill")]
# minres_base_* is the baseline kernel for the comparison with minres_fused_*. SciPy MINRES and the warm-start variants are not
# run at scale: warm starts save at most 4 % of the iterations on the sparse families and cost iterations on the dense ones.
scale_methods = ["cvxopt_default", "custom_exact", "custom_exact_reduced", "gmres_stalelu_comb", "minres_base_comb_cpu", "minres_fused_comb_cpu", "minres_fused_comb_floor8_cpu",
                 "minres_base_comb_gpu", "minres_fused_comb_gpu", "minres_fused_comb_auto", "oss_cg_comb"]       # oss_cg_comb: n <= oss_max_n
t_scale = run_methods(scale_insts, scale_methods, "E3_scaling", compute_cond_for=("custom_exact", "oss_cg_comb"))
REC.status("E3_scaling", "done", f"{len(scale_insts)} instances in {t_scale:.0f}s")

ns = sorted({j["n"] for j in main_insts if j["family"] == "dense_kappa"})
noise_insts = [i for i in main_insts if i["family"] == "dense_kappa" and i["n"] == ns[min(1, len(ns) - 1)]]
noise_methods = [n for n in SPECS if n.startswith("exact_noise")]
t_noise = run_methods(noise_insts, noise_methods, "E4_noise", repeats=1)
REC.status("E4_noise", "done", f"{len(noise_insts)} instances x {len(noise_methods)} noise settings")

REC.flush(final=False, min_interval=0)
df = REC.df("benchmark_results")
print("\nAcceptance by method (classical experiments):")
print(df[df.experiment.isin(["E1_contract_tiny", "E2_main", "E3_scaling"])].groupby("method").agg(runs=("run_id", "size"), accepted=("accepted", "mean"), pure=("pure_method_success", "mean"),
      iters=("iterations", "median"), wall_med=("wall_method", "median"), fallback=("n_fallback_total", "sum"), refine=("n_refine_total", "sum"), krylov=("krylov_iters_total", "median")).round(4).to_string())

### E4b, IPM tolerance and direction error

E4 uses CVXOPT's default tolerances. E4b repeats it at $\tau \in \{10^{-4}, 10^{-6}, 10^{-8}\}$ with `abstol` = `feastol` = $\tau$ and `reltol` = $10\tau$. The acceptance thresholds of the final iterate scale with $\tau$ (pres, dres $\le 10\tau$; relative gap and objective error $\le 100\tau$; $\tau = 10^{-7}$ gives the defaults). The inner solvers are exact solves with isotropic ($\delta = 10^{-8} \dots 10^{-5}$) or block-relative error, the combined-test MINRES and the unmodified solver. The result is the largest tolerated $\delta$ as a function of $\tau$.

In [ ]:
# E4b: tolerated direction error as a function of the IPM tolerance level.
TS = CONFIG["tolerance_sweep"]
ts_insts = sorted([i for i in INSTANCES.values() if i["family"] in TS["families"] and i["n"] <= TS["n_max"]], key=lambda i: (i["n"], i["id"]))[:TS["max_instances"]]
def sweep_specs(tau):
    cvx_o = {"abstol": tau, "reltol": 10 * tau, "feastol": tau}; acc_o = {"pres": 10 * tau, "dres": 10 * tau, "relgap": 100 * tau, "obj_rel_err": 100 * tau, "cone_viol": 1e-8}
    out = []
    for d in TS["noise_iso"]:
        out.append({"name": f"exact_noise_iso_{d:.0e}@tau{tau:.0e}", "kind": "noise", "noise_model": "iso", "delta": d, "fallback": False, "max_refine": 0, "criterion": "relres"})
    for d in TS["noise_block"]:
        out.append({"name": f"exact_noise_block_{d:.0e}@tau{tau:.0e}", "kind": "noise", "noise_model": "block", "delta": d, "fallback": False, "max_refine": 0, "criterion": "block"})
    for m in TS["extra_methods"]:
        if m in SPECS: out.append(dict(SPECS[m], name=f"{m}@tau{tau:.0e}"))
    for s in out: s.update({"cvxopt_overrides": cvx_o, "accept_overrides": acc_o, "tol_level": tau, "base_method": s["name"].split("@")[0]})
    return out
t_ts = time.perf_counter(); n_ts = 0
for tau in TS["levels"]:
    for spec in sweep_specs(tau):
        for inst in ts_insts:
            if spec.get("requires_gpu") and not GPU_OK: continue
            run_instance(inst, spec, repeat=0, experiment="E4b_tolerance_sweep"); n_ts += 1
REC.status("E4b_tolerance_sweep", "done" if n_ts else "skipped", f"{n_ts} runs ({len(ts_insts)} instances x {len(TS['levels'])} tolerance levels) in {time.perf_counter() - t_ts:.0f}s")
REC.flush(final=False, min_interval=0)
df = REC.df("benchmark_results"); tsd = df[df.experiment == "E4b_tolerance_sweep"].copy()
if len(tsd):
    tsd["base"] = tsd.method.str.split("@").str[0]
    print("E4b - fraction accepted (thresholds scaled with tau) by inner solver x tolerance level tau:")
    print(tsd.pivot_table(index="base", columns="ipm_tol_level", values="accepted", aggfunc="mean").round(3).to_string())
    print("\nmedian IPM iterations:"); print(tsd.pivot_table(index="base", columns="ipm_tol_level", values="iterations", aggfunc="median").to_string())

### E8, Maros-Mészáros QPs

The unmodified solver runs first; its objective is the reference for `obj_rel_err`. Then the reduced direct solver, lazy refactorisation, the combined-test MINRES (CPU, and GPU from KKT dimension 2 000) and the switching variant `_switch` (exact solver after the first rejected direction; the share of solves delivered by MINRES is reported as `utilisation`). The power-law skip rule is off because the instances are heterogeneous.

In [ ]:
# E8: Maros-Meszaros QPs.
mm_insts = sorted(MM_INSTANCES.values(), key=lambda i: i["kkt_dim"]) if MM_CFG.get("enabled", True) else []
if not mm_insts:
    REC.status("E8_maros_meszaros", "skipped", "no Maros-Meszaros instances loaded")
else:
    t_mm = time.perf_counter(); n_base = 0
    for inst in mm_insts:                                   # 1) reference run: unmodified coneqp; its solution becomes the reference for obj_rel_err / x_rel_err
        rec, sol, _ = run_instance(inst, SPECS["cvxopt_default"], repeat=0, experiment="E8_maros_meszaros"); n_base += 1
        if sol is not None and rec["status"] == "optimal" and rec["accepted"]:
            inst.update({"f_star": rec["pobj"], "x_star": to_np(sol["x"]), "known_solution": True, "solution_source": "cvxopt_default"}); rec["obj_rel_err"] = 0.0; rec["x_rel_err"] = 0.0; rec["solution_source"] = "cvxopt_default"
        else:
            inst.update({"known_solution": False, "solution_source": f"none (baseline {rec['status']}, accepted={rec['accepted']})"})
        print(f"  {inst['id']}: baseline status={rec['status']} iters={rec['iterations']} accepted={rec['accepted']} ({rec['accept_fail_reasons'] or 'ok'}) wall={rec['wall_method']:.2f}s pobj={rec.get('pobj', float('nan')):.6g}")
    others = [m for m in MM_CFG["methods"] if m != "cvxopt_default"]
    t_o = run_methods(mm_insts, others, "E8_maros_meszaros", repeats=1, budget=MM_CFG["budget_s"], predictive=False)
    REC.status("E8_maros_meszaros", "done", f"{len(mm_insts)} instances; baseline + {len(others)} methods in {time.perf_counter() - t_mm:.0f}s")
    REC.flush(final=False, min_interval=0)
    d8 = REC.df("benchmark_results"); d8 = d8[d8.experiment == "E8_maros_meszaros"]
    print("\nE8 by method:"); print(d8.groupby("method").agg(runs=("run_id", "size"), optimal=("status", lambda s_: int((s_ == "optimal").sum())), accepted=("accepted", "mean"), strict=("strict_method_success", "mean"),
                                                            iters=("iterations", "median"), wall=("wall_method", "median"), fallbacks=("n_fallback_total", "sum"), krylov=("krylov_iters_total", "median"), peak_mem_mb=("mem_rss_increase_mb", "median")).round(3).to_string())
    base_ok = set(d8[(d8.method == "cvxopt_default") & d8.accepted].instance_id)
    print(f"\ninstances the unmodified solver accepts: {len(base_ok)} of {len(mm_insts)}")
    piv = d8[d8.instance_id.isin(base_ok)].pivot_table(index="instance_id", columns="method", values="wall_method", aggfunc="first")
    if len(piv): print("wall time [s] on those instances:"); print(piv.round(2).to_string())

## 13. Statevector simulator

Two quantum linear solvers are simulated: phase-estimation HHL (Harrow, Hassidim and Lloyd 2009; Sections 14 and 16) as the reference algorithm, and a polynomial solver over a qubitised walk (Section 15), whose cost grows with $\log(1/\epsilon)$ instead of $1/\epsilon$. Both use the same oracle model: the matrix and the right-hand side are loaded by ideal block encoding and state preparation, $\|A\|$ and $\kappa$ are known, and the classical cost of building these oracles in the simulation is timed separately as `t_prep`. Every number reported for them is a classical simulation of an ideal quantum computer.

`MiniSim` applies H, controlled-$U$, controlled-phase, swap and multi-controlled $R_y$ gates to a $2^n$ statevector, logs every gate for counts and depth, and samples measurements from the Born probabilities. Qubit $q$ is bit $q$ of the basis index (little-endian, as in Qiskit). The same code runs on NumPy or CuPy. The cell checks the QFT against the DFT matrix, the ordering of controlled gates, measurement marginals, state preparation, and the vectorised reciprocal rotation against a loop of multi-controlled gates. The quantum path uses no linear solve, inverse or eigendecomposition; exact eigenvalues are computed only for diagnostics ($\kappa$, eigenvalues below the clock resolution) and timed separately.

`max_qubits` and `max_state_bytes` in the configuration bound the simulated register; a larger circuit is reported as `skipped`.

In [ ]:
# MiniSim: gate-level statevector simulator, little-endian qubit order as in Qiskit.
class ResourceLimit(Exception):
    pass

H_GATE = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
X_GATE = np.array([[0, 1], [1, 0]], dtype=complex)
SWAP_GATE = np.array([[1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]], dtype=complex)
def RY(theta): c, s = np.cos(theta / 2), np.sin(theta / 2); return np.array([[c, -s], [s, c]], dtype=complex)
def PHASE(phi): return np.array([[1, 0], [0, np.exp(1j * phi)]], dtype=complex)

def check_qubit_budget(n_qubits):
    q = CONFIG["quantum"]
    if n_qubits > q["max_qubits"] or 16 * 2 ** n_qubits > q["max_state_bytes"]:
        raise ResourceLimit(f"{n_qubits} qubits exceeds the configured limit (max_qubits={q['max_qubits']}, "
                            f"max_state_bytes={q['max_state_bytes']:.0f})")

class MiniSim:
    """Statevector simulator. Basis index i = sum_q bit_q(i) 2^q  (qubit q <-> bit q). Gates are applied one by one and logged."""
    def __init__(self, n_qubits, xp=None):
        check_qubit_budget(n_qubits)
        self.n = n_qubits; self.xp = np if xp is None else xp          # numpy or cupy
        self.psi = self.xp.zeros(2 ** n_qubits, dtype=complex); self.psi[0] = 1.0
        self.gates = []; self.counts = {}; self._last = np.zeros(n_qubits, dtype=int); self.depth = 0
    def _log(self, name, qubits):
        self.gates.append((name, tuple(qubits))); self.counts[name] = self.counts.get(name, 0) + 1
        d = 1 + max(self._last[q] for q in qubits)
        for q in qubits: self._last[q] = d
        self.depth = max(self.depth, d)
    def apply(self, U, targets, controls=(), ctrl_values=None, name="U"):
        n = self.n; controls = list(controls); targets = list(targets)
        psi = self.psi.reshape([2] * n)
        idx = [slice(None)] * n
        if controls:
            ctrl_values = [1] * len(controls) if ctrl_values is None else list(ctrl_values)
            for c, v in zip(controls, ctrl_values): idx[n - 1 - c] = int(v)
        sub = psi[tuple(idx)]
        cset = set(controls)
        remaining = [q for q in range(n - 1, -1, -1) if q not in cset]
        pos = {q: i for i, q in enumerate(remaining)}
        k = len(targets)
        src = [pos[t] for t in reversed(targets)]
        dst = list(range(sub.ndim - k, sub.ndim))
        xp = self.xp; U = xp.asarray(U, dtype=complex)
        moved = xp.moveaxis(sub, src, dst)
        shp = moved.shape
        out = moved.reshape(-1, 2 ** k) @ U.T
        moved[...] = out.reshape(shp)
        self._log(name, controls + targets)
    # gates
    def h(self, q): self.apply(H_GATE, [q], name="h")
    def x(self, q): self.apply(X_GATE, [q], name="x")
    def ry(self, theta, q): self.apply(RY(theta), [q], name="ry")
    def cp(self, phi, c, t): self.apply(PHASE(phi), [t], controls=[c], name="cp")
    def swap(self, a, b): self.apply(SWAP_GATE, [a, b], name="swap")
    def cu(self, U, c, targets, name="c-U"): self.apply(U, targets, controls=[c], name=name)
    def mcry(self, theta, controls, ctrl_values, target): self.apply(RY(theta), [target], controls=controls, ctrl_values=ctrl_values, name="mcry")
    def uniform_ry(self, thetas_by_value, controls, target):
        """All 2^c multi-controlled RY rotations of the exact reciprocal as one vectorised operation.
        thetas_by_value[k] is the angle applied when the control register holds the little-endian value k (0 = identity).
        Mathematically identical to the loop of mcry gates; logged as one mcry per non-zero angle for gate counting."""
        xp = self.xp; n = self.n; c = len(controls); th = np.asarray(thetas_by_value, dtype=float)
        R = np.zeros((2 ** c, 2, 2), dtype=complex); cs, sn = np.cos(th / 2), np.sin(th / 2)
        R[:, 0, 0] = cs; R[:, 0, 1] = -sn; R[:, 1, 0] = sn; R[:, 1, 1] = cs
        psi = self.psi.reshape([2] * n)
        src = [n - 1 - q for q in reversed(controls)] + [n - 1 - target]          # control MSB first ... control LSB, then target
        dst = list(range(n - c - 1, n))
        moved = xp.moveaxis(psi, src, dst); shp = moved.shape
        flat = moved.reshape(-1, 2 ** c, 2)
        out = xp.einsum("kji,bki->bkj", xp.asarray(R), flat)
        moved[...] = out.reshape(shp)
        for k in np.nonzero(th)[0]: self._log("mcry", list(controls) + [target])
    def prepare(self, vec, targets):
        """Amplitude encoding of a normalised vector on a register that is in |0...0>: one unitary whose first column is vec."""
        N = 2 ** len(targets); v = np.asarray(vec, dtype=complex)
        assert v.shape == (N,) and abs(np.linalg.norm(v) - 1) < 1e-9
        M = np.eye(N, dtype=complex); M[:, 0] = v
        Q, R = np.linalg.qr(M)
        Q[:, 0] *= (R[0, 0] / abs(R[0, 0])) if abs(R[0, 0]) > 0 else 1.0      # make the first column exactly v (up to sign fix)
        if np.linalg.norm(Q[:, 0] - v) > 1e-8: Q[:, 0] = v                   # (numerically safe: v is unit norm and orthogonal to the rest)
        self.apply(Q, targets, name="state_prep")
    def qft(self, qs):
        """Standard QFT: |x> -> 1/sqrt(N) sum_k exp(2 pi i x k / N)|k> with little-endian integers on qs."""
        n = len(qs)
        for i in range(n - 1, -1, -1):
            self.h(qs[i])
            for j in range(i - 1, -1, -1):
                self.cp(np.pi / 2 ** (i - j), qs[j], qs[i])
        for i in range(n // 2): self.swap(qs[i], qs[n - 1 - i])
    def iqft(self, qs):
        n = len(qs)
        for i in range(n // 2): self.swap(qs[i], qs[n - 1 - i])
        for i in range(n):
            for j in range(i):
                self.cp(-np.pi / 2 ** (i - j), qs[j], qs[i])
            self.h(qs[i])
    # measurement
    def probs(self, qubits):
        """Marginal Born probabilities of the listed qubits, indexed little-endian in the given order (qubits[0] = LSB)."""
        p = (self.xp.abs(self.psi) ** 2).reshape([2] * self.n)
        others = tuple(self.n - 1 - q for q in range(self.n) if q not in qubits)
        if others: p = p.sum(axis=others)
        p = to_host(p)
        kept_desc = [q for q in range(self.n - 1, -1, -1) if q in qubits]
        perm = [kept_desc.index(q) for q in reversed(qubits)]
        return np.transpose(p, perm).reshape(-1)
    def amplitudes(self, fixed):
        """Sub-vector of amplitudes with some qubits fixed: fixed = {qubit: value}; remaining qubits little-endian."""
        psi = self.psi.reshape([2] * self.n)
        idx = [slice(None)] * self.n
        for q, v in fixed.items(): idx[self.n - 1 - q] = int(v)
        return to_host(psi[tuple(idx)].reshape(-1))
    def gate_summary(self):
        return dict(self.counts)

def sample_counts(p, shots, rng):
    p = np.clip(p, 0, None); p = p / p.sum()
    return rng.multinomial(int(shots), p)

# self-tests: QFT against the DFT matrix, controlled gates, marginals, state preparation
def _dft_matrix(n):
    N = 2 ** n; w = np.exp(2j * np.pi / N)
    return np.array([[w ** (x * k) for k in range(N)] for x in range(N)]) / np.sqrt(N)
_rng = rng_for("minisim_selftest")
for _n in (2, 3, 4):
    _v = _rng.standard_normal(2 ** _n) + 1j * _rng.standard_normal(2 ** _n); _v /= np.linalg.norm(_v)
    _s = MiniSim(_n); _s.psi[:] = _v; _s.qft(list(range(_n)))
    assert np.allclose(_s.psi, _dft_matrix(_n).T @ _v) or np.allclose(_s.psi, _dft_matrix(_n) @ _v), "QFT convention mismatch"
    _s.iqft(list(range(_n)))
    assert np.allclose(_s.psi, _v), "iQFT is not the inverse of QFT"
_s = MiniSim(3); _s.x(2); _s.cu(X_GATE, 2, [0]); assert abs(_s.psi[0b101]) == 1.0, "controlled gate ordering"
_s = MiniSim(3); _s.h(0); _s.h(2); _p = _s.probs([2, 0]); assert np.allclose(_p, 0.25), "marginals"
_s = MiniSim(2); _s.prepare(np.array([0.6, 0.0, 0.8, 0.0]), [0, 1]); assert np.allclose(_s.psi, [0.6, 0, 0.8, 0]), "state prep"
# vectorised exact-reciprocal rotation == loop of multi-controlled RY gates
_th = _rng.uniform(-1, 1, 8); _th[0] = 0.0
_v = _rng.standard_normal(32) + 1j * _rng.standard_normal(32); _v /= np.linalg.norm(_v)
_a = MiniSim(5); _a.psi[:] = _v; _b = MiniSim(5); _b.psi[:] = _v
for _k in range(1, 8): _a.mcry(_th[_k], [1, 2, 3], [(_k >> j) & 1 for j in range(3)], 0)
_b.uniform_ry(_th, [1, 2, 3], 0)
assert np.allclose(_a.psi, _b.psi), "uniform_ry differs from the mcry loop"; assert _a.counts["mcry"] == _b.counts["mcry"]
if DEVICE_XP is not np:
    _g = MiniSim(5, xp=DEVICE_XP); _g.psi[:] = DEVICE_XP.asarray(_v); _g.h(2); _g.cu(X_GATE, 1, [4]); _g.uniform_ry(_th, [1, 2, 3], 0)
    _c = MiniSim(5); _c.psi[:] = _v; _c.h(2); _c.cu(X_GATE, 1, [4]); _c.uniform_ry(_th, [1, 2, 3], 0)
    assert np.allclose(to_host(_g.psi), _c.psi), "device simulator differs from CPU simulator"
    print("device (GPU) simulator matches the CPU simulator")
print("MiniSim self-tests passed (QFT = DFT matrix, controlled-U ordering, marginals, state preparation)")

## 14. HHL

### Circuit

Registers, little-endian: ancilla `anc` (1 qubit), clock `clk` ($n_c$), data `dat` ($n_d = \log_2 N$), reference flag `flg` (1).

1. **Input.** $\tilde K$ (real symmetric, indefinite), optionally Ruiz-scaled, is padded to $N = 2^{n_d}$ with $\hat\lambda_{\max} I$. The padding lies inside the spectral range, so $\kappa$ is unchanged and the padded components of the solution are zero.
2. **Spectral bound.** $\hat\lambda_{\max} \ge \|\tilde K\|_2$ comes from 40 power iterations with a safety factor 1.15. The evolution time $t = 2\pi(2^{n_c-1}-1)/(2^{n_c}\hat\lambda_{\max})$ maps every eigenvalue in $[-\hat\lambda_{\max}, \hat\lambda_{\max}]$ to a signed clock value $k_s \in [-(2^{n_c-1}-1),\, 2^{n_c-1}-1]$ (two's complement), which handles the negative eigenvalues of the saddle-point matrix. With clock resolution $\Delta = 2\pi/(2^{n_c} t)$ the rotation for $k_s$ is $R_y(2\arcsin(C/(k_s\Delta)))$, the exact reciprocal (Qiskit's `ExactReciprocal` with `neg_vals=True`). `reciprocal = "resolution"` uses $C = \Delta$; `"kappa_known"` uses $C = k_{\min}\Delta$ with $k_{\min} = \lfloor |\lambda|_{\min}/\Delta \rfloor$, the standard HHL assumption that $\kappa$ is known.
3. **State preparation.** The data and flag registers are prepared in $|v\rangle \propto |0\rangle_{\text{flg}}|\tilde r\rangle + |1\rangle_{\text{flg}}|\tilde K r_{\text{ref}}\rangle$ with a known positive reference vector $r_{\text{ref}}$ scaled so that $\|\tilde K r_{\text{ref}}\| = \|\tilde r\|$. HHL is linear, so the post-selected output is $\propto |0\rangle|x\rangle + |1\rangle|r_{\text{ref}}\rangle$ with a common scale.
4. **Phase estimation** with $U = e^{i\tilde K t}$. The powers $U^{2^j}$ are formed classically by `expm` and repeated squaring ($O(N^3)$, part of `t_prep`); on hardware they would come from Hamiltonian simulation with total evolution time $2(2^{n_c}-1)t\|\tilde K\|$.
5. **Eigenvalue inversion** with $2^{n_c}-1$ multi-controlled $R_y$ rotations on `anc`.
6. **Uncomputation** of phase estimation. Measuring `anc = 1` selects $\propto \sum_j \beta_j (C/\tilde\lambda_j)|u_j\rangle$.

### Readout of a signed, scaled vector

Basis-state probabilities give $|x_i|^2$ up to a global scale. With the reference branch, two measurement settings recover signs and scale, each post-selected on `anc = 1`:

* **Z setting:** $P(\text{flg}=0, i) \propto x_i^2$ and $P(\text{flg}=1, i) \propto r_{\text{ref},i}^2$, so $\|x\| = \|r_{\text{ref}}\|\sqrt{P(\text{flg}=0)/P(\text{flg}=1)}$ and $|x_i| = \|x\|\sqrt{P(\text{flg}=0,i)/P(\text{flg}=0)}$.
* **X setting** (H on `flg` first): $P(\text{flg}=0,i) - P(\text{flg}=1,i) \propto x_i r_{\text{ref},i}$, whose sign is that of $x_i$.

Three readouts are recorded:

* `statevector`: the amplitudes of the `anc = 1, clk = 0` branch, a diagnostic that separates the error of the algorithm from the sampling error;
* `shots`: raw finite shots including post-selection;
* `aa_shots`: a fixed number of accepted samples drawn from the distribution conditioned on `anc = 1`. This is what ideal amplitude amplification would deliver at about $1/\sqrt{P}$ circuit repetitions per sample (Brassard, Høyer, Mosca and Tapp 2002). The amplification circuit is not simulated; its overhead is reported next to the raw-shot equivalent $1/P$.

The leakage $P(\text{anc}=1, \text{clk}\neq 0)$ is measured, and the finite-shot readout marginalises over the clock register as hardware would.

### Inside `coneqp`

The `hhl` solver kind runs inside the protocol of Section 8: each KKT solve is one HHL run, followed by the acceptance test on the full system, up to `max_refine` HHL runs on the residual, and a classical fallback if allowed. Runs with `fallback = False` show what the interior-point method does with quantum directions alone. The cell ends with a functional test on a $4\times4$ indefinite system.

In [ ]:
# HHL: circuit, readout of a signed and scaled vector, and the 'hhl' KKT-solver kind.
def pad_to_pow2(A, b, lam_pad):
    N0 = A.shape[0]; n_d = max(1, int(math.ceil(math.log2(N0)))); N = 2 ** n_d
    Ap = np.zeros((N, N)); Ap[:N0, :N0] = A
    if N > N0: Ap[N0:, N0:] = lam_pad * np.eye(N - N0)
    bp = np.zeros(N); bp[:N0] = b
    return Ap, bp, N0

def spectral_bound(A, method, rng):
    """Upper estimate of ||A||_2 by power iteration (oracle-compatible) or exact eigvalsh (diagnostic)."""
    if method == "exact":
        return float(np.abs(np.linalg.eigvalsh(A)).max()), "exact"
    q = CONFIG["quantum"]; v = rng.standard_normal(A.shape[0]); v /= np.linalg.norm(v); lam = 0.0
    for _ in range(q["power_iters"]):
        w = A @ v; lam = float(np.linalg.norm(w))
        if lam == 0: break
        v = w / lam
    return lam * q["power_safety"], "power_iteration"

class HHLSystem:
    """HHL circuit for a fixed real symmetric matrix A (N = 2^n_d). Built once per matrix; .run() solves for a given rhs."""
    def __init__(self, A, n_clock, rng, spectral="power", c_mode="resolution"):
        t0 = time.perf_counter()
        self.c_mode = c_mode
        A = np.asarray(A, dtype=float)
        assert A.shape[0] == A.shape[1] and np.allclose(A, A.T, atol=1e-10 * max(1, np.abs(A).max())), "A must be symmetric"
        self.N = A.shape[0]; self.n_d = int(round(math.log2(self.N))); assert 2 ** self.n_d == self.N
        self.n_c = int(n_clock); self.A = A; self.rng = rng
        self.anc = 0; self.clk = list(range(1, 1 + self.n_c)); self.dat = list(range(1 + self.n_c, 1 + self.n_c + self.n_d))
        self.flg = 1 + self.n_c + self.n_d; self.n_qubits = self.flg + 1
        check_qubit_budget(self.n_qubits)
        self.lam_max_est, self.bound_method = spectral_bound(A, spectral, rng)
        Nc = 2 ** self.n_c
        self.t = 2 * np.pi * (Nc // 2 - 1) / (Nc * self.lam_max_est)
        self.Delta = 2 * np.pi / (Nc * self.t)                  # clock resolution
        # reciprocal constant C: "resolution" -> C = Delta (no knowledge of lambda_min assumed; worst postselection);
        # "kappa_known" -> C = k_min*Delta with k_min = floor(|lambda|_min/Delta) (HHL's standard assumption that kappa is known;
        # here |lambda|_min is taken from the exact spectrum; this is an assumption of the method and timed as t_spectral_assumption)
        t_sa = time.perf_counter()
        if c_mode == "kappa_known":
            lam_min_abs = float(np.abs(np.linalg.eigvalsh(A)).min())
            self.k_min = int(max(1, min(Nc // 2 - 1, math.floor(lam_min_abs / self.Delta))))
        else:
            self.k_min = 1
        self.t_spectral_assumption = time.perf_counter() - t_sa
        self.C = self.k_min * self.Delta
        U = sla.expm(1j * A * self.t)
        self.U_pow = [U]
        for j in range(1, self.n_c): self.U_pow.append(self.U_pow[-1] @ self.U_pow[-1])
        self.U_pow_dag = [u.conj().T for u in self.U_pow]
        self.xp = DEVICE_XP if CONFIG["quantum"].get("device", "cpu") == "gpu" else np      # simulator backend
        self.U_pow_x = [self.xp.asarray(u) for u in self.U_pow]; self.U_pow_dag_x = [self.xp.asarray(u) for u in self.U_pow_dag]
        # rotation table: clock value k -> signed integer k_s -> theta
        self.thetas = {}
        for k in range(1, Nc):
            ks = k if k < Nc // 2 else k - Nc
            if ks == 0: continue
            amp = np.sign(ks) * min(1.0, self.k_min / abs(ks))      # C / lambda_tilde, clipped to 1 for |lambda| below the assumed minimum
            self.thetas[k] = 2 * np.arcsin(amp)
        self.t_prep = time.perf_counter() - t0
        # diagnostics from the exact spectrum, not used by the circuit
        t1 = time.perf_counter()
        ev = np.linalg.eigvalsh(A); self.eig = ev
        self.kappa = float(np.abs(ev).max() / max(np.abs(ev).min(), 1e-300))
        self.lam_max_true = float(np.abs(ev).max()); self.lam_min_abs = float(np.abs(ev).min())
        self.n_eig_below_resolution = int(np.sum(np.abs(ev) < self.Delta / 2))
        self.n_eig_unresolved_10pct = int(np.sum(np.abs(ev) < 5 * self.Delta))   # eigenvalues with >10% relative quantisation error
        self.t_diag = time.perf_counter() - t1

    def reference(self, b):
        r = np.ones(self.N) / np.sqrt(self.N)
        Ar = self.A @ r
        s = np.linalg.norm(b) / max(np.linalg.norm(Ar), 1e-300)
        return r * s, Ar * s

    def run(self, b, shots=0, sampling_repeats=1, shots_list=None, x_exact=None, eff_shots=0):
        """Run the circuit for rhs b. Returns dict with ideal-statevector and finite-shot reconstructions and diagnostics."""
        b = np.asarray(b, dtype=float); nb = np.linalg.norm(b)
        out = {"n_qubits": self.n_qubits, "n_clock": self.n_c, "n_data": self.n_d, "N": self.N, "kappa": self.kappa,
               "lam_max_est": self.lam_max_est, "lam_max_true": self.lam_max_true, "lam_min_abs": self.lam_min_abs,
               "clock_resolution": self.Delta, "reciprocal_mode": self.c_mode, "k_min": self.k_min, "C_over_lam_max": self.C / self.lam_max_est,
               "n_eig_below_resolution": self.n_eig_below_resolution,
               "n_eig_unresolved_10pct": self.n_eig_unresolved_10pct, "t_prep": self.t_prep, "t_diag": self.t_diag,
               "spectral_bound_method": self.bound_method, "rhs_norm": nb}
        if nb == 0:
            out.update({"x_ideal": np.zeros(self.N), "x_shots": np.zeros(self.N), "status": "zero_rhs"}); return out
        t0 = time.perf_counter()
        r_ref, Ar = self.reference(b)
        v = np.concatenate([b, Ar]); vnorm = np.linalg.norm(v); v = v / vnorm
        sim = MiniSim(self.n_qubits, xp=self.xp)
        sim.prepare(v.astype(complex), self.dat + [self.flg])
        for q in self.clk: sim.h(q)
        for j, q in enumerate(self.clk): sim.cu(self.U_pow_x[j], q, self.dat, name="c-U^2^j")
        sim.iqft(self.clk)
        Nc = 2 ** self.n_c
        th_arr = np.zeros(Nc)
        for k, th in self.thetas.items(): th_arr[k] = th
        sim.uniform_ry(th_arr, self.clk, self.anc)               # all 2^n_c - 1 reciprocal rotations, vectorised
        sim.qft(self.clk)
        for j in reversed(range(self.n_c)): sim.cu(self.U_pow_dag_x[j], self.clk[j], self.dat, name="c-U^-2^j")
        for q in self.clk: sim.h(q)
        out["t_circuit_sim"] = time.perf_counter() - t0
        out["gate_counts"] = sim.gate_summary(); out["n_gates_highlevel"] = len(sim.gates); out["depth_highlevel"] = sim.depth
        # ideal statevector diagnostics (not available on hardware)
        t1 = time.perf_counter()
        p_anc = sim.probs([self.anc]); out["p_postselect_exact"] = float(p_anc[1])
        p_anc_clk = sim.probs([self.anc] + self.clk).reshape(-1)     # index = anc + 2*clockvalue
        out["p_leak_exact"] = float(p_anc_clk[1::2][1:].sum())      # anc=1 and clock != 0
        fixed = {self.anc: 1}; fixed.update({q: 0 for q in self.clk})
        amp = sim.amplitudes(fixed)                                  # remaining qubits: dat (low) then flg (high)
        x_id, r_id = amp[:self.N], amp[self.N:]
        c = np.vdot(r_ref, r_id) / np.vdot(r_ref, r_ref)
        x_ideal = (x_id / c) if abs(c) > 0 else np.full(self.N, np.nan)
        out["ideal_imag_fraction"] = float(np.linalg.norm(x_ideal.imag) / max(np.linalg.norm(x_ideal), 1e-300))
        out["x_ideal"] = x_ideal.real
        out["scale_from_postselection"] = float(np.sqrt(max(out["p_postselect_exact"] * vnorm ** 2 / self.C ** 2 - np.dot(r_ref, r_ref), 0.0)))
        # finite-shot readout: Z setting and X setting on the flag
        meas = [self.anc, self.flg] + self.dat                       # index = anc + 2*flg + 4*i
        pZ = sim.probs(meas)
        sim.h(self.flg); pX = sim.probs(meas)
        out["pZ"], out["pX"] = pZ, pX
        out["t_readout_probs"] = time.perf_counter() - t1
        out["r_ref"] = r_ref; out["v_norm"] = vnorm; out["b"] = b
        out["x_exact"] = np.linalg.solve(self.A, b) if x_exact is None else x_exact      # classical reference, diagnostic only
        if shots:
            recs = reconstruct_from_shots(self, out, shots, sampling_repeats, self.rng)
            out["x_shots"] = recs[0]["x_rec"]; out["shot_records"] = recs
        if eff_shots:
            recs = reconstruct_aa_emulated(self, out, eff_shots, sampling_repeats, self.rng)
            out["x_aa"] = recs[0]["x_rec"]; out["aa_records"] = recs
        return out

def reconstruct_from_counts(sys_, out, cZ, cX, denom, rep=0, mode="shots"):
    """Rebuild a signed, scaled vector from Z- and X-setting counts (shape [N, flg, anc]); denom = shots per setting."""
    N = sys_.N; r_ref = out["r_ref"]; x_exact = out["x_exact"]; nx = max(np.linalg.norm(x_exact), 1e-300); t0 = time.perf_counter()
    z0, z1 = cZ[:, 0, 1].astype(float), cZ[:, 1, 1].astype(float); n0, n1 = z0.sum(), z1.sum()
    post_rate = (n0 + n1) / denom
    if n0 == 0 or n1 == 0:
        return {"repeat": rep, "readout": mode, "shots_per_setting": denom, "postselect_rate": post_rate, "status": "postselection_failed", "x_rec": np.full(N, np.nan),
                "rel_err": np.nan, "rel_err_ls": np.nan, "rel_resid": np.nan, "t_reconstruct": time.perf_counter() - t0}
    norm_x = np.linalg.norm(r_ref) * np.sqrt(n0 / n1); mag = norm_x * np.sqrt(z0 / n0)
    sign = np.sign(cX[:, 0, 1].astype(float) - cX[:, 1, 1].astype(float)); sign[sign == 0] = 1.0
    x_rec = sign * mag; b = out["b"]; Ax = sys_.A @ x_rec
    alpha = float(Ax @ b) / max(float(Ax @ Ax), 1e-300); x_ls = alpha * x_rec
    n_sign_err = int(np.sum((np.sign(x_rec) != np.sign(x_exact)) & (np.abs(x_exact) > 1e-3 * nx)))
    return {"repeat": rep, "readout": mode, "shots_per_setting": denom, "postselect_rate": post_rate, "status": "ok", "postselected_shots": int(n0 + n1), "x_rec": x_rec, "x_rec_ls": x_ls,
            "rel_err": float(np.linalg.norm(x_rec - x_exact) / nx), "rel_err_ls": float(np.linalg.norm(x_ls - x_exact) / nx), "rel_resid": float(relnorm(Ax - b, b)),
            "scale_rel_err": float(abs(norm_x - nx) / nx), "n_sign_errors": n_sign_err, "t_reconstruct": time.perf_counter() - t0}

def reconstruct_from_shots(sys_, out, shots, repeats, rng):
    """Raw finite-shot readout (hardware-like): sample the full Born distribution, postselect anc = 1."""
    recs = []
    for rep in range(repeats):
        cZ = sample_counts(out["pZ"], shots, rng).reshape(sys_.N, 2, 2); cX = sample_counts(out["pX"], shots, rng).reshape(sys_.N, 2, 2)
        recs.append(reconstruct_from_counts(sys_, out, cZ, cX, shots, rep, "shots"))
    return recs

def reconstruct_aa_emulated(sys_, out, eff_shots, repeats, rng):
    """Readout with emulated amplitude amplification: samples are drawn from the distribution conditioned on anc = 1 (what ideal AA
    would deliver), so eff_shots is the number of accepted samples per setting. The AA circuit is not simulated; its cost is reported as
    aa_overhead = 1/sqrt(P(anc=1)) circuit repetitions per accepted sample (raw-shot equivalent without AA: eff_shots / P(anc=1))."""
    recs = []; p_post = out["p_postselect_exact"]
    for rep in range(repeats):
        pZ = out["pZ"].reshape(sys_.N, 2, 2).copy(); pZ[:, :, 0] = 0.0; pX = out["pX"].reshape(sys_.N, 2, 2).copy(); pX[:, :, 0] = 0.0
        if pZ.sum() <= 0 or pX.sum() <= 0:
            recs.append({"repeat": rep, "readout": "aa_shots", "shots_per_setting": eff_shots, "status": "postselection_failed", "x_rec": np.full(sys_.N, np.nan), "rel_err": np.nan, "rel_err_ls": np.nan, "rel_resid": np.nan, "t_reconstruct": 0.0}); continue
        cZ = sample_counts(pZ.reshape(-1), eff_shots, rng).reshape(sys_.N, 2, 2); cX = sample_counts(pX.reshape(-1), eff_shots, rng).reshape(sys_.N, 2, 2)
        rec = reconstruct_from_counts(sys_, out, cZ, cX, eff_shots, rep, "aa_shots")
        rec.update({"aa_overhead_per_sample": 1.0 / math.sqrt(max(p_post, 1e-300)), "raw_shots_equivalent_no_AA": eff_shots / max(p_post, 1e-300), "postselect_rate": p_post})
        recs.append(rec)
    return recs

def regularize_kkt(K, asm, delta):
    """Primal-dual regularisation K + delta*J, J = diag(+I_n, -I_{p+m}) (same sign convention as CVXOPT's kktreg)."""
    if delta == 0: return K
    N = K.shape[0]; J = np.ones(N); J[asm.n:] = -1.0
    return K + delta * np.diag(J)

class HHLKind:
    """KKT-solver kind 'hhl' inside the protocol of KKTSolveEngine.
    target = (optionally regularised) K -> (optionally Ruiz-scaled) -> padded to 2^n -> HHL circuit -> readout (ideal / raw shots / AA-emulated).
    The engine verifies the direction on the original system and, if the test fails, calls base_solve again on the true residual
    (iterative refinement with the quantum solver as preconditioner), then falls back classically if allowed."""
    def __init__(self, spec, K, asm, W, eng):
        Kd = K.toarray() if sp.issparse(K) else np.asarray(K, dtype=float)
        self.reg = float(spec.get("reg", 0.0))
        self.scaled = spec.get("scaling", "ruiz") == "ruiz"
        self.d = ruiz_equilibrate(Kd) if self.scaled else None
        Ks = apply_scaling(Kd, self.d) if self.scaled else Kd
        Ks = regularize_kkt(Ks, asm, self.reg)            # on the scaled system (unit row norms), so kappa <= ~(1+delta)/delta
        self.rng = rng_for("hhl", asm.inst["id"], spec["name"])
        lam, _ = spectral_bound(Ks, CONFIG["quantum"]["spectral_bound"], self.rng)
        self.Ap, _, self.N0 = pad_to_pow2(Ks, np.zeros(Ks.shape[0]), lam)
        self.sys = HHLSystem(self.Ap, spec["n_clock"], self.rng, spectral=CONFIG["quantum"]["spectral_bound"], c_mode=spec.get("reciprocal", "kappa_known"))
        self.recon = spec.get("recon", "statevector"); self.shots = int(spec.get("shots", 0)); self.eff_shots = int(spec.get("eff_shots", 0))
        self.qlog = []
    def base_solve(self, r, ctx):
        rs = self.d * r if self.scaled else r
        rp = np.zeros(self.Ap.shape[0]); rp[:self.N0] = rs
        out = self.sys.run(rp, shots=(self.shots if self.recon == "shots" else 0), eff_shots=(self.eff_shots if self.recon == "aa_shots" else 0), sampling_repeats=1)
        if self.recon == "shots": up, rec = out["x_shots"], out["shot_records"][0]
        elif self.recon == "aa_shots": up, rec = out["x_aa"], out["aa_records"][0]
        else: up, rec = out["x_ideal"], None
        u = up[:self.N0]
        if self.scaled: u = self.d * u
        xe = out["x_exact"]; nx = max(np.linalg.norm(xe), 1e-300)
        info = {"q_n_qubits": out["n_qubits"], "q_n_clock": out["n_clock"], "q_kappa_target": out["kappa"], "q_reg": self.reg, "q_scaling": "ruiz" if self.scaled else "none",
                "q_reciprocal_mode": out["reciprocal_mode"], "q_k_min": out["k_min"], "q_p_postselect": out.get("p_postselect_exact", np.nan), "q_p_leak": out.get("p_leak_exact", np.nan),
                "q_rel_err_ideal_target": float(np.linalg.norm(out["x_ideal"] - xe) / nx) if "x_ideal" in out else np.nan,
                "q_rel_err_used_target": float(np.linalg.norm(up - xe) / nx) if np.all(np.isfinite(up)) else np.inf, "q_recon": self.recon, "q_shots": self.shots, "q_eff_shots": self.eff_shots,
                "q_t_prep": out["t_prep"], "q_t_circuit_sim": out.get("t_circuit_sim", np.nan), "q_t_readout": out.get("t_readout_probs", np.nan) + (rec["t_reconstruct"] if rec else 0.0),
                "q_gates": out.get("n_gates_highlevel", np.nan), "q_depth": out.get("depth_highlevel", np.nan), "q_n_eig_unresolved_10pct": out["n_eig_unresolved_10pct"]}
        if rec is not None:
            info.update({"q_postselect_rate_empirical": rec.get("postselect_rate", np.nan), "q_shot_status": rec["status"], "q_sign_errors": rec.get("n_sign_errors", np.nan), "q_aa_overhead": rec.get("aa_overhead_per_sample", np.nan)})
        self.qlog.append(info)
        return u, info
SOLVER_KINDS["hhl"] = HHLKind

# quick functional test on a 4x4 symmetric indefinite system
_A = np.array([[3.0, 0.5, 0, 0.2], [0.5, -1.0, 0.3, 0], [0, 0.3, 2.0, 0.1], [0.2, 0, 0.1, -2.5]]); _b = np.array([1.0, -0.5, 0.3, 0.8])
_sys = HHLSystem(_A, 6, rng_for("hhl_selftest"))
_out = _sys.run(_b, shots=20000, sampling_repeats=2, eff_shots=2000)
_xe = np.linalg.solve(_A, _b)
print(f"HHL self-test (N=4, n_clock=6, kappa={_sys.kappa:.1f}): rel_err ideal={np.linalg.norm(_out['x_ideal']-_xe)/np.linalg.norm(_xe):.3e}, "
      f"shots(2x20000)={_out['shot_records'][0]['rel_err']:.3e}, AA-emulated(2x2000 accepted)={_out['aa_records'][0]['rel_err']:.3e}, P(anc=1)={_out['p_postselect_exact']:.3f}, leak={_out['p_leak_exact']:.2e}, "
      f"qubits={_out['n_qubits']}, gates={_out['n_gates_highlevel']}, depth={_out['depth_highlevel']}, sim {_out['t_circuit_sim']*1e3:.0f} ms")
assert np.linalg.norm(_out["x_ideal"] - _xe) / np.linalg.norm(_xe) < 0.2, "HHL ideal reconstruction is off; check the qubit-order conventions"

## 15. Polynomial linear solver

HHL's error is set by the clock register ($\propto 2^{-n_c}$) and is systematic, so refinement cannot remove it: with ten clock qubits and ideal readout the forward error stays at $10^{-3}$ to $10^{-2}$, against the roughly $10^{-7}$ that the interior-point method needs (Section 16).

Childs, Kothari and Somma (*SIAM J. Comput.* 46, 2017) approximate $A^{-1}$ by an odd Chebyshev polynomial of degree $O(\kappa\log(\kappa/\epsilon))$ and implement it as a linear combination of unitaries over a qubitised walk. The cost of precision is then $\log(1/\epsilon)$. QSVT (Gilyén et al. 2019) realises the same polynomial with fewer ancillas. Two polynomials are available:

* `poly = "optimal"` (default): the minimax-optimal odd polynomial of Sünderhauf et al. (arXiv:2507.15537). If its double-precision evaluation misses the requested accuracy or its degree exceeds 60 000, the CKS expansion is used and the run is labelled accordingly.
* `poly = "cks"`: the CKS expansion.

The remaining error of a finite-shot readout is zero-mean sampling noise, which iterative refinement on the true system contracts geometrically, as in the iterative-refinement QIPM literature (Mohammadisiahroudi, Wu, Augustino, Terlaky et al.). The default operating point is therefore a low-precision circuit ($\epsilon = 0.3$) with few samples per refinement round, in line with the fixed-precision analysis of Harkness et al. (arXiv:2609.10250).

**Simulated.** A dense Hermitian unitary block encoding of $A/\|A\|$, the walk $W$ with $\Pi W^k \Pi = T_k(A)$ (asserted), LCU post-selection with its exact success probability, and the readout of Section 14 (statevector, raw shots, emulated amplitude amplification). SELECT only controls on the index register, so the post-selected output is accumulated term by term; the self-test compares this with an explicit index-register statevector. **Not simulated:** gate synthesis of the block encoding and of PREP, amplitude amplification, hardware noise. The reported walk steps, qubit counts and success probabilities are those of the full circuit.

**Simulation shortcut.** For a fixed matrix the output is $p(A/\alpha)\,b$ for every right-hand side. The walk is simulated once per KKT matrix and compared with the spectral form $V p(\Lambda) V^\top$ of the same polynomial (difference logged as `q_walk_check_diff`, required below $10^{-7}$); the second right-hand side and all refinement rounds reuse $p(\lambda_i)$. This affects only the cost of the classical simulation.

**SPD target.** For the null-space system $Z^\top S Z$ the polynomial is the shifted Chebyshev expansion of $1/x$ on $[1/\kappa, 1]$, with $c_0 = \sqrt\kappa$, $c_k = 2\sqrt\kappa(-1)^k\rho^k$ and $\rho = (\sqrt\kappa-1)/(\sqrt\kappa+1)$. Its degree is $O(\sqrt\kappa\log(\kappa/\epsilon))$, the counterpart of the $\sqrt\kappa$ of CG. The walk acts on a block encoding of the shifted matrix $(2A/\alpha - (1+a)I)/(1-a)$, so that every $T_k$ is available; the LCU normalisation of about 3 for building that encoding from one of $A$ is reported, not simulated.

`block_encoding_costs` reports the normalisation of two standard encodings (sparse access, Pauli LCU) for comparison with the ideal $\alpha = \|A\|_2$ used in the simulation.

In [ ]:
# Polynomial linear solver: p(A) ~ A^-1 as a linear combination of unitaries over a qubitised walk (Childs, Kothari, Somma 2017).
# The error decreases exponentially in the degree d = O(kappa log(kappa/eps)), so precision costs log(1/eps) instead of the
# 1/eps of phase estimation.
#
# Simulated at the statevector level, with the same oracle model as the HHL cell:
#   * block encoding U_A = [[A, S], [S, -A]], S = sqrt(I - A^2) (one ancilla, built densely);
#   * walk W = (2 Pi - I) U_A with Pi W^k Pi = T_k(A) (checked below);
#   * LCU: PREP on an index register, SELECT = sum_j |j><j| (x) W^(2j+1), PREP^dagger, post-selection on index = 0 and
#     ancilla = 0. SELECT only controls on the index register, so the post-selected output is accumulated term by term;
#     an explicit index-register statevector is compared with this on a small case;
#   * readout as for HHL (reference branch on a flag qubit, Z and X settings).
# Assumed: ||A|| and kappa known, block encoding and state preparation free. Not simulated: gate synthesis of U_A and PREP,
# amplitude amplification, noise. Reported: walk steps (block-encoding queries), qubits, ||c||_1, post-selection probability.
from scipy.stats import binom as _binom

def cks_coefficients(kappa, eps):
    b = int(math.ceil(kappa ** 2 * math.log(kappa / eps))); j0 = int(math.ceil(math.sqrt(b * math.log(4 * b / eps))))
    j = np.arange(j0 + 1); return 4.0 * (-1.0) ** j * _binom.sf(b + j, 2 * b, 0.5), b

def optimal_inverse_coefficients(kappa, eps):
    """Minimax-optimal odd polynomial for 1/x on [-1,-a] U [a,1], a = 1/kappa (Sunderhauf, Nemeth, Walayat, Patterson, Berntson,
    arXiv:2507.15537, Theorem 1; stable recurrence Eqs. (9)-(12), Chebyshev coefficients by exact interpolation at degree d = 2n-1).
    Uniform error eps_d = (1-a)^n / (a (1+a)^(n-1)).  Returns (coefficients of T_1, T_3, ..., degree, measured max |p(x) - 1/x| on the target set)."""
    from scipy.fft import dct
    a = 1.0 / kappa; n = max(1, int(math.ceil((math.log(1 / eps) + math.log(1 / a) + math.log(1 + a)) / (math.log(1 + a) - math.log(1 - a)))))
    al = (1 + a) / (2 * (1 - a)); d = 2 * n - 1; xs = np.cos(np.pi * (np.arange(d + 1) + 0.5) / (d + 1)); y = (2 * xs ** 2 - (1 + a * a)) / (1 - a * a)
    L1 = (y + (1 - a) / (1 + a)) / al; L2 = (y * y + (1 - a) / (2 * (1 + a)) * y - 0.5) / al ** 2
    Lm, Ln = (L1, L1) if n == 1 else (L1, L2)
    for _ in range(max(n - 2, 0)): Lm, Ln = Ln, y * Ln / al - Lm / (4 * al * al)
    pv = (1 - (-1) ** n * (1 + a) ** 2 / (4 * a) * Ln) / xs                       # d + 1 is even, so x = 0 is never a node
    ck = dct(pv, type=2) / (d + 1); c = ck[1::2].copy()
    ng = 3000 if d <= 50000 else 600                                                # check grid inside [a, 1], denser near a where the error peaks
    xg = np.unique(np.concatenate([np.linspace(a, min(3 * a, 1.0), ng // 2), np.geomspace(a, 1.0, ng // 2)])); acc = c[0] * xg; tp, tc = np.ones_like(xg), xg.copy()
    for j in range(1, len(c)):                                                      # evaluate sum c_j T_{2j+1}(x) on a grid with the three-term recurrence
        tp = 2 * xg * tc - tp; tc = 2 * xg * tp - tc; acc += c[j] * tc
    return c, d, float(np.max(np.abs(acc - 1 / xg)))

def polynomial_degree(kappa, eps, poly):
    """Degree implied by (kappa, eps) without generating coefficients (used for the simulation budget)."""
    if poly == "optimal":
        a = 1.0 / kappa; return 2 * max(1, int(math.ceil((math.log(1 / eps) + math.log(1 / a) + math.log(1 + a)) / (math.log(1 + a) - math.log(1 - a))))) - 1
    b = int(math.ceil(kappa ** 2 * math.log(kappa / eps))); return 2 * int(math.ceil(math.sqrt(b * math.log(4 * b / eps)))) + 1

OPTIMAL_POLY_MAX_DEGREE = 60000      # the recurrence costs O(d^2) (~10 s at 60 000); above this the CKS expansion is used (labelled)
def inverse_polynomial(kappa, eps, poly="optimal"):
    """Coefficients of the inversion polynomial. 'optimal' falls back to the CKS expansion (labelled) if its double-precision
    evaluation misses the requested accuracy (seen only for kappa >= 1000 with eps <= 1e-6) or its degree exceeds OPTIMAL_POLY_MAX_DEGREE."""
    if poly == "optimal":
        if polynomial_degree(kappa, eps, "optimal") > OPTIMAL_POLY_MAX_DEGREE:
            c, _ = cks_coefficients(kappa, eps); return c, "cks (fallback: optimal-polynomial degree above the generation cap)", float("nan")
        c, d, err = optimal_inverse_coefficients(kappa, eps)
        if err <= 3.0 * eps: return c, "optimal", err
        c, _ = cks_coefficients(kappa, eps); return c, "cks (fallback: optimal polynomial lost accuracy in double precision)", float("nan")
    c, _ = cks_coefficients(kappa, eps); return c, "cks", float("nan")

def block_encode(A):
    """Hermitian unitary block encoding of a Hermitian A with ||A|| <= 1."""
    lam, V = np.linalg.eigh(A); lam = np.clip(lam, -1.0, 1.0); S = (V * np.sqrt(1.0 - lam ** 2)) @ V.T
    return np.block([[A, S], [S, -A]])

def block_encoding_costs(A):
    """Normalisation factors alpha of two standard block encodings of A (the walk degree scales with kappa_eff = kappa * alpha / ||A||_2; the simulation uses the ideal
    alpha = ||A||_2).  (1) sparse-access encoding: alpha = s * max|a_ij|, s = max nonzeros per row;  (2) LCU of Pauli strings: alpha = sum_P |c_P|, plus the number of
    Pauli terms (Ozbaygin, Kocuk and Moran report SDP matrices ~98 % dense in the Pauli basis)."""
    N = A.shape[0]; q = int(round(math.log2(N))); nz = np.abs(A) > 1e-14 * np.abs(A).max(); out = {"alpha_ideal": float(np.linalg.norm(A, 2)), "alpha_sparse": float(nz.sum(axis=1).max() * np.abs(A).max())}
    T = A.astype(complex).reshape([2] * (2 * q))                       # axes: row bits, column bits; one (row bit, column bit) pair -> one Pauli axis per step
    for k in range(q):
        T = np.moveaxis(T, [0, q - k], [-2, -1]); a, b_, c, d_ = T[..., 0, 0], T[..., 0, 1], T[..., 1, 0], T[..., 1, 1]
        T = np.stack([(a + d_) / 2, (b_ + c) / 2, 1j * (b_ - c) / 2, (a - d_) / 2], axis=-1)
    coef = np.abs(T).ravel(); out["alpha_pauli"] = float(coef.sum()); out["pauli_terms"] = int((coef > 1e-12 * coef.max()).sum()); out["pauli_terms_possible"] = int(N * (N + 1) // 2)       # a real symmetric matrix has at most N(N+1)/2 Pauli strings (those with an even number of Y)
    return out

class CKSSystem:
    def __init__(self, A, eps, rng, kappa_margin=1.05, max_degree=None, poly=None):
        t0 = time.perf_counter(); self.A = np.asarray(A, dtype=float); self.N = self.A.shape[0]; self.n_d = int(round(math.log2(self.N))); self.rng = rng
        ev = np.linalg.eigvalsh(self.A); self.t_spectral_assumption = time.perf_counter() - t0          # assumed known, as in HHL 'kappa_known'
        self.alpha = float(np.abs(ev).max()) * (1 + 1e-9); self.kappa = float(np.abs(ev).max() / np.abs(ev).min()); self.eps = float(eps)
        poly = poly or CONFIG["quantum"].get("cks", {}).get("poly", "optimal")
        # simulation budget: degree of the cheapest admissible polynomial, before any coefficients are generated
        d_est = polynomial_degree(self.kappa * kappa_margin, self.eps, "cks" if (poly == "cks" or polynomial_degree(self.kappa * kappa_margin, self.eps, "optimal") > OPTIMAL_POLY_MAX_DEGREE) else "optimal")
        if max_degree and d_est > max_degree:
            self.c = np.zeros(1); self.poly = poly + " (not generated: degree cap)"; self.poly_err = float("nan"); self.j0 = (d_est - 1) // 2; self.degree = d_est; self.l1 = float("nan"); self.over_budget = True
            self.n_index = max(1, int(math.ceil(math.log2(self.j0 + 1)))); self.n_qubits = self.n_d + 1 + 1 + self.n_index + 1; self.t_prep = time.perf_counter() - t0; self.W = None; self.walk_check_diff = None; return
        self.c, self.poly, self.poly_err = inverse_polynomial(self.kappa * kappa_margin, self.eps, poly); self.j0 = len(self.c) - 1; self.degree = 2 * self.j0 + 1; self.l1 = float(np.abs(self.c).sum())
        self.over_budget = bool(max_degree) and self.degree > max_degree          # the run is then reported as over budget, not replaced
        U = block_encode(self.A / self.alpha); N = self.N; Wm = U.copy(); Wm[N:, :] *= -1.0; self.W = Wm              # W = (2Pi - I) U
        self.n_index = max(1, int(math.ceil(math.log2(self.j0 + 1)))); self.n_qubits = self.n_d + 1 + 1 + self.n_index + 1       # data + flag + block-encoding ancilla + index + success flag
        # The circuit output for any right-hand side is p(A/alpha) b. The walk is simulated on the first solve and compared with
        # the spectral form V p(Lambda) V' of the same polynomial; later solves with this matrix reuse p(lambda_i). This only makes
        # the simulation cheaper; the reported resources are those of the circuit.
        lam_, self._V = np.linalg.eigh(self.A / self.alpha); tp, tc = np.ones_like(lam_), lam_.copy(); pl = self.c[0] * tc
        for j in range(1, self.j0 + 1):
            tp = 2 * lam_ * tc - tp; tc = 2 * lam_ * tp - tc; pl += self.c[j] * tc
        self._plam = pl; self.walk_check_diff = None
        self.t_prep = time.perf_counter() - t0
    def apply_poly_spectral(self, V): return self._V @ (self._plam[:, None] * (self._V.T @ V))
    def apply_poly(self, V):
        """Columns of V (N x k) -> sum_j c_j T_{2j+1}(A/alpha) V, accumulated from the ancilla-0 block of W^(2j+1) (walk-level statevector simulation)."""
        N = self.N; st = np.vstack([V, np.zeros_like(V)]); st = self.W @ st; acc = self.c[0] * st[:N]
        W2 = self.W @ self.W
        for j in range(1, self.j0 + 1):
            st = W2 @ st; acc += self.c[j] * st[:N]
        return acc
    def reference(self, b):
        r = np.ones(self.N) / np.sqrt(self.N); Ar = self.A @ r; s = np.linalg.norm(b) / max(np.linalg.norm(Ar), 1e-300); return r * s, Ar * s
    def run(self, b, shots=0, sampling_repeats=1, x_exact=None, eff_shots=0):
        b = np.asarray(b, dtype=float); nb = np.linalg.norm(b); N = self.N
        out = {"n_qubits": self.n_qubits, "n_index": self.n_index, "N": N, "kappa": self.kappa, "degree": self.degree, "walk_steps": self.degree, "l1_norm": self.l1, "walk_check_diff": self.walk_check_diff, "eps_target": self.eps,
               "t_prep": self.t_prep, "t_spectral_assumption": self.t_spectral_assumption, "rhs_norm": nb}
        if nb == 0: out.update({"x_ideal": np.zeros(N), "status": "zero_rhs"}); return out
        if self.over_budget:
            nanv = np.full(N, np.nan); out.update({"x_ideal": nanv, "x_shots": nanv, "x_aa": nanv, "status": "degree_cap", "x_exact": np.linalg.solve(self.A, b), "t_circuit_sim": 0.0,
                                                    "shot_records": [{"status": "degree_cap", "x_rec": nanv}], "aa_records": [{"status": "degree_cap", "x_rec": nanv}]}); return out
        t0 = time.perf_counter(); r_ref, Ar = self.reference(b); v = np.concatenate([b, Ar]); vnorm = np.linalg.norm(v)
        Vin = np.stack([b, Ar], axis=1) / vnorm
        if self.walk_check_diff is None:                                   # first solve: walk-level simulation and check of the spectral form
            Yw = self.apply_poly(Vin); Ys = self.apply_poly_spectral(Vin); self.walk_check_diff = float(np.max(np.abs(Yw - Ys)) / max(np.max(np.abs(Yw)), 1e-300))
            assert self.walk_check_diff < 1e-7, f"spectral form of the polynomial disagrees with the walk simulation ({self.walk_check_diff:.1e})"
            Y = Yw / self.l1
        else: Y = self.apply_poly_spectral(Vin) / self.l1                       # success-branch amplitudes (flag = 0 / 1 columns); LCU normalisation 1/||c||_1
        out["t_circuit_sim"] = time.perf_counter() - t0; y0, y1 = Y[:, 0], Y[:, 1]; p_s = float(y0 @ y0 + y1 @ y1); out["p_postselect_exact"] = p_s
        cfac = np.vdot(r_ref, y1) / np.vdot(r_ref, r_ref); out["x_ideal"] = (y0 / cfac) if abs(cfac) > 0 else np.full(N, np.nan)
        pZ = np.zeros((N, 2, 2)); pZ[:, 0, 1] = y0 ** 2; pZ[:, 1, 1] = y1 ** 2; pZ[0, 0, 0] += max(1.0 - p_s, 0.0)
        yp, ym = (y0 + y1) / np.sqrt(2), (y0 - y1) / np.sqrt(2); pX = np.zeros((N, 2, 2)); pX[:, 0, 1] = yp ** 2; pX[:, 1, 1] = ym ** 2; pX[0, 0, 0] += max(1.0 - p_s, 0.0)
        out["pZ"], out["pX"] = pZ.reshape(-1), pX.reshape(-1); out["r_ref"] = r_ref; out["v_norm"] = vnorm; out["b"] = b
        out["x_exact"] = np.linalg.solve(self.A, b) if x_exact is None else x_exact               # classical reference, diagnostic only
        if shots: recs = reconstruct_from_shots(self, out, shots, sampling_repeats, self.rng); out["x_shots"] = recs[0]["x_rec"]; out["shot_records"] = recs
        if eff_shots: recs = reconstruct_aa_emulated(self, out, eff_shots, sampling_repeats, self.rng); out["x_aa"] = recs[0]["x_rec"]; out["aa_records"] = recs
        return out

class CKSKind:
    """KKT-solver kind 'cks': Ruiz-scaled KKT matrix -> padded to 2^n -> CKS/LCU walk simulation -> readout; same protocol and accounting as 'hhl'."""
    def __init__(self, spec, K, asm, W, eng):
        Kd = K.toarray() if sp.issparse(K) else np.asarray(K, dtype=float)
        self.scaled = spec.get("scaling", "ruiz") == "ruiz"; self.d = ruiz_equilibrate(Kd) if self.scaled else None
        Ks = apply_scaling(Kd, self.d) if self.scaled else Kd; self.rng = rng_for("cks", asm.inst["id"], spec["name"])
        lam, _ = spectral_bound(Ks, CONFIG["quantum"]["spectral_bound"], self.rng); self.Ap, _, self.N0 = pad_to_pow2(Ks, np.zeros(Ks.shape[0]), lam)
        self.sys = CKSSystem(self.Ap, float(spec.get("eps", 1e-10)), self.rng, max_degree=CONFIG["quantum"].get("cks", {}).get("max_degree"), poly=spec.get("poly"))
        self.recon = spec.get("recon", "statevector"); self.shots = int(spec.get("shots", 0)); self.eff_shots = int(spec.get("eff_shots", 0))
    def base_solve(self, r, ctx):
        rs = self.d * r if self.scaled else r; rp = np.zeros(self.Ap.shape[0]); rp[:self.N0] = rs
        out = self.sys.run(rp, shots=(self.shots if self.recon == "shots" else 0), eff_shots=(self.eff_shots if self.recon == "aa_shots" else 0))
        if self.recon == "shots": up, rec = out["x_shots"], out["shot_records"][0]
        elif self.recon == "aa_shots": up, rec = out["x_aa"], out["aa_records"][0]
        else: up, rec = out["x_ideal"], None
        u = up[:self.N0]; u = self.d * u if self.scaled else u; xe = out["x_exact"]; nx = max(np.linalg.norm(xe), 1e-300)
        info = {"q_solver": "lcu_walk", "q_poly": self.sys.poly, "q_eps_circuit": self.sys.eps, "q_n_qubits": out["n_qubits"], "q_n_index": out["n_index"], "q_kappa_target": out["kappa"], "q_degree": out["degree"], "q_l1_norm": out["l1_norm"],
                "q_p_postselect": out.get("p_postselect_exact", np.nan), "q_walk_check_diff": self.sys.walk_check_diff, "q_status": out.get("status", "ok"), "q_rel_err_ideal_target": float(np.linalg.norm(out["x_ideal"] - xe) / nx) if np.all(np.isfinite(out["x_ideal"])) else np.inf,
                "q_rel_err_used_target": float(np.linalg.norm(up - xe) / nx) if np.all(np.isfinite(up)) else np.inf, "q_recon": self.recon, "q_shots": self.shots, "q_eff_shots": self.eff_shots,
                "q_t_prep": out["t_prep"], "q_t_circuit_sim": out.get("t_circuit_sim", np.nan), "q_scaling": "ruiz" if self.scaled else "none"}
        if rec is not None: info.update({"q_shot_status": rec["status"], "q_sign_errors": rec.get("n_sign_errors", np.nan), "q_aa_overhead": rec.get("aa_overhead_per_sample", np.nan)})
        return u, info
SOLVER_KINDS["cks"] = CKSKind


# Polynomial solver for an SPD target (null-space system). Shifted Chebyshev expansion of 1/x on [a, 1], a = 1/kappa, in
# y = (2x - (1+a))/(1-a): c_0 = sqrt(kappa), c_k = 2 sqrt(kappa) (-1)^k rho^k, rho = (sqrt(kappa)-1)/(sqrt(kappa)+1), so the degree
# is O(sqrt(kappa) log(kappa/eps)) against O(kappa log(1/eps)) for the odd polynomial of the indefinite augmented system.
# The walk acts on a block encoding of A~ = (2A/alpha - (1+a)I)/(1-a), with Pi W^k Pi = T_k(A~) for every k. The LCU
# normalisation ~ (3+a)/(1-a) of building that encoding from one of A is reported, not simulated.
def polynomial_degree_spd(kappa, eps):
    """Smallest degree whose Chebyshev tail bound  sqrt(kappa)(sqrt(kappa)+1) rho^(d+1)  is <= eps (closed form, no coefficients generated)."""
    rho = (math.sqrt(kappa) - 1.0) / (math.sqrt(kappa) + 1.0)
    if rho <= 0: return 1
    return max(1, int(math.ceil(math.log(eps / (math.sqrt(kappa) * (math.sqrt(kappa) + 1.0))) / math.log(rho))) - 1)

def spd_inverse_coefficients(kappa, eps):
    """Coefficients c_0..c_d of the shifted Chebyshev expansion of 1/x on [1/kappa, 1] truncated at the degree of polynomial_degree_spd, and the
    measured max |p(x) - 1/x| on a grid inside [1/kappa, 1] (denser near 1/kappa, where the error peaks)."""
    a = 1.0 / kappa; sk = math.sqrt(kappa); rho = (sk - 1.0) / (sk + 1.0); d = polynomial_degree_spd(kappa, eps)
    k = np.arange(d + 1); c = 2.0 * sk * (-1.0) ** k * rho ** k; c[0] = sk
    xg = np.unique(np.concatenate([np.linspace(a, min(3 * a, 1.0), 1500), np.geomspace(a, 1.0, 1500)])); y = (2 * xg - (1 + a)) / (1 - a)
    tp, tc = np.ones_like(y), y.copy(); acc = c[0] * tp + (c[1] * tc if d >= 1 else 0.0)
    for j in range(2, d + 1):
        tp, tc = tc, 2 * y * tc - tp; acc += c[j] * tc
    return c, d, float(np.max(np.abs(acc - 1.0 / xg)))

class SPDPolySystem(CKSSystem):
    """Same simulation protocol, readout and accounting as CKSSystem, for an SPD matrix with the shifted Chebyshev polynomial (all degrees, not only odd)."""
    def __init__(self, A, eps, rng, kappa_margin=1.05, max_degree=None, poly=None):
        t0 = time.perf_counter(); self.A = np.asarray(A, dtype=float); self.N = self.A.shape[0]; self.n_d = int(round(math.log2(self.N))); self.rng = rng
        ev = np.linalg.eigvalsh(self.A); self.t_spectral_assumption = time.perf_counter() - t0
        assert ev.min() > 0, "SPD polynomial solver needs a positive definite target"
        self.alpha = float(ev.max()) * (1 + 1e-9); self.kappa = float(ev.max() / ev.min()); self.eps = float(eps); self.poly = "chebyshev_spd"
        kap = self.kappa * kappa_margin; self.a = 1.0 / kap; d_est = polynomial_degree_spd(kap, self.eps)
        self.be_shift_inflation = (3.0 + self.a) / (1.0 - self.a)          # reported, not simulated
        if max_degree and d_est > max_degree:
            self.c = np.zeros(1); self.poly = "chebyshev_spd (not generated: degree cap)"; self.poly_err = float("nan"); self.j0 = d_est; self.degree = d_est; self.l1 = float("nan"); self.over_budget = True
            self.n_index = max(1, int(math.ceil(math.log2(d_est + 1)))); self.n_qubits = self.n_d + 1 + 1 + self.n_index + 1; self.t_prep = time.perf_counter() - t0; self.W = None; self.walk_check_diff = None; return
        self.c, self.degree, self.poly_err = spd_inverse_coefficients(kap, self.eps); self.j0 = self.degree; self.l1 = float(np.abs(self.c).sum())
        self.over_budget = bool(max_degree) and self.degree > max_degree
        At = (2.0 * self.A / self.alpha - (1.0 + self.a) * np.eye(self.N)) / (1.0 - self.a)
        U = block_encode(At); N = self.N; Wm = U.copy(); Wm[N:, :] *= -1.0; self.W = Wm
        self.n_index = max(1, int(math.ceil(math.log2(self.degree + 1)))); self.n_qubits = self.n_d + 1 + 1 + self.n_index + 1
        lam_, self._V = np.linalg.eigh(At); tp, tc = np.ones_like(lam_), lam_.copy(); pl = self.c[0] * tp + (self.c[1] * tc if self.degree >= 1 else 0.0)
        for j in range(2, self.degree + 1):
            tp, tc = tc, 2 * lam_ * tc - tp; pl += self.c[j] * tc
        self._plam = pl; self.walk_check_diff = None; self.t_prep = time.perf_counter() - t0
    def apply_poly(self, V):
        """Columns of V -> sum_k c_k T_k(A~) V accumulated from the ancilla-0 block of W^k (walk-level simulation, one walk step per term)."""
        N = self.N; st = np.vstack([V, np.zeros_like(V)]); acc = self.c[0] * V.copy()
        for k in range(1, self.degree + 1):
            st = self.W @ st; acc += self.c[k] * st[:N]
        return acc

class OSSPolyKind:
    """KKT-solver kind 'lcu_oss': the null-space SPD system D Z'SZ D (Jacobi-scaled, padded with lambda_max I to 2^n) is solved by the SPD
    polynomial solver; the direction is completed classically as in OSSKind (x_p, u_z, u_y) and verified on the full KKT system. Also logged:
    kappa and odd-polynomial degree that the Ruiz-scaled augmented system would need at the same eps."""
    def __init__(self, spec, K, asm, W, eng):
        self.cls = OSSKind(dict(spec, max_n=10 ** 9), K, asm, W, eng); M = self.cls.matrix(); self.rng = rng_for("lcu_oss", asm.inst["id"], spec["name"])
        lam_max = float(np.linalg.eigvalsh(M).max()); self.Mp, _, self.N0 = pad_to_pow2(M, np.zeros(M.shape[0]), lam_max)
        self.sys = SPDPolySystem(self.Mp, float(spec.get("eps", 1e-10)), self.rng, max_degree=CONFIG["quantum"].get("cks", {}).get("max_degree"))
        self.recon = spec.get("recon", "statevector"); self.shots = int(spec.get("shots", 0)); self.eff_shots = int(spec.get("eff_shots", 0))
        Kd = K.toarray() if sp.issparse(K) else np.asarray(K, dtype=float); Ks = apply_scaling(Kd, ruiz_equilibrate(Kd)); self.kappa_aug = float(np.linalg.cond(Ks))
        self.deg_aug = polynomial_degree(self.kappa_aug * 1.05, self.sys.eps, "optimal"); self.qubits_aug = int(math.ceil(math.log2(Kd.shape[0]))) + 3 + max(1, int(math.ceil(math.log2((self.deg_aug + 1) // 2 + 1))))
        self.Kd = Kd; self._lu = None
    def base_solve(self, r, ctx):
        xp_, b = self.cls.reduced_rhs(r); rp = np.zeros(self.Mp.shape[0]); rp[:self.N0] = b
        out = self.sys.run(rp, shots=(self.shots if self.recon == "shots" else 0), eff_shots=(self.eff_shots if self.recon == "aa_shots" else 0))
        if self.recon == "shots": vp, rec = out["x_shots"], out["shot_records"][0]
        elif self.recon == "aa_shots": vp, rec = out["x_aa"], out["aa_records"][0]
        else: vp, rec = out["x_ideal"], None
        v = vp[:self.N0]; u = self.cls.full(v, r, xp_) if np.all(np.isfinite(v)) else np.full(r.shape, np.nan)
        if self._lu is None: self._lu = sla.lu_factor(self.Kd, check_finite=False)
        ue = sla.lu_solve(self._lu, r, check_finite=False); nu = max(np.linalg.norm(ue), 1e-300); xe = out["x_exact"]; nx = max(np.linalg.norm(xe), 1e-300)
        info = {"q_solver": "lcu_oss_spd", "q_target": "null_space_spd", "q_poly": self.sys.poly, "q_eps_circuit": self.sys.eps, "q_n_qubits": out["n_qubits"], "q_n_index": out["n_index"], "q_kappa_target": out["kappa"], "q_degree": out["degree"], "q_l1_norm": out["l1_norm"],
                "q_p_postselect": out.get("p_postselect_exact", np.nan), "q_walk_check_diff": self.sys.walk_check_diff, "q_status": out.get("status", "ok"), "q_dim_reduced": self.N0, "q_N_padded": self.Mp.shape[0],
                "q_rel_err_ideal_target": float(np.linalg.norm(out["x_ideal"] - xe) / nx) if np.all(np.isfinite(out["x_ideal"])) else np.inf, "q_rel_err_used_target": float(np.linalg.norm(vp - xe) / nx) if np.all(np.isfinite(vp)) else np.inf,
                "q_rel_err_used_full": float(np.linalg.norm(u - ue) / nu) if np.all(np.isfinite(u)) else np.inf, "q_recon": self.recon, "q_shots": self.shots, "q_eff_shots": self.eff_shots,
                "q_t_prep": out["t_prep"], "q_t_circuit_sim": out.get("t_circuit_sim", np.nan), "q_scaling": "jacobi", "q_kappa_augmented": self.kappa_aug, "q_degree_augmented_equiv": self.deg_aug, "q_n_qubits_augmented_equiv": self.qubits_aug,
                "q_be_shift_inflation": self.sys.be_shift_inflation, "kappa_reduced": self.cls.kappa_reduced, "reduced_dim": self.N0}
        if rec is not None: info.update({"q_shot_status": rec["status"], "q_sign_errors": rec.get("n_sign_errors", np.nan), "q_aa_overhead": rec.get("aa_overhead_per_sample", np.nan)})
        return u, info
SOLVER_KINDS["lcu_oss"] = OSSPolyKind

# self-tests: (1) qubitisation identity, (2) accuracy, (3) explicit index-register statevector against the accumulated walk output
_cs = CKSSystem(_A, 1e-10, rng_for("cks_selftest"), poly="cks"); _N = _cs.N; _Wk = np.eye(2 * _N); _T0, _T1 = np.eye(_N), _A / _cs.alpha
for _k in range(1, 6):
    _Wk = _cs.W @ _Wk
    if _k > 1: _T0, _T1 = _T1, 2 * (_A / _cs.alpha) @ _T1 - _T0
    assert np.allclose(_Wk[:_N, :_N], _T1, atol=1e-12), "qubitized walk does not block-encode T_k(A)"
_co = _cs.run(_b, shots=20000, eff_shots=2000)
print(f"CKS self-test (N=4, kappa={_cs.kappa:.1f}, eps=1e-10): degree {_cs.degree}, {_cs.n_index} index qubits, ||c||_1={_cs.l1:.1f}, P(success)={_co['p_postselect_exact']:.2e}; "
      f"rel_err ideal={np.linalg.norm(_co['x_ideal'] - _xe) / np.linalg.norm(_xe):.2e} (HHL n_clock=6 above: {np.linalg.norm(_out['x_ideal'] - _xe) / np.linalg.norm(_xe):.2e}), "
      f"shots(2x20000)={_co['shot_records'][0]['rel_err']:.2e}, AA-emulated(2x2000)={_co['aa_records'][0]['rel_err']:.2e}")
assert np.linalg.norm(_co["x_ideal"] - _xe) / np.linalg.norm(_xe) < 1e-8
# explicit index-register statevector on a low-degree instance of the same construction (eps = 0.3, few terms)
_cg = CKSSystem(_A, 0.3, rng_for("cks_selftest2"), poly="cks"); _ni = _cg.n_index; _J = 2 ** _ni
_cpad = np.zeros(_J); _cpad[:_cg.j0 + 1] = _cg.c; _prepR = np.sqrt(np.abs(_cpad) / _cg.l1) * np.sign(np.where(_cpad == 0, 1.0, _cpad)); _prepL = np.sqrt(np.abs(_cpad) / _cg.l1)
_bn = _b / np.linalg.norm(_b); _P = _cg.W @ _cg.W
_state = np.kron(_prepR, np.kron([1.0, 0.0], _bn)).reshape(_J, 2 * _N)                      # explicit statevector of (index) x (ancilla, data)
_state = (_cg.W @ _state.T).T                                                             # W on every index branch
for _bit in range(_ni):                                                                   # SELECT = W * prod_bits controlled-(W^2)^(2^bit)
    for _jv in range(_J):
        if (_jv >> _bit) & 1: _state[_jv] = _P @ _state[_jv]
    _P = _P @ _P
_post = (_prepL[:, None] * _state).sum(axis=0)[:_N]                                       # PREP_L^dagger, post-select index = 0 and ancilla = 0
_walk = _cg.apply_poly((_bn)[:, None])[:, 0] / _cg.l1
print(f"  explicit index-register statevector ({_ni} index qubits, degree {_cg.degree}) vs walk-level accumulation: max |diff| = {np.max(np.abs(_post - _walk)):.1e}")
assert np.max(np.abs(_post - _walk)) < 1e-12

# optimal polynomial against the CKS expansion, on the same 4x4 system and over a (kappa, eps) grid
_co2 = CKSSystem(_A, 1e-10, rng_for("opt_selftest"), poly="optimal"); _oo = _co2.run(_b)
print(f"optimal polynomial (N=4, eps=1e-10): degree {_co2.degree} (CKS {_cs.degree}), ||c||_1 {_co2.l1:.1f} (CKS {_cs.l1:.1f}), P(success) {_oo['p_postselect_exact']:.2e} (CKS {_co['p_postselect_exact']:.2e}), "
      f"rel_err ideal {np.linalg.norm(_oo['x_ideal'] - _xe) / np.linalg.norm(_xe):.2e}; polynomial used: {_co2.poly}")
assert np.linalg.norm(_oo["x_ideal"] - _xe) / np.linalg.norm(_xe) < 1e-8
POLY_TABLE = []
for _kp in [5, 20, 100, 1000]:
    for _ep in [3e-1, 1e-2, 1e-6]:
        _c0, _ = cks_coefficients(_kp, _ep); _c1, _lab, _er = inverse_polynomial(_kp, _ep, "optimal")
        POLY_TABLE.append({"kappa": _kp, "eps": _ep, "degree_cks": 2 * len(_c0) - 1, "degree_optimal": 2 * len(_c1) - 1, "l1_cks": float(np.abs(_c0).sum()), "l1_optimal": float(np.abs(_c1).sum()), "optimal_measured_err": _er, "used": _lab})
POLY_TABLE = pd.DataFrame(POLY_TABLE); POLY_TABLE["degree_ratio"] = POLY_TABLE.degree_cks / POLY_TABLE.degree_optimal
POLY_TABLE["cost_ratio_degree_over_psuccess"] = POLY_TABLE.degree_ratio * (POLY_TABLE.l1_cks / POLY_TABLE.l1_optimal) ** 2
# degree and 1-norm of the SPD polynomial at the same kappa and eps: the effect of the target system alone
POLY_TABLE["degree_spd"] = [polynomial_degree_spd(k_, e_) for k_, e_ in zip(POLY_TABLE.kappa, POLY_TABLE.eps)]
POLY_TABLE["l1_spd"] = [float(np.abs(spd_inverse_coefficients(k_, e_)[0]).sum()) for k_, e_ in zip(POLY_TABLE.kappa, POLY_TABLE.eps)]
POLY_TABLE["spd_measured_err"] = [spd_inverse_coefficients(k_, e_)[2] for k_, e_ in zip(POLY_TABLE.kappa, POLY_TABLE.eps)]
POLY_TABLE["degree_ratio_optimal_over_spd"] = POLY_TABLE.degree_optimal / POLY_TABLE.degree_spd
print(POLY_TABLE.round(3).to_string(index=False))
# self-test of the SPD polynomial solver on a 4x4 SPD system (walk vs spectral form, accuracy, readout)
_As = _A @ _A + 0.5 * np.eye(4); _xs = np.linalg.solve(_As, _b); _ss = SPDPolySystem(_As, 1e-10, rng_for("spd_selftest")); _so = _ss.run(_b, shots=20000, eff_shots=2000)
print(f"SPD polynomial self-test (N=4, kappa={_ss.kappa:.1f}, eps=1e-10): degree {_ss.degree} (odd optimal polynomial at this kappa: {polynomial_degree(_ss.kappa * 1.05, 1e-10, 'optimal')}), ||c||_1={_ss.l1:.1f}, "
      f"P(success)={_so['p_postselect_exact']:.2e}, walk-vs-spectral diff {_ss.walk_check_diff:.1e}, rel_err ideal={np.linalg.norm(_so['x_ideal'] - _xs) / np.linalg.norm(_xs):.2e}, AA-emulated(2x2000)={_so['aa_records'][0]['rel_err']:.2e}; measured polynomial error {_ss.poly_err:.1e}")
assert np.linalg.norm(_so["x_ideal"] - _xs) / np.linalg.norm(_xs) < 1e-8 and _ss.walk_check_diff < 1e-7

## 16. HHL experiments

* **Q1, captured systems.** Clock qubits × scaling × reciprocal constant on the KKT systems captured in E1, with up to `ir_rounds` HHL refinement rounds on the true system and all three readouts.
* **Q2, Qiskit cross-check** of the simulator on the same circuit: statevector fidelity, transpiled depth and CX count, and the Aer shot sampler. Reported as `skipped` if Qiskit is not available.
* **Q3, HHL inside `coneqp`** with the combined test ($\eta = 0.1$, $\tau = 10^{-8}$, so $\omega \le 10^{-7}$) on the original system: without scaling and with $C = \Delta$, with Ruiz scaling and known $\kappa$, with and without fallback, with emulated amplitude amplification, and with the block test alone as an ablation. Each run records the directions accepted directly, accepted after refinement, used although rejected (only without fallback), fallbacks and failures.
* **Resource estimates** for the HHL circuits (qubits, gates, evolution time, post-selection probability, repetitions with and without amplitude amplification). These depend on the stated oracle assumptions and are kept apart from the measurements.

In [ ]:
# HHL experiments: Q1 captured systems, Q2 Qiskit cross-check, Q3 HHL inside coneqp, resource estimates.
QCFG = CONFIG["quantum"]

def q_row(base, **kw):
    r = dict(base); r.update(kw); REC.add("quantum_results", r); return r

def select_factor_indices(caps):
    idx = [c["factor_index"] for c in caps]
    if QCFG["captured_iters"] != "auto": return [i for i in QCFG["captured_iters"] if i in idx]
    L = len(idx); return sorted({idx[0], idx[L // 4], idx[L // 2], idx[(3 * L) // 4], idx[-1]})

def evaluate_captured_system(inst_id, cap, n_clock, scaling, c_mode, reg, shots_list, eff_shots_list, sampling_repeats, ir_rounds, asm_n):
    K = cap["K"]; solve0 = cap["solves"][0]; r = solve0["rhs"]; u_exact = solve0["u"]; N0 = K.shape[0]
    base = {"experiment": "Q1_captured", "instance_id": inst_id, "factor_index": cap["factor_index"], "kappa_kkt": cap["kappa"], "nt_proxy": cap["nt_proxy"], "kkt_dim": N0,
            "n_clock": n_clock, "scaling": scaling, "reciprocal_mode": c_mode, "regularization": reg, "engine": "minisim"}
    J = np.ones(N0); J[asm_n:] = -1.0
    Kt = K + reg * np.diag(J) if reg else K
    d = ruiz_equilibrate(Kt) if scaling == "ruiz" else None
    Ks = apply_scaling(Kt, d) if scaling == "ruiz" else Kt
    rng = rng_for("captured", inst_id, cap["factor_index"], n_clock, scaling, c_mode, reg)
    lam, _ = spectral_bound(Ks, QCFG["spectral_bound"], rng)
    Ap, _, _ = pad_to_pow2(Ks, np.zeros(N0), lam)
    n_q = int(round(math.log2(Ap.shape[0]))) + n_clock + 2
    if n_q > QCFG["max_qubits"] or 16 * 2 ** n_q > QCFG["max_state_bytes"]:
        q_row(base, status="skipped", reason=f"needs {n_q} qubits > max_qubits={QCFG['max_qubits']}", n_qubits=n_q); return None
    try:
        sysm = HHLSystem(Ap, n_clock, rng, spectral=QCFG["spectral_bound"], c_mode=c_mode)
    except ResourceLimit as e:
        q_row(base, status="skipped", reason=str(e)); return None
    nK = np.linalg.norm(K, 2); nr = np.linalg.norm(r)
    def solve_target(rhs):                      # one HHL run for a right-hand side of the original system; returns (u, out)
        rs = d * rhs if scaling == "ruiz" else rhs
        rp = np.zeros(Ap.shape[0]); rp[:N0] = rs
        out = sysm.run(rp); up = out["x_ideal"][:N0]
        return (d * up if scaling == "ruiz" else up), out
    def measures(u):
        e = K @ u - r; return {"rel_resid_original": float(np.linalg.norm(e) / nr), "backward_err_original": float(np.linalg.norm(e) / (nK * np.linalg.norm(u) + nr)), "rel_err_original": float(relnorm(u - u_exact, u_exact))}
    common = dict(status="ok", reason="", k_min=None, n_qubits=sysm.n_qubits, N_padded=Ap.shape[0], kappa_target=sysm.kappa, kappa_original=float(np.linalg.cond(K)),
                  lam_max_est=sysm.lam_max_est, lam_min_abs=sysm.lam_min_abs, clock_resolution=sysm.Delta, n_eig_below_resolution=sysm.n_eig_below_resolution,
                  n_eig_unresolved_10pct=sysm.n_eig_unresolved_10pct, t_prep=sysm.t_prep, t_diag=sysm.t_diag, t_spectral_assumption=sysm.t_spectral_assumption,
                  n_mc_rotations=len(sysm.thetas), evolution_time_x_norm=2 * (2 ** n_clock - 1) * sysm.t * sysm.lam_max_est)
    common["k_min"] = sysm.k_min
    # ideal statevector, iterative refinement on the true system
    u, out = solve_target(r); xe = out["x_exact"]; nx = max(np.linalg.norm(xe), 1e-300)
    common.update(p_postselect_exact=out["p_postselect_exact"], p_leak_exact=out["p_leak_exact"], gates_highlevel=out["n_gates_highlevel"], depth_highlevel=out["depth_highlevel"],
                  gate_counts=json.dumps(out["gate_counts"]), t_circuit_sim=out["t_circuit_sim"], t_readout_probs=out["t_readout_probs"], ideal_imag_fraction=out["ideal_imag_fraction"],
                  expected_repetitions_no_AA=1.0 / max(out["p_postselect_exact"], 1e-300), expected_repetitions_with_AA=1.0 / math.sqrt(max(out["p_postselect_exact"], 1e-300)),
                  scale_from_postselection_rel_err=abs(out["scale_from_postselection"] - nx) / nx)
    q_row(base, recon="ideal_statevector", ir_round=0, shots_per_setting=0, sampling_repeat=0, rel_err_target=float(np.linalg.norm(out["x_ideal"] - xe) / nx), **measures(u), **common)
    for rd in range(1, ir_rounds + 1):
        du, out2 = solve_target(r - K @ u); u = u + du; m = measures(u)
        q_row(base, recon="ideal_statevector", ir_round=rd, shots_per_setting=0, sampling_repeat=0, rel_err_target=np.nan, **m, **common)
        if m["rel_resid_original"] < 1e-12: break
    # finite-shot readouts at round 0
    for shots in shots_list:
        for rec in reconstruct_from_shots(sysm, out, shots, sampling_repeats, rng):
            if rec["status"] != "ok":
                q_row(base, recon="shots", ir_round=0, shots_per_setting=shots, sampling_repeat=rec["repeat"], status="failed", reason=rec["status"], postselect_rate_empirical=rec["postselect_rate"], **{k: v for k, v in common.items() if k not in ("status", "reason")}); continue
            us = rec["x_rec"][:N0]; us = d * us if scaling == "ruiz" else us; uls = rec["x_rec_ls"][:N0]; uls = d * uls if scaling == "ruiz" else uls
            q_row(base, recon="shots", ir_round=0, shots_per_setting=shots, sampling_repeat=rec["repeat"], rel_err_target=rec["rel_err"], rel_err_ls_rescale=rec["rel_err_ls"], rel_err_original=relnorm(us - u_exact, u_exact),
                  rel_resid_original=relnorm(K @ us - r, r), rel_resid_original_ls=relnorm(K @ uls - r, r), scale_rel_err=rec["scale_rel_err"], n_sign_errors=rec["n_sign_errors"],
                  postselect_rate_empirical=rec["postselect_rate"], postselected_shots=rec["postselected_shots"], t_reconstruct=rec["t_reconstruct"], **common)
    for eff in eff_shots_list:
        for rec in reconstruct_aa_emulated(sysm, out, eff, sampling_repeats, rng):
            if rec["status"] != "ok":
                q_row(base, recon="aa_shots", ir_round=0, shots_per_setting=eff, sampling_repeat=rec["repeat"], status="failed", reason=rec["status"], **{k: v for k, v in common.items() if k not in ("status", "reason")}); continue
            us = rec["x_rec"][:N0]; us = d * us if scaling == "ruiz" else us
            q_row(base, recon="aa_shots", ir_round=0, shots_per_setting=eff, sampling_repeat=rec["repeat"], rel_err_target=rec["rel_err"], rel_err_ls_rescale=rec["rel_err_ls"], rel_err_original=relnorm(us - u_exact, u_exact),
                  rel_resid_original=relnorm(K @ us - r, r), scale_rel_err=rec["scale_rel_err"], n_sign_errors=rec["n_sign_errors"], aa_overhead_per_sample=rec["aa_overhead_per_sample"],
                  raw_shots_equivalent_no_AA=rec["raw_shots_equivalent_no_AA"], postselected_shots=rec["postselected_shots"], t_reconstruct=rec["t_reconstruct"], **common)
    return out, sysm, Ap, N0, d

Q1_KEEP = {}
if not QCFG["enabled"]:
    REC.status("Q1_captured", "skipped", "quantum disabled in CONFIG")
elif not CAPTURED:
    REC.status("Q1_captured", "skipped", "no captured KKT systems")
else:
    t_q1 = time.perf_counter(); n_eval = 0
    for inst_id, caps in CAPTURED.items():
        if not inst_id.startswith("tiny"): continue
        asm_n = INSTANCES[inst_id]["n"]
        for fi in select_factor_indices(caps):
            cap = caps[fi]
            for n_clock in QCFG["n_clock_list"]:
                grid = [(sc, cm, 0.0) for sc in QCFG["scalings"] for cm in QCFG["reciprocal_modes"]]      # regularisation: Q4, Section 17
                for sc, cm, rg in grid:
                    res = evaluate_captured_system(inst_id, cap, n_clock, sc, cm, rg, QCFG["shots_list"], QCFG["eff_shots_list"], QCFG["sampling_repeats"], QCFG["ir_rounds"], asm_n)
                    n_eval += 1
                    if res is not None and sc == "ruiz" and cm == "kappa_known" and rg == 0.0: Q1_KEEP[(inst_id, fi, n_clock)] = res
    REC.flush(min_interval=0)
    REC.status("Q1_captured", "done", f"{n_eval} configurations in {time.perf_counter() - t_q1:.0f}s")
    qdf = REC.df("quantum_results"); ok = qdf[(qdf.experiment == "Q1_captured") & (qdf.status == "ok")]
    idl = ok[(ok.recon == "ideal_statevector") & (ok.ir_round == 0)]
    print("\nIdeal HHL error on the original system (median) by n_clock x (scaling, reciprocal, reg):")
    print(idl.pivot_table(index="n_clock", columns=["scaling", "reciprocal_mode", "regularization"], values="rel_err_original", aggfunc="median").round(4).to_string())
    print("\nPostselection probability (median):"); print(idl.pivot_table(index="n_clock", columns=["scaling", "reciprocal_mode", "regularization"], values="p_postselect_exact", aggfunc="median").round(5).to_string())
    irr = ok[(ok.recon == "ideal_statevector") & (ok.scaling == "ruiz") & (ok.reciprocal_mode == "kappa_known")]
    print("\nIterative refinement with HHL (delta = 0), residual on the true system (median) by round, largest n_clock:")
    print(irr[irr.n_clock == irr.n_clock.max()].groupby("ir_round").rel_resid_original.median().round(9).to_string())
    aa = ok[(ok.recon == "aa_shots") & (ok.scaling == "ruiz") & (ok.reciprocal_mode == "kappa_known") & (ok.regularization == 0.0)]
    print("\nAA-emulated readout error (median) by accepted samples per setting x n_clock:"); print(aa.pivot_table(index="shots_per_setting", columns="n_clock", values="rel_err_target", aggfunc="median").round(4).to_string())


# Q2: the same circuit in Qiskit (Ruiz scaling, kappa-known reciprocal)
def qiskit_crosscheck(inst_id, fi, n_clock, res):
    out, sysm, Ap, N0, d = res
    from qiskit import QuantumCircuit, transpile
    from qiskit.circuit.library import StatePreparation, UnitaryGate, QFTGate, RYGate
    from qiskit.quantum_info import Statevector
    base = {"experiment": "Q2_qiskit_crosscheck", "instance_id": inst_id, "factor_index": fi, "n_clock": n_clock, "scaling": "ruiz", "reciprocal_mode": sysm.c_mode, "regularization": 0.0, "engine": "qiskit",
            "kkt_dim": N0, "N_padded": Ap.shape[0], "n_qubits": sysm.n_qubits, "kappa_target": sysm.kappa}
    t0 = time.perf_counter(); qc = QuantumCircuit(sysm.n_qubits)
    b = out["b"]; r_ref, Ar = sysm.reference(b); v = np.concatenate([b, Ar]); v = v / np.linalg.norm(v)
    qc.append(StatePreparation(v.astype(complex)), sysm.dat + [sysm.flg])
    for q in sysm.clk: qc.h(q)
    for j, q in enumerate(sysm.clk): qc.append(UnitaryGate(sysm.U_pow[j]).control(1, annotated=False), [q] + sysm.dat)
    qc.append(QFTGate(sysm.n_c).inverse(), sysm.clk)
    for k, th in sysm.thetas.items(): qc.append(RYGate(th).control(sysm.n_c, ctrl_state=k, annotated=False), sysm.clk + [sysm.anc])
    qc.append(QFTGate(sysm.n_c), sysm.clk)
    for j in reversed(range(sysm.n_c)): qc.append(UnitaryGate(sysm.U_pow_dag[j]).control(1, annotated=False), [sysm.clk[j]] + sysm.dat)
    for q in sysm.clk: qc.h(q)
    t_build = time.perf_counter() - t0; t0 = time.perf_counter(); sv = Statevector.from_instruction(qc); t_sv = time.perf_counter() - t0
    sim = MiniSim(sysm.n_qubits); sim.prepare(v.astype(complex), sysm.dat + [sysm.flg])
    for q in sysm.clk: sim.h(q)
    for j, q in enumerate(sysm.clk): sim.cu(sysm.U_pow[j], q, sysm.dat)
    sim.iqft(sysm.clk)
    for k, th in sysm.thetas.items(): sim.mcry(th, sysm.clk, [(k >> j) & 1 for j in range(sysm.n_c)], sysm.anc)
    sim.qft(sysm.clk)
    for j in reversed(range(sysm.n_c)): sim.cu(sysm.U_pow_dag[j], sysm.clk[j], sysm.dat)
    for q in sysm.clk: sim.h(q)
    row = dict(base, status="ok", reason="", statevector_fidelity_vs_minisim=float(abs(np.vdot(sv.data, sim.psi)) ** 2), t_qiskit_build=t_build, t_qiskit_statevector=t_sv, qiskit_version=QISKIT_STATUS.get("version"))
    try:
        t0 = time.perf_counter(); tq = transpile(qc, basis_gates=["u", "cx"], optimization_level=1, seed_transpiler=GLOBAL_SEED); ops = dict(tq.count_ops())
        row.update(transpiled_depth=tq.depth(), transpiled_cx=ops.get("cx", 0), transpiled_u=ops.get("u", 0), transpiled_total=sum(ops.values()), t_transpile=time.perf_counter() - t0)
    except Exception as e:
        row.update(transpile_status=f"failed: {e!r}"[:200])
    if QISKIT_STATUS.get("aer"):
        try:
            from qiskit_aer import AerSimulator
            meas = [sysm.anc, sysm.flg] + sysm.dat
            def run_counts(circ, shots):
                c = circ.copy(); c.measure_all(); simr = AerSimulator(seed_simulator=GLOBAL_SEED); tc = transpile(c, simr)
                counts = simr.run(tc, shots=shots).result().get_counts(); p = np.zeros(2 ** len(meas))
                for bitstr, cnt in counts.items():
                    bits = bitstr.replace(" ", "")[::-1]; p[sum(int(bits[q]) << i for i, q in enumerate(meas))] += cnt
                return p / shots
            shots = int(QCFG["shots_list"][-1]); t0 = time.perf_counter()
            pZ = run_counts(qc, shots); qcX = qc.copy(); qcX.h(sysm.flg); pX = run_counts(qcX, shots); t_aer = time.perf_counter() - t0
            cZ = (pZ * shots).reshape(sysm.N, 2, 2); cX = (pX * shots).reshape(sysm.N, 2, 2)
            rec = reconstruct_from_counts(sysm, out, cZ, cX, shots, 0, "aer_shots")
            row.update(aer_shots_per_setting=shots, aer_rel_err=rec.get("rel_err", np.nan), aer_postselect_rate=rec.get("postselect_rate", np.nan), aer_status=rec["status"], t_aer_sampling=t_aer)
        except Exception as e:
            row.update(aer_status=f"failed: {e!r}"[:200])
    q_row(base, **{k: v for k, v in row.items() if k not in base}); return row

if not QCFG["enabled"] or not CAPTURED:
    REC.status("Q2_qiskit_crosscheck", "skipped", "quantum disabled or nothing captured")
elif not QISKIT_STATUS.get("available"):
    REC.status("Q2_qiskit_crosscheck", "skipped", f"qiskit not available: {QISKIT_STATUS.get('reason')}")
else:
    n_cc = 0; t_cc = time.perf_counter()
    for (inst_id, fi, n_clock), res in list(Q1_KEEP.items()):
        if n_clock > QCFG["qiskit_max_clock"] or res[1].n_qubits > QCFG["qiskit_max_qubits"]: continue
        if fi != select_factor_indices(CAPTURED[inst_id])[0]: continue
        try:
            row = qiskit_crosscheck(inst_id, fi, n_clock, res); n_cc += 1
            print(f"  qiskit check {inst_id} factor {fi} n_clock={n_clock}: fidelity={row['statevector_fidelity_vs_minisim']:.6f} depth={row.get('transpiled_depth')} cx={row.get('transpiled_cx')} aer_rel_err={row.get('aer_rel_err', float('nan')):.3e}")
        except Exception as e:
            q_row({"experiment": "Q2_qiskit_crosscheck", "instance_id": inst_id, "factor_index": fi, "n_clock": n_clock, "engine": "qiskit"}, status="failed", reason=repr(e)[:300]); print("  qiskit check failed:", repr(e)[:200])
    REC.status("Q2_qiskit_crosscheck", "done" if n_cc else "failed", f"{n_cc} circuits cross-checked in {time.perf_counter() - t_cc:.0f}s")
REC.flush(min_interval=0)


# Q3: HHL inside coneqp (combined test; block test alone as an ablation)
if not QCFG["enabled"]:
    REC.status("Q3_integrated", "skipped", "quantum disabled")
else:
    icfg = QCFG["integrated"]; n_int = 0; t_int = time.perf_counter()
    for inst_id in icfg["instances"]:
        inst = INSTANCES[inst_id]
        for rs in icfg["runs"]:
            if rs.get("instances_only") and inst_id not in rs["instances_only"]: continue
            spec = {"name": rs["name"], "kind": "hhl", "recon": rs["recon"], "n_clock": rs["n_clock"], "shots": rs.get("shots", 0), "eff_shots": rs.get("eff_shots", 0), "fallback": rs["fallback"],
                    "max_refine": icfg["max_refine"], "criterion": rs.get("criterion", icfg["accept_criterion"]), "eta": icfg["accept_eta"], "tol_rule": "fixed", "tol": icfg["accept_be_tol"],
                    "maxiters": icfg["maxiters"], "scaling": rs["scaling"], "reciprocal": rs["reciprocal"], "reg": rs["reg"], "switch_policy": bool(rs.get("switch_policy", False))}
            n_q = int(math.ceil(math.log2(inst["kkt_dim"]))) + rs["n_clock"] + 2
            if n_q > QCFG["max_qubits"]:
                REC.status(f"Q3_{inst_id}_{rs['name']}", "skipped", f"needs {n_q} qubits > max_qubits"); continue
            try:
                rec, sol, st = run_instance(inst, spec, repeat=0, compute_cond=True, experiment="Q3_integrated"); n_int += 1
                print(f"  {inst_id} {rs['name']}: status={rec['status']} iters={rec['iterations']} accepted={rec['accepted']} solves={rec['n_solve_calls']} direct={rec['n_solves_accepted_direct']} "
                      f"refined={rec['n_solves_accepted_refined']} unaccepted_used={rec['n_solves_unaccepted_used']} fallback={rec['n_fallback_total']} strict_quantum={rec['strict_method_success']} "
                      f"wall={rec['wall_method']:.1f}s obj_rel_err={rec.get('obj_rel_err', float('nan')):.2e}")
            except Exception as e:
                REC.failure(f"{inst_id}__{rs['name']}", "integrated_quantum", repr(e)[:300], method=rs["name"], instance_id=inst_id); print("  integrated run failed:", repr(e)[:200])
    REC.status("Q3_integrated", "done", f"{n_int} runs in {time.perf_counter() - t_int:.0f}s"); REC.flush(min_interval=0)
    dfq = REC.df("benchmark_results"); dfq = dfq[dfq.experiment == "Q3_integrated"]
    if len(dfq): print(dfq[["instance_id", "method", "criterion", "status", "iterations", "accepted", "n_solve_calls", "n_solves_accepted_direct", "n_solves_accepted_refined", "n_solves_unaccepted_used", "n_fallback_total", "strict_method_success", "obj_rel_err", "wall_method"]].to_string(index=False))


# resource estimates (depend on the stated assumptions)
qdf = REC.df("quantum_results")
if len(qdf) and "expected_repetitions_no_AA" in qdf:
    idl = qdf[(qdf.experiment == "Q1_captured") & (qdf.recon == "ideal_statevector") & (qdf.status == "ok") & (qdf.ir_round == 0)]
    if len(idl):
        RESOURCE_TABLE = idl.groupby(["instance_id", "n_clock", "scaling", "reciprocal_mode", "regularization"]).agg(
            kappa_target_med=("kappa_target", "median"), qubits=("n_qubits", "first"), highlevel_gates=("gates_highlevel", "first"), mc_rotations=("n_mc_rotations", "first"),
            evolution_time_x_norm=("evolution_time_x_norm", "median"), p_post_med=("p_postselect_exact", "median"), reps_no_AA_med=("expected_repetitions_no_AA", "median"),
            reps_with_AA_med=("expected_repetitions_with_AA", "median"), ideal_err_med=("rel_err_original", "median")).reset_index()
        RESOURCE_TABLE.to_csv(os.path.join(RESULTS_DIR, "quantum_resource_estimates.csv"), index=False)
        print("\nResource estimates written (assumption-dependent):", len(RESOURCE_TABLE), "rows")

### Q4, regularised target systems

On the tiny instances Ruiz scaling already brings $\kappa$ to about 35, so a regularisation sweep there tests little. Q4 uses every captured system whose scaled $\kappa$ is at least 50: late iterations of the tiny instances, of two dense instances with $\kappa(P) = 10^6$ (KKT dimension 64 and 128) and of SDPLIB `truss1`. $\delta \in \{0, 10^{-4}, 10^{-2}, 10^{-1}\}$ is added to the scaled system, so $\kappa(\tilde K_s + \delta J) \lesssim (1+\delta)/\delta$, and the direction is refined on the true system. The corrector is either LU of the regularised scaled system, a classical stand-in for any solver with bounded $\kappa$ that runs at all sizes, or the HHL statevector for padded dimension up to 32. Residuals are always those of the original system.

In [ ]:
# Q4: regularisation of the ill-conditioned late-path systems. delta is added to the Ruiz-scaled system and the direction is
# refined on the true system; the corrector is LU of the regularised system (classical diagnostic) or HHL (ideal statevector).
RCFG = QCFG["reg_experiment"]

def reg_experiment_on_system(inst_id, cap, asm_n, corrector="classical_lu", n_clock=None):
    K = cap["K"]; r = cap["solves"][0]["rhs"]; u_exact = cap["solves"][0]["u"]; N0 = K.shape[0]
    d = ruiz_equilibrate(K); Ks = apply_scaling(K, d); kappa_s = cond_exact(Ks, limit=RCFG["classical_max_dim"])
    if not (kappa_s >= RCFG["min_kappa_scaled"]): return "well_conditioned"
    nK = np.linalg.norm(K, 2); nr = np.linalg.norm(r); nu = max(np.linalg.norm(u_exact), 1e-300); J = np.ones(N0); J[asm_n:] = -1.0
    base = {"experiment": f"Q4_regularization_{'quantum' if corrector == 'hhl' else 'classical'}", "instance_id": inst_id, "factor_index": cap["factor_index"], "kappa_kkt": cap["kappa"],
            "kkt_dim": N0, "scaling": "ruiz", "reciprocal_mode": "kappa_known" if corrector == "hhl" else "", "corrector": ("hhl_ideal_statevector" if corrector == "hhl" else "classical_lu_regularized"),
            "engine": "minisim" if corrector == "hhl" else "classical_lu", "n_clock": n_clock if corrector == "hhl" else np.nan, "kappa_original": float(np.linalg.cond(K)), "kappa_scaled": float(kappa_s)}
    for delta in RCFG["deltas"]:
        Kd = Ks + delta * np.diag(J); kappa_d = cond_exact(Kd, limit=RCFG["classical_max_dim"])
        t0 = time.perf_counter()
        if corrector == "hhl":
            rng = rng_for("reg", inst_id, cap["factor_index"], delta, n_clock)
            lam, _ = spectral_bound(Kd, QCFG["spectral_bound"], rng); Ap, _, _ = pad_to_pow2(Kd, np.zeros(N0), lam)
            n_q = int(round(math.log2(Ap.shape[0]))) + n_clock + 2
            if n_q > QCFG["max_qubits"]:
                q_row(base, regularization=delta, status="skipped", reason=f"needs {n_q} qubits > max_qubits", kappa_target=kappa_d); continue
            try: sysm = HHLSystem(Ap, n_clock, rng, spectral=QCFG["spectral_bound"], c_mode="kappa_known")
            except ResourceLimit as e:
                q_row(base, regularization=delta, status="skipped", reason=str(e), kappa_target=kappa_d); continue
            t_prep = time.perf_counter() - t0
            def correct(e):
                rp = np.zeros(Ap.shape[0]); rp[:N0] = d * e; out = sysm.run(rp); return d * out["x_ideal"][:N0], out
        else:
            lu = sla.lu_factor(Kd, check_finite=False); t_prep = time.perf_counter() - t0
            def correct(e): return d * sla.lu_solve(lu, d * e, check_finite=False), None
        u = np.zeros(N0); t_rounds = 0.0; p_post = np.nan; t_circ = 0.0
        for rd in range(RCFG["ir_rounds"] + 1):
            t1 = time.perf_counter(); du, out = correct(r - K @ u); u = u + du; t_rounds += time.perf_counter() - t1
            if out is not None: p_post = out["p_postselect_exact"]; t_circ += out["t_circuit_sim"]
            e = K @ u - r; res = float(np.linalg.norm(e) / nr)
            q_row(base, regularization=delta, kappa_target=float(kappa_d), status="ok", reason="", recon="ideal_statevector" if corrector == "hhl" else "classical", ir_round=rd,
                  rel_resid_original=res, backward_err_original=float(np.linalg.norm(e) / (nK * np.linalg.norm(u) + nr)), rel_err_original=float(np.linalg.norm(u - u_exact) / nu),
                  t_prep=t_prep, t_rounds_cum=t_rounds, t_circuit_sim=t_circ, p_postselect_exact=p_post, n_qubits=(sysm.n_qubits if corrector == "hhl" else np.nan))
            if res < 1e-13 or not np.isfinite(res): break
    return "done"

def run_reg_experiment(capture_dict, label):
    n_sys, n_skipped = 0, 0
    for inst_id, caps in capture_dict.items():
        if inst_id not in INSTANCES: continue
        asm_n = INSTANCES[inst_id]["n"]
        for cap in caps:
            if cap["K"].shape[0] > RCFG["classical_max_dim"]: n_skipped += 1; continue
            st = reg_experiment_on_system(inst_id, cap, asm_n, corrector="classical_lu")
            if st != "done": continue
            n_sys += 1
            N0 = cap["K"].shape[0]
            if 2 ** int(math.ceil(math.log2(N0))) <= RCFG["quantum_max_dim"]:
                for nc in RCFG["quantum_n_clock"]: reg_experiment_on_system(inst_id, cap, asm_n, corrector="hhl", n_clock=nc)
    REC.status(f"Q4_regularization_{label}", "done", f"{n_sys} ill-conditioned systems (kappa_scaled >= {RCFG['min_kappa_scaled']}); {n_skipped} above classical_max_dim")
    REC.flush(min_interval=0)

if QCFG["enabled"] and CAPTURED:
    run_reg_experiment({k: v for k, v in CAPTURED.items() if not k.startswith("sdplib")}, "qp")
    q4 = REC.df("quantum_results"); q4 = q4[q4.experiment.str.startswith("Q4_")]
    if len(q4):
        print("Q4 (classical LU corrector): median residual on the true system by refinement round x delta")
        c4 = q4[q4.corrector == "classical_lu_regularized"]
        print(c4.pivot_table(index="ir_round", columns="regularization", values="rel_resid_original", aggfunc="median").to_string())
        print("kappa of the regularised scaled target (median) by delta:", c4[c4.ir_round == 0].groupby("regularization").kappa_target.median().round(1).to_dict())
        h4 = q4[q4.corrector == "hhl_ideal_statevector"]
        if len(h4):
            print("Q4 (HHL ideal-statevector corrector): median residual by round x delta, per n_clock")
            print(h4.pivot_table(index=["n_clock", "ir_round"], columns="regularization", values="rel_resid_original", aggfunc="median").to_string())
else:
    REC.status("Q4_regularization_qp", "skipped", "quantum disabled or nothing captured")

## 18. Diagnostics

* **D1, dequantisation probe.** For every captured system, the fraction of singular directions that a truncated solve needs to reproduce the Newton direction to 30 %, 10 % and 1 %, and the stable rank. A low-rank structure would let a sampling-based classical algorithm compete with a quantum solver.
* **E5, optimisation-level iterative refinement** (Gleixner et al.). The QP is solved to $10^{-3}$, shifted and scaled around the current point by $\Delta$, and the correction QP is solved again, five times. Inner solvers: CVXOPT's own, exact solves with injected error, loosely accepted MINRES (combined test, $\tau = 10^{-5}$), and HHL without fallback. The question is whether an outer loop makes a low-accuracy KKT solver sufficient.
* **E6, mat-vec benchmark.** CPU and GPU time of one sparse KKT mat-vec at every size of `sparse_banded`; `skipped` without a GPU.

Instances without equality constraints also log $\kappa(S)$ of the reduced system and $\kappa(B) = \sqrt{\kappa(S)}$ of its square-root factor next to $\kappa(\tilde K)$ and $\kappa(D\tilde K D)$ (Section 9), so the choice of target system is measured.

In [ ]:
# Diagnostics: D1 dequantisation probe, E5 optimisation-level refinement, E6 CPU/GPU mat-vec benchmark.

# D1: fraction of singular directions a truncated solve needs to reproduce the captured Newton direction
rows = []
for inst_id, caps in CAPTURED.items():
    for cap in caps:
        K = cap["K"]; r = cap["solves"][0]["rhs"]; u = cap["solves"][0]["u"]; N = K.shape[0]
        U_, sv, Vt = np.linalg.svd(K); Utb = U_.T @ r; nu = max(np.linalg.norm(u), 1e-300); need = {}
        for dlt in (0.3, 0.1, 0.01):
            need[dlt] = N
            for k in range(1, N + 1):
                if np.linalg.norm(Vt[:k].T @ (Utb[:k] / sv[:k]) - u) / nu <= dlt: need[dlt] = k; break
        rows.append({"instance_id": inst_id, "factor_index": cap["factor_index"], "kkt_dim": N, "kappa": float(sv[0] / max(sv[-1], 1e-300)),
                     "stable_rank": float((sv ** 2).sum() / sv[0] ** 2), **{f"frac_for_{d}": need[d] / N for d in need}})
DEQ = pd.DataFrame(rows); DEQ.to_csv(os.path.join(RESULTS_DIR, "dequantisation_probe.csv"), index=False)
if len(DEQ):
    print(f"D1 dequantisation probe on {len(DEQ)} captured systems: median fraction of singular directions needed for 30% / 10% / 1% accuracy = "
          f"{DEQ['frac_for_0.3'].median():.2f} / {DEQ['frac_for_0.1'].median():.2f} / {DEQ['frac_for_0.01'].median():.2f}; median stable rank / N = {(DEQ.stable_rank / DEQ.kkt_dim).median():.2f}")
REC.status("D1_dequantisation", "done" if len(DEQ) else "skipped", f"{len(DEQ)} systems")


# E5: optimisation-level iterative refinement (loose inner solves, shift-and-scale outer loop)
def outer_refinement(inst, spec=None, rounds=5, inner_tol=1e-3, label="cvxopt_default", target=1e-7):
    """x <- x + d/Delta, y <- y + yhat/Delta, z <- w/Delta, where (d, yhat, w) solves the shifted/scaled QP
         min 1/2 d'Pd + Delta (P x + q + A'y)' d   s.t.  G d <= Delta (h - G x),  A d = Delta (b - A x)
    to the loose tolerance inner_tol. The reformulation is exact; only the inner accuracy is low. `spec` selects the inner KKT solver."""
    P, q, G, h, A, b, p = inst["P"], inst["q"], inst["G"], inst["h"], inst["A"], inst["b"], inst["p"]
    x = np.zeros(inst["n"]); y = np.zeros(p); Delta = 1.0; out = []; tot_it = 0; tot = {"solves": 0, "fallback": 0, "rejected_used": 0, "refine": 0, "krylov": 0}
    for k in range(rounds):
        sub = {kk: vv for kk, vv in inst.items() if kk != "_cvx"}
        sub.update({"q": Delta * (P @ x + q + (A.T @ y if p else 0.0)), "h": Delta * (h - G @ x), "b": (Delta * (b - A @ x)) if p else None, "known_solution": False})
        cd = cvx_data(sub); st = {"factor_calls": 0, "solve_calls": 0, "factors": [], "solves": [], "captured": [], "compute_cond": True, "capture": False, "cond_scaled": True}     # also kappa of the Ruiz-scaled inner systems
        kkt = make_kktsolver(sub, spec, st) if spec is not None else None
        solvers.options.clear(); solvers.options.update({"show_progress": False, "abstol": inner_tol, "reltol": inner_tol, "feastol": inner_tol, "maxiters": 60, "refinement": 0})
        try:
            sol = solvers.coneqp(cd["P"], cd["q"], cd["G"], cd["h"], inst["dims"], cd["A"], cd["b"], kktsolver=kkt)
        except Exception as e:
            out.append({"instance_id": inst["id"], "inner_solver": label, "round": k, "status": f"error: {type(e).__name__}"}); break
        finally:
            solvers.options.clear()
        x = x + to_np(sol["x"]) / Delta; y = y + (to_np(sol["y"]) / Delta if p else 0.0); z = to_np(sol["z"]) / Delta; tot_it += sol["iterations"]
        S_ = st["solves"]; tot["solves"] += len(S_); tot["fallback"] += sum(s_["fallback"] for s_ in S_); tot["refine"] += sum(s_["n_refine"] for s_ in S_); tot["krylov"] += sum(s_.get("krylov_iters", 0) for s_ in S_)
        tot["rejected_used"] += sum((not s_["accepted"]) and (not s_["fallback"]) for s_ in S_)
        m = recompute_metrics_qp(inst, {"x": x, "y": y, "z": z, "s": np.maximum(h - G @ x, 0.0)}); err = max(m["pres_rec"], m["dres_rec"], m["relgap_valid"])
        kap = [f["kappa"] for f in st["factors"] if np.isfinite(f["kappa"])]; kaps_ = [f["kappa_scaled"] for f in st["factors"] if np.isfinite(f.get("kappa_scaled", np.nan))]
        out.append({"instance_id": inst["id"], "inner_solver": label, "inner_tol": inner_tol, "round": k, "status": sol["status"], "Delta": Delta, "inner_iterations": sol["iterations"], "cum_iterations": tot_it,
                    "pres": m["pres_rec"], "dres": m["dres_rec"], "relgap_valid": m["relgap_valid"], "cone_viol": m["cone_viol"], "obj_rel_err": m["obj_rel_err"], "err": err,
                    "kappa_max_inner": max(kap) if kap else np.nan, "kappa_scaled_max_inner": max(kaps_) if kaps_ else np.nan, "cum_solves": tot["solves"], "cum_fallback": tot["fallback"], "cum_rejected_used": tot["rejected_used"], "cum_refine": tot["refine"], "cum_krylov": tot["krylov"],
                    "reached_target": bool(err <= target and m["obj_rel_err"] <= CONFIG["accept"]["obj_rel_err"])})
        if err <= 1e-13: break
        Delta = float(min(Delta * 1e2, 1.0 / max(err, 1e-14)))
    return out

loose = {"criterion": "combined", "eta": 0.1, "tol_rule": "fixed", "tol": 1e-5, "scaling": "ruiz", "fallback": False, "max_refine": 4}
INNER = [("cvxopt_default", None),
         ("exact_noise_iso_1e-4", {"name": "or_noise_1e-4", "kind": "noise", "noise_model": "iso", "delta": 1e-4, "fallback": False, "max_refine": 0, "criterion": "relres"}),
         ("exact_noise_iso_1e-3", {"name": "or_noise_1e-3", "kind": "noise", "noise_model": "iso", "delta": 1e-3, "fallback": False, "max_refine": 0, "criterion": "relres"}),
         ("minres_base_loose_1e-5", {"name": "or_minres_base_loose", "kind": "minres_base", "backend": "cpu", "warm_start": True, **loose})]
ref_rows = []
or_insts = [i for i in INSTANCES.values() if i["family"] == "dense_kappa" and i["seed"] == 0 and i["n"] <= 20] + [INSTANCES[t] for t in ("tiny_A", "tiny_B", "tiny_C")]
for inst in or_insts:
    for label, spec in INNER: ref_rows += outer_refinement(inst, spec, label=label)
if QCFG_OUTER := CONFIG["quantum"]["enabled"]:
    for tid in CONFIG["quantum"]["integrated"]["instances"]:
        for nc in sorted({8, max(CONFIG["quantum"]["n_clock_list"])}):
            if int(math.ceil(math.log2(INSTANCES[tid]["kkt_dim"]))) + nc + 2 > CONFIG["quantum"]["max_qubits"]: continue
            hspec = {"name": f"or_hhl_nc{nc}", "kind": "hhl", "recon": "statevector", "n_clock": nc, "reciprocal": "kappa_known", "reg": 0.0, **loose}
            ref_rows += outer_refinement(INSTANCES[tid], hspec, label=f"hhl_ideal_nc{nc}_nofallback")
OREF = pd.DataFrame(ref_rows); OREF.to_csv(os.path.join(RESULTS_DIR, "outer_refinement.csv"), index=False)
if len(OREF) and "err" in OREF:
    print("E5 outer refinement, inner tolerance 1e-3: median max(pres, dres, relgap) by round and inner solver")
    print(OREF.pivot_table(index="round", columns="inner_solver", values="err", aggfunc="median").to_string())
    last = OREF.sort_values("round").groupby(["instance_id", "inner_solver"]).tail(1)
    print(last.groupby("inner_solver").agg(instances=("instance_id", "size"), reached_1e7=("reached_target", "mean"), cum_iters=("cum_iterations", "median"), cum_krylov=("cum_krylov", "median"), kappa_max=("kappa_max_inner", "median"), kappa_scaled_max=("kappa_scaled_max_inner", "median"),
                                           rejected_used=("cum_rejected_used", "sum"), fallback=("cum_fallback", "sum")).round(3).to_string())
REC.status("E5_outer_refinement", "done", f"{OREF.instance_id.nunique() if len(OREF) else 0} instances x {OREF.inner_solver.nunique() if len(OREF) else 0} inner solvers")


# E6: CPU and GPU mat-vec time on the sparse KKT matrices (W = I)
mv_rows = []
if GPU_OK:
    for inst in sorted([i for i in INSTANCES.values() if i["family"] == "sparse_banded" and i["seed"] == 0], key=lambda i: i["kkt_dim"]):
        asm = KKTAssembler(inst); W1 = {"d": matrix(1.0, (inst["m"], 1)), "di": matrix(1.0, (inst["m"], 1)), "v": [], "beta": [], "r": [], "rti": []}
        K = asm.build(W1); v = np.random.default_rng(0).standard_normal(K.shape[0]); Kd = to_device(K); vd = DEVICE_XP.asarray(v); reps = 200
        t0 = time.perf_counter(); [K @ v for _ in range(reps)]; t_cpu = (time.perf_counter() - t0) / reps
        Kd @ vd; dev_sync(); t0 = time.perf_counter(); [Kd @ vd for _ in range(reps)]; dev_sync(); t_gpu = (time.perf_counter() - t0) / reps
        mv_rows.append({"instance_id": inst["id"], "kkt_dim": K.shape[0], "nnz": int(K.nnz), "t_matvec_cpu": t_cpu, "t_matvec_gpu": t_gpu, "speedup_gpu": t_cpu / t_gpu, "device": GPU_STATUS.get("name")})
    MV = pd.DataFrame(mv_rows); MV.to_csv(os.path.join(RESULTS_DIR, "gpu_matvec_benchmark.csv"), index=False); print(MV.round(7).to_string(index=False))
    REC.status("E6_gpu_matvec", "done", f"{len(MV)} sizes on {GPU_STATUS.get('name')}")
else:
    MV = pd.DataFrame(); REC.status("E6_gpu_matvec", "skipped", f"device={CONFIG['device']}; {GPU_STATUS.get('reason') or 'GPU not requested'}")
REC.flush(min_interval=0)

## 19. Conditioning along one interior-point run (E7)

The total number of MINRES iterations per run grows from about 1 500 to 38 000 as the KKT dimension grows from 310 to 79 360. E7 captures every $W$ and right-hand side of one run of `sparse_banded` and computes, per IPM iteration:

1. $\kappa$ of the Ruiz-scaled KKT matrix, the share of eigenvalues near $\pm 1$ and the number of small eigenvalues, to separate ill-conditioning from the eigenvalue distribution; and which test binds (iterations with the backward-error test alone against the combined test, and the block with the largest ratio);
2. MINRES on the matrix-free, Jacobi-scaled reduced system $[[P + \hat G^\top\hat G, A^\top], [A, 0]]$, whose vectors are 2.8 times shorter. $\kappa(S)$ contains $W^{-2}$, so the reduced target should win early in the run and lose late;
3. at the last IPM iteration, MINRES with the $k$ eigenvectors of smallest $|\lambda|$ deflated exactly. This bounds what any recycling or deflation method can gain, since those use only approximations of these vectors.

In [ ]:
# E7: spectrum of the scaled KKT matrix along one IPM run, augmented against reduced target, exact-deflation bound.
def e7_spectrum(n_target):
    cands = sorted([i for i in INSTANCES.values() if i["family"] == "sparse_banded" and i["n"] <= n_target], key=lambda i: i["n"])
    if not cands: return pd.DataFrame(), pd.DataFrame()
    inst = cands[-1]; n, p, m = inst["n"], inst["p"], inst["m"]; P, G = inst["P"].tocsr(), inst["G"].tocsr(); A = inst["A"].tocsr() if p else None
    rs = {"factor_calls": 0, "solve_calls": 0, "factors": [], "solves": [], "captured": [], "compute_cond": False, "capture": False, "cond_scaled": False}
    base = make_kktsolver(inst, SPECS["custom_exact_reduced"], rs); asm0 = KKTAssembler(inst); cap = []
    def kkt(W):
        f = base(W); rec = {"di": to_np(W["di"]).copy(), "rhs": []}; cap.append(rec)
        def solve(x, y, z):
            rec["rhs"].append(asm0.rhs(x, y, z, W).copy()); f(x, y, z)
        return solve
    cd = cvx_data(inst); solvers.options.clear(); solvers.options.update({"show_progress": False, "maxiters": CONFIG["cvxopt"]["maxiters"]})
    sol = solvers.coneqp(cd["P"], cd["q"], cd["G"], cd["h"], inst["dims"], cd["A"], cd["b"], kktsolver=kkt); solvers.options.clear()
    sl = [q for q in (slice(0, n), slice(n, n + p), slice(n + p, n + p + m)) if q.stop > q.start]
    rows, last = [], None
    for k, rec in enumerate(cap):
        di = rec["di"]; Gh = (sp.diags(di) @ G).tocsr()
        K = sp.bmat([[P, A.T, Gh.T], [A, None, None], [Gh, None, -sp.eye(m)]], format="csr") if p else sp.bmat([[P, Gh.T], [Gh, -sp.eye(m)]], format="csr")
        nK = norm2_estimate(K); d = ruiz_equilibrate(K); Ks = apply_scaling(K, d).tocsr()
        def crit(u, r):
            e = K @ u - r; be = np.linalg.norm(e) / (nK * np.linalg.norm(u) + np.linalg.norm(r))
            return be <= 1e-9 and max(np.linalg.norm(e[q]) / max(np.linalg.norm(r[q]), 1e-9) for q in sl) <= 0.1
        ev = np.linalg.eigvalsh(Ks.toarray()) if K.shape[0] <= CONFIG["cond_limit"] * 3 else None
        a = np.abs(ev) if ev is not None else None
        # reduced, matrix-free, Jacobi-scaled
        dS = P.diagonal() + np.asarray(Gh.multiply(Gh).sum(axis=0)).ravel(); e1 = 1.0 / np.sqrt(dS)
        if p: AD = A @ sp.diags(e1); e2 = 1.0 / np.sqrt(np.maximum(np.asarray(AD.multiply(AD).sum(axis=1)).ravel(), 1e-300)); E = np.concatenate([e1, e2])
        else: E = e1
        GhT = Gh.T.tocsr(); AT = A.T.tocsr() if p else None
        def Mv(v):
            vx = v[:n]; t = P @ vx + GhT @ (Gh @ vx)
            return np.concatenate([t + AT @ v[n:], A @ vx]) if p else t
        ia = ir = ib = 0; ybind = []
        def be_only(u, r):
            e = K @ u - r; return np.linalg.norm(e) / (nK * np.linalg.norm(u) + np.linalg.norm(r)) <= 1e-9
        for r in rec["rhs"]:
            xb, _, its, _, _ = minres_fused_kernel(lambda v: Ks @ v, d * r, np, maxiter=50 * K.shape[0], stop_fn=lambda xk: be_only(d * xk, r), check_every=4, adaptive_div=10 ** 9); ib += its
            eb = K @ (d * xb) - r; ybind.append(max(range(len(sl)), key=lambda j: np.linalg.norm(eb[sl[j]]) / max(np.linalg.norm(r[sl[j]]), 1e-9)))
            _, _, its, _, _ = minres_fused_kernel(lambda v: Ks @ v, d * r, np, maxiter=50 * K.shape[0], stop_fn=lambda xk: crit(d * xk, r), check_every=4, adaptive_div=10 ** 9); ia += its
            bz = r[n + p:]; b = np.concatenate([r[:n] + GhT @ bz, r[n:n + p]])
            def full(xk):
                w_ = E * xk; ux = w_[:n]; return np.concatenate([ux, w_[n:], Gh @ ux - bz])
            _, _, its, _, _ = minres_fused_kernel(lambda v: E * Mv(E * v), E * b, np, maxiter=50 * K.shape[0], stop_fn=lambda xk: crit(full(xk), r), check_every=4, adaptive_div=10 ** 9); ir += its
        rows.append({"instance_id": inst["id"], "kkt_dim": K.shape[0], "ipm_iteration": k, "w_spread": float(di.max() / di.min()), "kappa_scaled": float(a.max() / a.min()) if a is not None else np.nan,
                     "n_eig_below_0.1": int((a < 0.1).sum()) if a is not None else -1, "n_eig_below_0.01": int((a < 0.01).sum()) if a is not None else -1,
                     "frac_eig_in_0.9_1.1": float(((a > 0.9) & (a < 1.1)).mean()) if a is not None else np.nan, "minres_its_backward_only": ib, "binding_block_when_backward_holds": "/".join("xyz"[j] if p else "xz"[j] for j in ybind), "minres_its_augmented": ia, "minres_its_reduced_jacobi": ir})
        last = (K, nK, d, Ks, rec, crit)
    drows = []
    if last is not None and last[0].shape[0] <= CONFIG["cond_limit"] * 3:
        K, nK, d, Ks, rec, crit = last; ev, V = np.linalg.eigh(Ks.toarray()); o = np.argsort(np.abs(ev)); N = len(ev); r = rec["rhs"][-1]; b = d * r
        for kdef in [0, 5, 10, 25, 50, 100, 200]:
            if kdef >= N // 4: break
            Z = V[:, o[:kdef]]; lam = ev[o[:kdef]]; x0 = Z @ ((Z.T @ b) / lam) if kdef else np.zeros(N)
            bb = b - Ks @ x0; bb = bb - Z @ (Z.T @ bb) if kdef else bb
            op = (lambda v: (lambda w_: w_ - Z @ (Z.T @ w_))(Ks @ v)) if kdef else (lambda v: Ks @ v)
            _, _, its, _, _ = minres_fused_kernel(op, bb, np, maxiter=50 * N, stop_fn=lambda xk: crit(d * (x0 + xk), r), check_every=4, adaptive_div=10 ** 9)
            drows.append({"instance_id": inst["id"], "kkt_dim": N, "n_deflated": kdef, "frac_deflated": kdef / N, "minres_its": its, "kappa_remaining": float(np.abs(ev).max() / np.abs(ev[o[kdef]]))})
    return pd.DataFrame(rows), pd.DataFrame(drows)

try:
    with Timer() as _t7:
        SPEC7, DEFL7 = e7_spectrum({"smoke": 200, "small": 400}.get(RUN_PROFILE, 800))
    SPEC7.to_csv(os.path.join(RESULTS_DIR, "spectrum_along_ipm.csv"), index=False); DEFL7.to_csv(os.path.join(RESULTS_DIR, "deflation_bound.csv"), index=False)
    print(SPEC7.round(3).to_string(index=False)); print(); print(DEFL7.round(4).to_string(index=False))
    REC.status("E7_spectrum_deflation", "done" if len(SPEC7) else "skipped", f"{len(SPEC7)} IPM iterations, {len(DEFL7)} deflation levels in {_t7.t:.0f}s")
except Exception as _e:
    SPEC7, DEFL7 = pd.DataFrame(), pd.DataFrame(); REC.status("E7_spectrum_deflation", "failed", f"{type(_e).__name__}: {_e}"[:300]); traceback.print_exc()

## 20. Polynomial-solver experiments (Q5)

* **Q5a.** On the captured KKT systems: forward error of HHL (largest clock register, ideal readout) against the polynomial solver at $\epsilon = 10^{-3}, 10^{-6}, 10^{-10}$, with walk steps, qubits, $\|c\|_1$ and post-selection probability.
* **Q5b.** Total cost (samples, block-encoding queries) of refining a finite-shot solve to a residual of $10^{-10}$, over circuit precision $\epsilon$ and samples per round, for both polynomials.
* **Q5c.** The polynomial solvers inside `coneqp` without classical fallback, with the combined test and the accounting of Q3, together with HHL at the same refinement budget, and the null-space solver `lcu_oss_*`.
* **Q5d.** On the same captured systems, the null-space target: dimension $n - p$ instead of $n + m + p$, Jacobi-scaled $\kappa$, the shifted-Chebyshev degree next to the odd-polynomial degree of the augmented system, qubits, and the forward error of the full KKT direction (`oss_target_comparison.csv`).
* **Q5e.** The finite-shot operating point ($\epsilon = 0.3$) over sampling seeds × accepted samples per round, so the success rate comes with a spread.

A run whose polynomial degree exceeds `max_degree` is reported as over the simulation budget; it is never replaced by a classical solve.

In [ ]:
# Q5: polynomial solver. Q5a/Q5d captured systems (augmented and null-space target), Q5b refinement cost grid, Q5c runs
# inside coneqp without fallback (with HHL at the same refinement budget), Q5e finite-shot runs over sampling seeds.
CKSCFG = QCFG.get("cks", {}); Q5A, Q5B, Q5D = [], [], []
def poly_tag(p_):
    p_ = str(p_); return "not-run(cap)" if "not generated" in p_ else ("cks-fallback" if "fallback" in p_ else p_)
if not QCFG.get("enabled", True) or not CKSCFG:
    REC.status("Q5_cks", "skipped", "quantum disabled")
else:
    t5 = time.perf_counter(); n_ir = 0
    for inst_id, caps in CAPTURED.items():
        if inst_id.startswith("sdplib") or not caps: continue
        for fi in sorted({0, len(caps) // 2, len(caps) - 1}):
            K = np.asarray(caps[fi]["K"], dtype=float)
            if K.shape[0] > 64 or not caps[fi]["solves"]: continue
            r = caps[fi]["solves"][-1]["rhs"]; d = ruiz_equilibrate(K); Ks = apply_scaling(K, d); lam, _ = spectral_bound(Ks, "exact", rng_for("q5", inst_id, fi))
            Ap, _, N0 = pad_to_pow2(Ks, np.zeros(K.shape[0]), lam); rp = np.zeros(Ap.shape[0]); rp[:N0] = d * r; u_true = np.linalg.solve(K, r); nu = np.linalg.norm(u_true); bec = block_encoding_costs(Ap)
            base = {"instance_id": inst_id, "factor_index": fi, "kkt_dim": K.shape[0], "N_padded": Ap.shape[0], "kappa_original": float(np.linalg.cond(K)),
                    "be_inflation_sparse_access": bec["alpha_sparse"] / bec["alpha_ideal"], "be_inflation_pauli_lcu": bec["alpha_pauli"] / bec["alpha_ideal"], "pauli_density": bec["pauli_terms"] / bec["pauli_terms_possible"]}
            nc = max(n_ for n_ in QCFG["n_clock_list"] if int(math.log2(Ap.shape[0])) + n_ + 2 <= QCFG["max_qubits"])
            hs = HHLSystem(Ap, nc, rng_for("q5h", inst_id, fi), spectral="exact", c_mode="kappa_known"); ho = hs.run(rp)
            Q5A.append({**base, "solver": f"hhl_nc{nc}", "poly": "", "eps_target": np.nan, "kappa_target": hs.kappa, "n_qubits": ho["n_qubits"], "oracle_queries": 2 * (2 ** nc - 1), "l1_norm": np.nan,
                        "p_success": ho["p_postselect_exact"], "rel_err_ideal": float(np.linalg.norm(d * ho["x_ideal"][:N0] - u_true) / nu), "t_sim": ho["t_circuit_sim"], "status": "ok"})
            for eps in CKSCFG["eps_list"]:
                for poly in ["cks", "optimal"]:
                    cs = CKSSystem(Ap, eps, rng_for("q5c", inst_id, fi, eps), max_degree=CKSCFG["max_degree"], poly=poly)
                    row = {**base, "solver": "lcu_walk", "poly": cs.poly, "eps_target": eps, "kappa_target": cs.kappa, "n_qubits": cs.n_qubits, "oracle_queries": cs.degree, "l1_norm": cs.l1}
                    if cs.over_budget: Q5A.append({**row, "p_success": np.nan, "rel_err_ideal": np.nan, "t_sim": np.nan, "status": "degree_cap (simulation budget)"}); continue
                    co = cs.run(rp); Q5A.append({**row, "p_success": co["p_postselect_exact"], "rel_err_ideal": float(np.linalg.norm(d * co["x_ideal"][:N0] - u_true) / nu), "t_sim": co["t_circuit_sim"], "status": "ok"})
            # Q5d: the same captured system on the null-space SPD target: kappa, dimension, qubits, degree, forward error (ideal readout)
            try:
                inst_ = INSTANCES[inst_id]; W_ = {"di": matrix(np.asarray(caps[fi]["di"], dtype=float).reshape(-1, 1)), "d": matrix(1.0 / np.asarray(caps[fi]["di"], dtype=float).reshape(-1, 1)), "v": [], "beta": [], "r": [], "rti": []}
                asm_ = KKTAssembler(inst_); eng_ = KKTSolveEngine({"name": "q5d", "kind": "exact", "criterion": "combined", "eta": 0.1, "tol": 1e-9, "tol_rule": "fixed"}, K, asm_, W_)
                oss_ = OSSKind({"name": "q5d_oss", "max_n": 10 ** 9}, K, asm_, W_, eng_); M_ = oss_.matrix(); kap_red = float(np.linalg.cond(M_)); lam_mx = float(np.linalg.eigvalsh(M_).max())
                Mp_, _, N0r = pad_to_pow2(M_, np.zeros(M_.shape[0]), lam_mx); xp_, b_ = oss_.reduced_rhs(r); rq = np.zeros(Mp_.shape[0]); rq[:N0r] = b_
                kap_aug_s = float(np.linalg.cond(Ks)); drow = {**base, "dim_reduced": M_.shape[0], "N_padded_reduced": Mp_.shape[0], "kappa_augmented_scaled": kap_aug_s, "kappa_reduced_jacobi": kap_red, "sqrt_kappa_reduced": math.sqrt(kap_red)}
                for eps in CKSCFG["eps_list"]:
                    drow[f"degree_augmented_eps{eps:.0e}"] = polynomial_degree(kap_aug_s * 1.05, eps, "optimal"); drow[f"degree_oss_eps{eps:.0e}"] = polynomial_degree_spd(kap_red * 1.05, eps)
                    ss = SPDPolySystem(Mp_, eps, rng_for("q5d", inst_id, fi, eps), max_degree=CKSCFG["max_degree"])
                    row = {**base, "solver": "lcu_oss", "poly": ss.poly, "eps_target": eps, "kappa_target": ss.kappa, "n_qubits": ss.n_qubits, "oracle_queries": ss.degree, "l1_norm": ss.l1, "target": "null_space_spd", "dim_target": M_.shape[0]}
                    if ss.over_budget: Q5A.append({**row, "p_success": np.nan, "rel_err_ideal": np.nan, "t_sim": np.nan, "status": "degree_cap (simulation budget)"}); continue
                    so = ss.run(rq); v_ = so["x_ideal"][:N0r]; u_ = oss_.full(v_, r, xp_)
                    Q5A.append({**row, "p_success": so["p_postselect_exact"], "rel_err_ideal": float(np.linalg.norm(u_ - u_true) / nu), "t_sim": so["t_circuit_sim"], "status": "ok"})
                    drow[f"p_success_oss_eps{eps:.0e}"] = so["p_postselect_exact"]; drow[f"fwd_err_oss_eps{eps:.0e}"] = float(np.linalg.norm(u_ - u_true) / nu)
                drow["n_qubits_oss_eps0.3"] = SPDPolySystem(Mp_, 0.3, rng_for("q5d", inst_id, fi, "q"), max_degree=10 ** 9).n_qubits
                nq_aug = int(math.log2(Ap.shape[0])) + 3 + max(1, int(math.ceil(math.log2((polynomial_degree(kap_aug_s * 1.05, 0.3, "optimal") + 1) // 2 + 1)))); drow["n_qubits_augmented_eps0.3"] = nq_aug
                Q5D.append(drow)
            except Exception as e_:
                REC.failure(f"{inst_id}__q5d_{fi}", "q5d_oss", repr(e_)[:300], instance_id=inst_id); print("  Q5d failed:", inst_id, fi, repr(e_)[:160])
            # Q5b: cost of refinement to a 1e-10 residual against circuit precision and samples per round (emulated AA readout)
            if n_ir >= CKSCFG["ir_max_systems"]: continue
            n_rows_before = len(Q5B)
            for poly, eps in [("cks", 1e-6)] + [("optimal", e_) for e_ in CKSCFG["ir_eps_list"]]:
                cs = CKSSystem(Ap, eps, rng_for("q5b", inst_id, fi, eps, poly), max_degree=CKSCFG["max_degree"], poly=poly)
                if cs.over_budget: continue
                aa = 1.0 / math.sqrt(max(cs.run(rp)["p_postselect_exact"], 1e-300))
                for spr in CKSCFG["ir_samples_per_round"]:
                    u = np.zeros_like(r); hist = []
                    for rd in range(CKSCFG["ir_rounds"]):
                        res = r - K @ u; rp2 = np.zeros(Ap.shape[0]); rp2[:N0] = d * res; xa = cs.run(rp2, eff_shots=max(spr // 2, 1))["x_aa"][:N0]
                        if not np.all(np.isfinite(xa)): break
                        u = u + d * xa; hist.append(float(np.linalg.norm(K @ u - r) / np.linalg.norm(r)))
                        if hist[-1] < 1e-10 or hist[-1] > 1e6: break
                    if not hist: continue
                    Q5B.append({**base, "poly": cs.poly, "eps_circuit": eps, "kappa_target": cs.kappa, "walk_steps": cs.degree, "p_success": 1.0 / aa ** 2, "samples_per_round": spr, "rounds": len(hist), "resid_final": hist[-1],
                                "reached_1e-10": bool(hist[-1] < 1e-10), "contraction_per_round": float((hist[-1] / hist[0]) ** (1.0 / max(len(hist) - 1, 1))) if len(hist) > 1 else np.nan,
                                "total_samples": spr * len(hist), "block_encoding_queries_total_with_AA": float(len(hist) * spr * aa * cs.degree)})
            n_ir += int(len(Q5B) > n_rows_before)                      # a system counts only if at least one configuration was within the simulation budget
    Q5A, Q5B, Q5D = pd.DataFrame(Q5A), pd.DataFrame(Q5B), pd.DataFrame(Q5D)
    Q5A.to_csv(os.path.join(RESULTS_DIR, "qls_comparison_hhl_vs_cks.csv"), index=False); Q5B.to_csv(os.path.join(RESULTS_DIR, "cks_refinement.csv"), index=False); Q5D.to_csv(os.path.join(RESULTS_DIR, "oss_target_comparison.csv"), index=False)
    if len(Q5D):
        print("Q5d - augmented (Ruiz-scaled, indefinite) vs null-space (Jacobi-scaled, SPD) target on the same captured KKT systems:")
        cols_ = ["instance_id", "factor_index", "kkt_dim", "dim_reduced", "kappa_augmented_scaled", "kappa_reduced_jacobi"] + [c_ for c_ in Q5D.columns if c_.startswith("degree_")] + ["n_qubits_augmented_eps0.3", "n_qubits_oss_eps0.3"]
        print(Q5D[cols_].to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    if len(Q5A):
        a_ = Q5A.assign(solver=lambda x: np.where(x.solver == "lcu_walk", "LCU " + x.poly.map(poly_tag) + " eps=" + x.eps_target.map(lambda v: f"{v:.0e}"), x.solver))
        print("Q5a - forward error (ideal readout) on captured KKT systems:")
        print(a_.groupby("solver").agg(systems_run=("rel_err_ideal", "count"), err_median=("rel_err_ideal", "median"), err_max=("rel_err_ideal", "max"), queries_median=("oracle_queries", "median"), l1_median=("l1_norm", "median"),
              p_success_median=("p_success", "median")).to_string(float_format=lambda v: f"{v:.2e}"))
    if len(Q5B):
        print("\nQ5b - cost of refining to a 1e-10 residual (median over systems; queries include ideal amplitude amplification):")
        print(Q5B.groupby(["poly", "eps_circuit", "samples_per_round"]).agg(systems=("rounds", "size"), reached=("reached_1e-10", "mean"), rounds=("rounds", "median"), walk_steps=("walk_steps", "median"),
              total_samples=("total_samples", "median"), queries=("block_encoding_queries_total_with_AA", "median")).to_string(float_format=lambda v: f"{v:.3g}"))
    REC.status("Q5ab_cks_captured", "done", f"{len(Q5A)} solver x system rows, {len(Q5B)} refinement rows in {time.perf_counter() - t5:.0f}s")

    # Q5c: inside coneqp, no classical fallback
    icfg = QCFG["integrated"]; t5 = time.perf_counter(); n5 = 0
    common = {"fallback": False, "max_refine": CKSCFG["integrated_max_refine"], "criterion": icfg["accept_criterion"], "eta": icfg["accept_eta"], "tol_rule": "fixed", "tol": CKSCFG.get("integrated_be_tol", icfg["accept_be_tol"]), "scaling": "ruiz"}
    jobs = [(i_, {"name": rs["name"], "kind": "cks", "eps": rs["eps"], "poly": rs.get("poly"), "recon": rs["recon"], "eff_shots": rs.get("eff_shots", 0), "maxiters": icfg["maxiters"], **common})
            for i_ in list(icfg["instances"]) + [x for x in CKSCFG["integrated_extra_instances"] if x in INSTANCES] for rs in CKSCFG["integrated_runs"]]
    jobs += [(i_, {"name": rs["name"], "kind": "hhl", "n_clock": rs["n_clock"], "recon": rs["recon"], "eff_shots": rs.get("eff_shots", 0), "shots": 0, "reciprocal": "kappa_known", "reg": 0.0, "maxiters": CKSCFG.get("hhl_fair_maxiters", 20), **common})
             for i_ in list(icfg["instances"]) + [x for x in CKSCFG.get("hhl_fair_extra_instances", []) if x in INSTANCES] for rs in CKSCFG.get("hhl_fair_runs", [])
             if int(math.ceil(math.log2(INSTANCES[i_]["kkt_dim"]))) + rs["n_clock"] + 2 <= QCFG["max_qubits"] and (i_ in icfg["instances"] or rs["name"] in CKSCFG.get("hhl_fair_extra_runs", [rs["name"]]))]
    # the same instances with the null-space polynomial solver
    jobs += [(i_, {"name": rs["name"], "kind": "lcu_oss", "eps": rs["eps"], "recon": rs["recon"], "eff_shots": rs.get("eff_shots", 0), "maxiters": icfg["maxiters"], **common})
             for i_ in list(icfg["instances"]) + [x for x in CKSCFG["integrated_extra_instances"] if x in INSTANCES] for rs in CKSCFG.get("oss_runs", [])]
    for inst_id, spec in jobs:
        try:
            rec, sol, st = run_instance(INSTANCES[inst_id], spec, repeat=0, compute_cond=True, experiment="Q5_integrated_cks"); n5 += 1
            print(f"  {inst_id} {spec['name']}: status={rec['status']} iters={rec['iterations']} accepted={rec['accepted']} solves={rec['n_solve_calls']} refinement_rounds={rec['n_refine_total']} "
                  f"unaccepted_used={rec['n_solves_unaccepted_used']} fallback={rec['n_fallback_total']} strict={rec['strict_method_success']} obj_rel_err={rec.get('obj_rel_err', float('nan')):.2e} wall={rec['wall_method']:.1f}s")
        except Exception as e:
            REC.failure(f"{inst_id}__{spec['name']}", "integrated_cks", repr(e)[:300], method=spec["name"], instance_id=inst_id); print("  integrated run failed:", repr(e)[:200])
    REC.status("Q5c_integrated_cks", "done", f"{n5} runs in {time.perf_counter() - t5:.0f}s"); REC.flush(min_interval=0)

    # Q5e: finite-shot runs over sampling seeds x accepted samples per refinement round
    sw = CKSCFG.get("shot_sweep"); t5e = time.perf_counter(); n5e = 0
    if sw:
        sw_insts = (list(icfg["instances"]) + [x for x in CKSCFG["integrated_extra_instances"] if x in INSTANCES])[:sw["max_instances"]]
        for inst_id in sw_insts:
            for spr in sw["samples_per_round"]:
                for sd in sw["seeds"]:
                    spec = {"name": f"lcu_opt_eps{sw['eps']:g}_aa{spr}_seed{sd}_ir_nofb", "kind": "cks", "eps": sw["eps"], "poly": "optimal", "recon": "aa_shots", "eff_shots": max(spr // 2, 1), "maxiters": icfg["maxiters"],
                            "sweep_samples": spr, "sweep_seed": sd, **common}
                    try:
                        rec, sol, st = run_instance(INSTANCES[inst_id], spec, repeat=0, compute_cond=False, experiment="Q5e_shot_sweep"); n5e += 1
                    except Exception as e:
                        REC.failure(f"{inst_id}__{spec['name']}", "shot_sweep", repr(e)[:300], method=spec["name"], instance_id=inst_id); print("  shot-sweep run failed:", repr(e)[:200])
        REC.flush(min_interval=0); d5e = REC.df("benchmark_results"); d5e = d5e[d5e.experiment == "Q5e_shot_sweep"]
        if len(d5e):
            print("Q5e - strict success rate / median IPM iterations / median refinement rounds over instances x seeds, by accepted samples per round:")
            print(d5e.groupby("sweep_samples").agg(runs=("run_id", "size"), strict=("strict_method_success", "mean"), accepted=("accepted", "mean"), ipm_its=("iterations", "median"), refine=("n_refine_total", "median"),
                                                  rejected_used=("n_solves_unaccepted_used", "sum"), wall=("wall_method", "median")).round(3).to_string())
            print("  per-instance spread across seeds (strict successes / seeds):"); print(d5e.pivot_table(index="instance_id", columns="sweep_samples", values="strict_method_success", aggfunc="sum").to_string())
    REC.status("Q5e_shot_sweep", "done" if n5e else "skipped", f"{n5e} runs in {time.perf_counter() - t5e:.0f}s")

## 21. SDPLIB with `conelp` (optional, S1)

If the SDPLIB `*.dat-s` files are found (for example as a Kaggle dataset) or can be downloaded, they are converted to CVXOPT's `conelp` form, $\min c^\top x$ s.t. $Gx + s = h$, $s \in \mathcal K$. SDPA's $X = \sum_i x_i F_i - F_0 \succeq 0$ becomes $G_k = [-\mathrm{vec}\, F_i^{(k)}]$ and $h_k = -\mathrm{vec}\, F_0^{(k)}$; diagonal blocks become rows of the linear cone. Objective values are compared with the values in the SDPLIB README. The callback uses the generic cone path of the assembler (`misc.scale` and packed storage), so the KKT systems of real semidefinite programs go through the same solvers and tests. The `hinf*` problems are degenerate.

In [ ]:
# S1: SDPLIB problems with conelp (optional).
SDPLIB_OPT = {"truss1": -8.999996, "truss2": -123.3804, "truss3": -9.109996, "truss4": -9.009996, "truss5": -132.6357,
              "hinf1": 2.0326, "hinf2": 10.967, "hinf3": 56.9, "hinf4": 274.764, "hinf5": 363.0, "hinf6": 449.0, "hinf7": 391.0, "hinf8": 116.0,
              "hinf9": 236.25, "hinf10": 109.0, "hinf11": 65.9, "hinf12": 0.2, "hinf13": 46.0, "hinf14": 13.0, "hinf15": 25.0,
              "control1": 17.78463, "control2": 8.3, "control3": 13.63327, "control4": 19.79423, "qap5": -436.0, "qap6": -381.44,
              "theta1": 23.0, "theta2": 32.87917, "mcp100": 226.1574, "arch0": 0.566517, "gpp100": -44.9435}

def find_sdplib_file(name):
    fname = f"{name}.dat-s"
    for d in CONFIG["sdplib"]["search_dirs"]:
        if not os.path.isdir(d): continue
        p = os.path.join(d, fname)
        if os.path.isfile(p): return p
        for root, dirs, files in os.walk(d):
            if fname in files: return os.path.join(root, fname)
            if root[len(d):].count(os.sep) >= 3: dirs[:] = []
    return None

def download_sdplib(name):
    import urllib.request
    cache = os.path.join(CONFIG["output_root"], "sdplib_cache"); os.makedirs(cache, exist_ok=True)
    p = os.path.join(cache, f"{name}.dat-s")
    if os.path.isfile(p): return p
    try:
        urllib.request.urlretrieve(CONFIG["sdplib"]["download_url"].format(name=name), p)
        return p if os.path.getsize(p) > 0 else None
    except Exception as e:
        return None

def parse_sdpa_sparse(path):
    lines = [l for l in open(path) if l.strip() and l.lstrip()[0] not in '"*']
    clean = lambda s: re.sub(r"[{}(),]", " ", s)
    m = int(clean(lines[0]).split()[0]); nblocks = int(clean(lines[1]).split()[0])
    sizes = [int(float(t)) for t in clean(lines[2]).split()][:nblocks]
    c = np.array([float(t) for t in clean(lines[3]).split()])[:m]
    entries = []
    for l in lines[4:]:
        t = l.split()
        if len(t) < 5: continue
        entries.append((int(t[0]), int(t[1]), int(t[2]), int(t[3]), float(t[4])))
    return m, sizes, c, entries

def sdpa_to_instance(name, path):
    m, sizes, c, entries = parse_sdpa_sparse(path)
    lp_blocks = [(bi, -s) for bi, s in enumerate(sizes) if s < 0]
    sdp_blocks = [(bi, s) for bi, s in enumerate(sizes) if s > 0]
    off_l = {}; n_l = 0
    for bi, s in lp_blocks: off_l[bi] = n_l; n_l += s
    off_s = {}; pos = n_l
    for bi, s in sdp_blocks: off_s[bi] = pos; pos += s * s
    cdim = pos
    G = np.zeros((cdim, m)); h = np.zeros(cdim)
    for matno, blk, i, j, v in entries:
        bi = blk - 1
        if bi in off_l:
            if i != j: continue
            row = off_l[bi] + (i - 1)
            if matno == 0: h[row] = -v
            else: G[row, matno - 1] = -v
        else:
            s = sizes[bi]; base = off_s[bi]
            rows = {base + (j - 1) * s + (i - 1), base + (i - 1) * s + (j - 1)}
            for row in rows:
                if matno == 0: h[row] = -v
                else: G[row, matno - 1] = -v
    dims = {"l": n_l, "q": [], "s": [s for _, s in sdp_blocks]}
    kkt_dim = m + n_l + sum(s * (s + 1) // 2 for s in dims["s"])
    opt = SDPLIB_OPT.get(name)
    return dict(id=f"sdplib_{name}", family="sdplib", kind="conelp", n=m, m=cdim, p=0, kkt_dim=kkt_dim, kappa_P_target=np.nan, kappa_P_measured=np.nan,
                density=float(np.count_nonzero(G) / G.size), seed=0, sparse=False, P=None, q=c, G=G, h=h, A=None, b=None, dims=dims,
                f_star=opt if opt is not None else np.nan, known_solution=opt is not None, sdplib_name=name, block_sizes=sizes, path=path)

def recompute_metrics_sdp(inst, sol):
    x, s, z = to_np(sol["x"]), to_np(sol["s"]), to_np(sol["z"])
    c, G, h, dims = inst["q"], inst["G"], inst["h"], inst["dims"]
    pobj = float(c @ x); dobj = float(-h @ z)
    pres = np.linalg.norm(G @ x + s - h) / max(1.0, np.linalg.norm(h))
    dres = np.linalg.norm(G.T @ z + c) / max(1.0, np.linalg.norm(c))
    def min_eig_cone(v):
        worst = 0.0; l = dims["l"]
        if l: worst = min(worst, float(v[:l].min()))
        off = l
        for k in dims["s"]:
            M = v[off:off + k * k].reshape((k, k), order="F"); M = 0.5 * (M + M.T)
            worst = min(worst, float(np.linalg.eigvalsh(M).min())); off += k * k
        return worst
    cone_viol = max(0.0, -min_eig_cone(s), -min_eig_cone(z))
    constr_viol = max(0.0, -min_eig_cone(h - G @ x)) / max(1.0, np.linalg.norm(h))
    gap = float(s @ z)
    out = {"pobj": pobj, "dobj_valid": np.nan, "dobj_cvx": dobj, "gap_rec": gap, "relgap_valid": np.nan, "relgap_rec": gap / max(1.0, abs(pobj)),
           "pres_rec": float(pres), "dres_rec": float(dres), "cone_viol": float(cone_viol), "constraint_viol": float(constr_viol), "mu_final": gap / max(1, dims["l"] + sum(dims["s"]))}
    out["obj_rel_err"] = float(abs(pobj - inst["f_star"]) / max(1.0, abs(inst["f_star"]))) if inst.get("known_solution") else np.nan
    out["x_rel_err"] = np.nan
    return out
METRIC_FUNCS["conelp"] = recompute_metrics_sdp

SDPLIB_INSTANCES = {}
if not CONFIG["sdplib"]["enabled"]:
    REC.status("S1_sdplib", "skipped", "disabled in CONFIG")
else:
    missing = []
    for name in CONFIG["sdplib"]["instances"]:
        path = find_sdplib_file(name)
        src = "local"
        if path is None and CONFIG["sdplib"]["download_if_missing"]:
            path = download_sdplib(name); src = "download"
        if path is None:
            missing.append(name); continue
        try:
            inst = sdpa_to_instance(name, path); inst["source"] = src
            SDPLIB_INSTANCES[inst["id"]] = inst; INSTANCES[inst["id"]] = inst
        except Exception as e:
            REC.failure(f"sdplib_{name}", "parse", repr(e)[:300])
    if not SDPLIB_INSTANCES:
        REC.status("S1_sdplib", "skipped", f"no SDPLIB files found in {CONFIG['sdplib']['search_dirs']} and download failed (Internet off?)" if missing else "no instances configured")
    else:
        print("SDPLIB instances loaded:", ", ".join(f"{i['sdplib_name']}(m={i['n']}, blocks={i['block_sizes'] if len(i['block_sizes'])<8 else len(i['block_sizes'])}, kkt_dim={i['kkt_dim']}, {i['source']})" for i in SDPLIB_INSTANCES.values()))
        if missing: print("  not found / not downloadable:", missing)
        t_s = time.perf_counter(); n_runs = 0
        sd_specs = {k: SPECS[k] for k in ["cvxopt_default", "custom_exact", "minres_relres_1e-06", "minres_rrfloor_1e-06", "minres_be_1e-08", "minres_be_1e-12", "minres_block_0.1", "minres_comb_0.1_1e-10"] if k in SPECS}
        max_inexact_dim = CONFIG["sdplib"].get("max_kkt_dim_inexact", 300)
        for inst in SDPLIB_INSTANCES.values():
            for name, spec in sd_specs.items():
                if spec["kind"] != "baseline" and inst["kkt_dim"] > CONFIG["sdplib"]["max_kkt_dim_custom"]:
                    REC.status(f"S1_{inst['sdplib_name']}_{name}", "skipped", f"kkt_dim {inst['kkt_dim']} > max_kkt_dim_custom"); continue
                if spec["kind"] == "minres" and inst["kkt_dim"] > max_inexact_dim:
                    REC.status(f"S1_{inst['sdplib_name']}_{name}", "skipped", f"kkt_dim {inst['kkt_dim']} > max_kkt_dim_inexact={max_inexact_dim}"); continue
                try:
                    rec, sol, st = run_instance(inst, spec, repeat=0, compute_cond=(name == "custom_exact"),
                                                capture=(name == "custom_exact" and inst["sdplib_name"] in CONFIG["sdplib"]["quantum_instances"]),
                                                cond_scaled=(name == "custom_exact"), experiment="S1_sdplib")
                    n_runs += 1
                except Exception as e:
                    REC.failure(f"{inst['id']}__{name}", "sdplib_run", repr(e)[:300], method=name, instance_id=inst["id"])
        REC.status("S1_sdplib", "done", f"{n_runs} runs in {time.perf_counter() - t_s:.0f}s")
        dfs = REC.df("benchmark_results"); dfs = dfs[dfs.experiment == "S1_sdplib"]
        print(dfs[["instance_id", "method", "status", "iterations", "pobj", "obj_rel_err", "pres_rec", "dres_rec", "cone_viol", "kappa_max", "n_fallback_total", "wall_method", "accepted"]].round(6).to_string(index=False))
        # quantum prototype on captured SDP KKT systems (tiny ones only)
        qn = 0
        for inst in SDPLIB_INSTANCES.values():
            if inst["id"] in CAPTURED and QCFG["enabled"]:
                caps = CAPTURED[inst["id"]]
                for fi in select_factor_indices(caps):
                    for n_clock in QCFG["n_clock_list"]:
                        for sc, cm in [("none", "resolution"), ("ruiz", "kappa_known")]:
                            evaluate_captured_system(inst["id"], caps[fi], n_clock, sc, cm, 0.0, QCFG["shots_list"], QCFG["eff_shots_list"], QCFG["sampling_repeats"], QCFG["ir_rounds"], inst["n"]); qn += 1
        if qn:
            REC.status("Q1_captured_sdplib", "done", f"{qn} configurations on captured SDPLIB KKT systems")
            run_reg_experiment({k: v for k, v in CAPTURED.items() if k.startswith("sdplib")}, "sdplib")
            qq = REC.df("quantum_results"); qq = qq[(qq.instance_id.str.startswith("sdplib")) & (qq.recon == "ideal_statevector")]
            qq = qq[qq.ir_round == 0]
            print(qq[["instance_id", "factor_index", "kappa_kkt", "n_clock", "scaling", "reciprocal_mode", "status", "reason", "n_qubits", "p_postselect_exact", "rel_err_original"]].to_string(index=False))
    REC.flush(min_interval=0)

## 22. Figures

Diagnostic figures of this run, written to `results/plots/`. Each method keeps its colour in every figure; the fixed-tolerance MINRES variants share one sequential ramp ordered by tolerance, and the unmodified CVXOPT solver is the black reference. A figure whose data is missing is skipped with a printed reason. The figures of the paper are made from the exported tables by `scripts/make_figures.py`.

In [ ]:
# Diagnostic figures, written to results/plots/. A figure whose data is missing in this run is skipped with a printed reason.
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "axes.edgecolor": "#c3c2b7",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e", "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 7,
                     "legend.frameon": False, "lines.linewidth": 2, "lines.markersize": 5, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
_CAT = {"cvxopt_default": "#0b0b0b", "custom_exact": "#eb6834", "minres_relres_1e-06_noscale": "#898781", "minres_relres_1e-06": "#52514e",
        "minres_be_1e-10_noscale": "#e87ba4", "minres_block_0.1_noscale": "#eda100", "minres_adaptive_nt_be": "#4a3aa7", "cg_reduced_be_1e-10": "#008300",
        "custom_exact_reduced": "#c98500", "minres_fused_comb_cpu": "#5598e7", "minres_fused_comb_floor8_cpu": "#1baf7a", "minres_fused_comb_gpu": "#eb6834", "minres_fused_comb_auto": "#9085e9", "gmres_stalelu_comb": "#008300", "minres_base_comb_cpu": "#2a78d6", "minres_base_comb_ws_cpu": "#0d366b", "minres_base_comb_ws_switch_cpu": "#1baf7a",
        "minres_base_comb_gpu": "#e87ba4", "minres_base_comb_ws_gpu": "#d03b3b", "lsqr_sqrt_comb": "#4a3aa7", "minres_rrfloor_1e-06": "#d55181", "minres_rrfloor_1e-06_noscale": "#f2b3c9", "minres_comb_0.1_1e-10": "#c98500",
        "minres_fused_comb_switch_cpu": "#6a5bd0", "oss_cg_comb": "#0a6b4a", "minres_fused_comb_gpu_mm": "#eb6834"}
_BLUES = ["#9ec5f4", "#5598e7", "#2a78d6", "#0d366b"]; _GREENS = ["#8fd3b6", "#1baf7a", "#0a6b4a"]; _VIOLETS = ["#b3a9ec", "#9085e9", "#6a5bd0", "#4a3aa7", "#2f2470", "#1c1545"]
def method_color(name):
    if name in _CAT: return _CAT[name]
    if name.startswith("minres_be_"):
        tols = sorted({float(n.split("_")[-1]) for n in SPECS if n.startswith("minres_be_") and not n.endswith("noscale")} | {float(name.split("_")[-1])}, reverse=True)
        return _BLUES[min(len(_BLUES) - 1, tols.index(float(name.split("_")[-1])) * (len(_BLUES) - 1) // max(1, len(tols) - 1))]
    if name.startswith("minres_block_"):
        etas = sorted({float(n.split("_")[-1]) for n in SPECS if n.startswith("minres_block_") and not n.endswith("noscale")} | {float(name.split("_")[-1])}, reverse=True)
        return _GREENS[min(len(_GREENS) - 1, etas.index(float(name.split("_")[-1])) * (len(_GREENS) - 1) // max(1, len(etas) - 1))]
    if name.startswith("hhl"):
        names = [r["name"] for r in CONFIG["quantum"]["integrated"]["runs"]]
        return _VIOLETS[names.index(name) % len(_VIOLETS)] if name in names else "#4a3aa7"
    if name.startswith("exact_noise_iso"): return "#e34948"
    if name.startswith("exact_noise_block"): return "#8b1a1a"
    return "#898781"
def method_style(name): return {"linestyle": "--" if name == "cvxopt_default" else "-", "marker": "D" if name.startswith("hhl") else "o"}
def savefig(fig, name):
    path = os.path.join(PLOTS_DIR, name); fig.tight_layout(); fig.savefig(path); plt.close(fig); print("  saved", os.path.relpath(path, CONFIG["output_root"]))
def with_data(fn):
    def wrapper(*a, **k):
        try: fn(*a, **k)
        except Exception as e: print(f"  plot {fn.__name__} skipped: {type(e).__name__}: {e}")
    return wrapper
def order_methods(names):
    pref = ["cvxopt_default", "custom_exact", "custom_exact_reduced", "gmres_stalelu_comb", "minres_fused_comb_cpu", "minres_fused_comb_gpu", "minres_fused_comb_auto", "minres_base_comb_cpu", "minres_base_comb_ws_cpu", "minres_base_comb_ws_switch_cpu", "minres_base_comb_gpu", "minres_base_comb_ws_gpu", "lsqr_sqrt_comb", "minres_relres_1e-06_noscale", "minres_relres_1e-06", "minres_rrfloor_1e-06_noscale", "minres_rrfloor_1e-06"] + \
           sorted([n for n in names if n.startswith("minres_be_") and not n.endswith("noscale")], key=lambda m: -float(m.split("_")[-1])) + ["minres_be_1e-10_noscale"] + \
           sorted([n for n in names if n.startswith("minres_block_") and not n.endswith("noscale")], key=lambda m: -float(m.split("_")[-1])) + ["minres_block_0.1_noscale", "minres_comb_0.1_1e-10", "minres_adaptive_nt_be", "cg_reduced_be_1e-10"]
    return [m for m in pref if m in set(names)] + [m for m in names if m not in pref]

DF = REC.df("benchmark_results"); IT = REC.df("iteration_logs"); LS = REC.df("linear_solve_logs"); QD = REC.df("quantum_results")

@with_data
def plot_runtime_vs_size():
    d = DF[(DF.experiment == "E3_scaling")]
    if d.empty: raise ValueError("no E3 data")
    fams = [f for f in ["sparse_banded", "sparse_scale"] if f in set(d.family)]
    fig, axes = plt.subplots(1, len(fams) + 1, figsize=(5.5 * (len(fams) + 1), 4.3))
    for ax, fam in zip(axes, fams):
        g0 = d[d.family == fam]
        for meth, g in g0.groupby("method"):
            s = g.groupby("kkt_dim")["wall_method"].agg(["median", "min", "max"]).reset_index()
            ax.plot(s.kkt_dim, s["median"], color=method_color(meth), label=meth, **method_style(meth)); ax.fill_between(s.kkt_dim, s["min"], s["max"], color=method_color(meth), alpha=0.12, linewidth=0)
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("KKT dimension n+m+p"); ax.set_ylabel("wall time per coneqp call [s] (median, min–max)")
        ax.set_title(f"Runtime vs. size, family {fam}"); ax.legend()
    ax = axes[-1]; fam = fams[0]; big = d[(d.family == fam) & (d.kkt_dim == d[d.family == fam].kkt_dim.max())]
    comps = ["t_assemble", "t_factor", "t_solve", "t_check", "t_verify", "t_refine", "t_fallback"]; rows = []
    for meth, g in big.groupby("method"):
        r = g[comps].median(); r["cvxopt_iteration"] = max(0.0, g["wall_method"].median() - r.sum()); rows.append(r.rename(meth))
    comp_df = pd.DataFrame(rows).reindex(order_methods(list(pd.DataFrame(rows).index))); bottom = np.zeros(len(comp_df))
    cols = ["#9ec5f4", "#2a78d6", "#0d366b", "#e87ba4", "#1baf7a", "#eda100", "#e34948", "#898781"]
    for c, colr in zip(comps + ["cvxopt_iteration"], cols):
        ax.bar(comp_df.index, comp_df[c], bottom=bottom, color=colr, label=c, width=0.6, edgecolor="#fcfcfb", linewidth=1); bottom += comp_df[c].values
    ax.set_ylabel("seconds (median)"); ax.set_title(f"Time components, {fam}, KKT dim {int(big.kkt_dim.max())}"); ax.tick_params(axis="x", rotation=60); ax.legend(ncol=2)
    savefig(fig, "01_runtime_vs_size.png")

@with_data
def plot_accuracy_vs_runtime():
    d = DF[DF.experiment == "E2_main"].copy()
    if d.empty: raise ValueError("no E2 data")
    d["final_err"] = d[["pres_rec", "dres_rec", "obj_rel_err"]].max(axis=1).clip(lower=1e-16)
    fig, ax = plt.subplots(figsize=(8, 5))
    for meth in order_methods(list(d.method.unique())):
        g = d[d.method == meth]; ax.scatter(g.wall_method, g.final_err, s=16, color=method_color(meth), label=meth, alpha=0.75, edgecolors="none")
    ax.axhline(max(CONFIG["accept"]["pres"], CONFIG["accept"]["obj_rel_err"]), color="#e34948", linewidth=1, linestyle=":", label="acceptance threshold")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("wall time per coneqp call [s]"); ax.set_ylabel("max(pres, dres, objective rel. error)"); ax.set_title("Accuracy vs. runtime (dense families, every run)"); ax.legend(ncol=2, fontsize=6)
    savefig(fig, "02_accuracy_vs_runtime.png")

@with_data
def plot_convergence():
    d = DF[(DF.experiment == "E2_main") & (DF.family == "dense_kappa") & (DF.repeat == 0)]
    if d.empty: raise ValueError("no E2 data")
    inst_id = d[d.kappa_P_target == d.kappa_P_target.max()].sort_values("n").instance_id.iloc[-1]
    it = IT[(IT.instance_id == inst_id) & (IT.repeat == 0)]
    keep = ["cvxopt_default", "custom_exact", "minres_relres_1e-06_noscale", "minres_relres_1e-06", "minres_rrfloor_1e-06", "minres_be_1e-08", "minres_be_1e-12", "minres_block_0.1", "minres_comb_0.1_1e-10"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for meth in keep:
        g = it[it.method == meth]
        if g.empty: continue
        axes[0].plot(g["iter"], g["gap"].clip(lower=1e-16), color=method_color(meth), label=meth, **method_style(meth))
        axes[1].plot(g["iter"], np.maximum(g["pres"], g["dres"]).clip(lower=1e-16), color=method_color(meth), label=meth, **method_style(meth))
    axes[0].set_yscale("log"); axes[0].set_xlabel("IPM iteration"); axes[0].set_ylabel("duality gap sᵀz (CVXOPT report)"); axes[0].set_title(f"Convergence by iteration, {inst_id}")
    axes[1].set_yscale("log"); axes[1].set_xlabel("IPM iteration"); axes[1].set_ylabel("max(pres, dres)"); axes[1].set_title("Residuals by iteration"); axes[1].legend(fontsize=6)
    savefig(fig, "03_convergence_by_iteration.png")

@with_data
def plot_conditioning():
    it = IT[(IT.method == "custom_exact") & (IT.repeat == 0) & IT.kappa.notna()].merge(DF[DF.method == "custom_exact"][["run_id", "kappa_P_target", "family", "n"]], on="run_id", how="left")
    it = it[it.family.isin(["dense_kappa", "sanity"])]
    if it.empty: raise ValueError("no conditioning data")
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2)); kps = sorted(it.kappa_P_target.dropna().unique())
    ramp = [plt.cm.Blues(0.35 + 0.6 * i / max(1, len(kps) - 1)) for i in range(len(kps))]
    for kp, colr in zip(kps, ramp):
        med = it[it.kappa_P_target == kp].groupby("iter")[["kappa", "kappa_scaled", "nt_proxy"]].median()
        axes[0].plot(med.index, med.kappa, color=colr, label=f"κ(P)={kp:.0e}: κ(K̃)")
        if med.kappa_scaled.notna().any(): axes[0].plot(med.index, med.kappa_scaled, color=colr, linestyle="--", label=f"κ(P)={kp:.0e}: κ(DK̃D) Ruiz")
        axes[1].plot(med.index, med.nt_proxy, color=colr, label=f"κ(P)={kp:.0e}")
    axes[0].set_yscale("log"); axes[0].set_xlabel("IPM iteration"); axes[0].set_ylabel("κ₂ of the scaled KKT matrix (median)"); axes[0].set_title("Conditioning along the path"); axes[0].legend(fontsize=6)
    axes[1].set_yscale("log"); axes[1].set_xlabel("IPM iteration"); axes[1].set_ylabel("NT-scaling spread min(d)/max(d)"); axes[1].set_title("μ proxy (adaptive_nt)"); axes[1].legend(fontsize=6)
    ex = LS[(LS.method_kind == "exact") & LS.kappa.notna()]
    axes[2].scatter(ex.kappa, ex.rel_resid_final.clip(lower=1e-17), s=8, color="#898781", label="relative residual ‖K̃ũ−r̃‖/‖r̃‖", edgecolors="none")
    axes[2].scatter(ex.kappa, ex.backward_err_final.clip(lower=1e-17), s=8, color="#2a78d6", label="backward error", edgecolors="none")
    axes[2].set_xscale("log"); axes[2].set_yscale("log"); axes[2].set_xlabel("κ₂(K̃)"); axes[2].set_ylabel("residual measure of the LU solve"); axes[2].set_title("Attainability: LU residual vs. conditioning"); axes[2].legend(fontsize=6)
    savefig(fig, "04_conditioning_across_iterations.png")

@with_data
def plot_criterion_ablation():
    d = DF[(DF.experiment == "E2_main") & (~DF.method.isin(["cvxopt_default", "custom_exact"]))]
    if d.empty: raise ValueError("no E2 data")
    meths = order_methods(list(d.method.unique()))
    s = d.groupby("method").agg(pure=("pure_method_success", "mean"), acc=("accepted", "mean"), fb=("n_fallback_total", "mean"), kry=("krylov_iters_total", "median"), wall=("wall_method", "median"), it=("iterations", "median")).reindex(meths)
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.4)); x = np.arange(len(meths)); cols = [method_color(m) for m in meths]
    axes[0].bar(x - 0.2, s.acc, 0.38, color="#c3c2b7", label="accepted (fallback allowed)"); axes[0].bar(x + 0.2, s.pure, 0.38, color=cols, label="accepted with zero fallbacks")
    axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("fraction of runs"); axes[0].set_title("Acceptance (dense families)"); axes[0].legend(fontsize=6)
    axes[1].bar(x, s.fb, color=cols); axes[1].set_ylabel("mean classical fallbacks per run"); axes[1].set_title("Fallbacks")
    axes[2].bar(x, s.kry, color=cols); axes[2].set_yscale("log"); axes[2].set_ylabel("median Σ Krylov iterations per run"); axes[2].set_title("Krylov work")
    axes[3].bar(x, s.wall, color=cols); axes[3].set_yscale("log"); axes[3].set_ylabel("median wall time [s]"); axes[3].set_title("Runtime")
    for ax in axes: ax.set_xticks(x); ax.set_xticklabels(meths, rotation=70, fontsize=7)
    fig.suptitle("Acceptance-test ablation (relres = relative residual; rrfloor = relres with direct-solver floor; be = backward error; block = block forcing; comb = block and backward; noscale = no Ruiz scaling)", fontsize=8)
    savefig(fig, "05_criterion_ablation.png")

@with_data
def plot_variability():
    d = DF[(DF.experiment == "E2_main")]
    if d.empty: raise ValueError("no E2 data")
    meths = order_methods(list(d.method.unique())); fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
    data = [d[d.method == m].wall_method.values for m in meths]; bkw = dict(patch_artist=True, widths=0.6, showfliers=True, flierprops={"markersize": 3, "markeredgecolor": "#898781"})
    try: bp = axes[0].boxplot(data, tick_labels=meths, **bkw)
    except TypeError: bp = axes[0].boxplot(data, labels=meths, **bkw)
    for patch, m in zip(bp["boxes"], meths): patch.set_facecolor(method_color(m)); patch.set_alpha(0.55); patch.set_edgecolor(method_color(m))
    axes[0].set_yscale("log"); axes[0].set_ylabel("wall time per coneqp call [s]"); axes[0].set_title("Runtime variability across instances and seeds"); axes[0].tick_params(axis="x", rotation=70, labelsize=7)
    piv = d.pivot_table(index="method", columns="kappa_P_target", values="pure_method_success", aggfunc="mean").reindex(meths)
    im = axes[1].imshow(piv.values, cmap="Blues", vmin=0, vmax=1, aspect="auto"); axes[1].set_yticks(range(len(meths))); axes[1].set_yticklabels(meths, fontsize=7)
    axes[1].set_xticks(range(piv.shape[1])); axes[1].set_xticklabels([f"{c:.0e}" for c in piv.columns]); axes[1].set_xlabel("κ(P)"); axes[1].set_title("Fallback-free success rate by κ(P)"); axes[1].grid(False)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]): axes[1].text(j, i, f"{piv.values[i, j]:.2f}", ha="center", va="center", fontsize=7, color="#0b0b0b" if piv.values[i, j] < 0.6 else "#ffffff")
    fig.colorbar(im, ax=axes[1], label="fraction of runs"); savefig(fig, "06_variability_and_success.png")

@with_data
def plot_noise():
    nz = DF[DF.experiment == "E4_noise"].copy()
    if nz.empty: raise ValueError("no noise data")
    nz["delta"] = nz.method.str.split("_").str[-1].astype(float); nz["model"] = np.where(nz.method.str.contains("_iso_"), "isotropic", "block-relative")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for model, colr in [("isotropic", "#e34948"), ("block-relative", "#8b1a1a")]:
        s = nz[nz.model == model].groupby("delta").agg(acc=("accepted", "mean"), it=("iterations", "median")).reset_index()
        if s.empty: continue
        axes[0].plot(s.delta, s.acc, marker="o", color=colr, label=model); axes[1].plot(s.delta, s.it, marker="o", color=colr, label=model)
    axes[0].set_xscale("log"); axes[0].set_ylim(-0.05, 1.05); axes[0].set_xlabel("injected relative direction error δ"); axes[0].set_ylabel("fraction accepted"); axes[0].set_title("Noise-injection sensitivity (not a quantum method)"); axes[0].legend()
    axes[1].set_xscale("log"); axes[1].set_xlabel("δ"); axes[1].set_ylabel("median IPM iterations"); axes[1].set_title("Iterations under injected direction error"); axes[1].legend()
    savefig(fig, "07_noise_sensitivity.png")

@with_data
def plot_quantum_error_vs_clock():
    q = QD[(QD.experiment == "Q1_captured") & (QD.status == "ok") & (QD.recon == "ideal_statevector") & (QD.ir_round == 0)]
    if q.empty: raise ValueError("no Q1 data")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    settings = [("none", "resolution", 0.0, "#898781", "no scaling, C=Δ"), ("ruiz", "resolution", 0.0, "#5598e7", "Ruiz, C=Δ"), ("none", "kappa_known", 0.0, "#eda100", "no scaling, C=k_min·Δ"), ("ruiz", "kappa_known", 0.0, "#0d366b", "Ruiz, C=k_min·Δ")]
    for rg in sorted(set(q.regularization) - {0.0}): settings.append(("ruiz", "kappa_known", rg, "#1baf7a" if rg < 1e-3 else "#0a6b4a", f"Ruiz, κ-known, δ={rg:g} (before refinement)"))
    for sc, cm, rg, colr, lab in settings:
        g = q[(q.scaling == sc) & (q.reciprocal_mode == cm) & (q.regularization == rg)]
        if g.empty: continue
        s = g.groupby("n_clock").rel_err_original.quantile([0.25, 0.5, 0.75]).unstack()
        axes[0].plot(s.index, s[0.5].clip(lower=1e-16), marker="o", color=colr, label=lab); axes[0].fill_between(s.index, s[0.25].clip(lower=1e-16), s[0.75].clip(lower=1e-16), color=colr, alpha=0.12, linewidth=0)
    axes[0].set_yscale("log"); axes[0].set_xlabel("clock qubits n_c"); axes[0].set_ylabel("ideal-statevector rel. error, original system (median, IQR)"); axes[0].set_title("HHL intrinsic error vs. clock register and settings"); axes[0].legend(fontsize=6)
    g = q[(q.scaling == "ruiz") & (q.reciprocal_mode == "kappa_known") & (q.regularization == 0.0)]
    sc_ = axes[1].scatter(g.kappa_original, g.rel_err_original.clip(lower=1e-16), c=g.n_clock, cmap="Blues", s=22, edgecolors="#52514e", linewidths=0.4)
    axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].set_xlabel("κ₂ of the original KKT system"); axes[1].set_ylabel("ideal rel. error (Ruiz, κ known)"); axes[1].set_title("Error vs. conditioning (colour = n_c)"); fig.colorbar(sc_, ax=axes[1], label="clock qubits")
    savefig(fig, "08_quantum_error_vs_clock.png")

@with_data
def plot_quantum_readout():
    q = QD[(QD.experiment == "Q1_captured") & (QD.status == "ok") & (QD.scaling == "ruiz") & (QD.reciprocal_mode == "kappa_known") & (QD.regularization == 0.0)]
    if q.empty: raise ValueError("no Q1 data")
    nc = q.n_clock.max(); idl = q[(q.recon == "ideal_statevector") & (q.ir_round == 0) & (q.n_clock == nc)]
    res = set(map(tuple, idl[idl.rel_err_target <= 0.5][["instance_id", "factor_index"]].values))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
    for recon, colr, lab in [("aa_shots", "#4a3aa7", "AA-emulated: accepted samples per setting"), ("shots", "#eb6834", "raw shots per setting (postselection included)")]:
        g = q[(q.recon == recon) & (q.n_clock == nc) & np.array([tuple(v) in res for v in q[["instance_id", "factor_index"]].values], dtype=bool)]
        if g.empty: continue
        s = g.groupby("shots_per_setting").rel_err_target.quantile([0.25, 0.5, 0.75]).unstack()
        axes[0].plot(s.index, s[0.5], marker="o", color=colr, label=lab); axes[0].fill_between(s.index, s[0.25], s[0.75], color=colr, alpha=0.15, linewidth=0)
    axes[0].axhline(idl[[tuple(v) in res for v in idl[["instance_id", "factor_index"]].values]].rel_err_target.median() if res else idl.rel_err_target.median(), color="#0b0b0b", linestyle="--", linewidth=1, label="ideal-statevector error (median)")
    axes[0].set_xscale("log"); axes[0].set_yscale("log"); axes[0].set_xlabel("samples per measurement setting"); axes[0].set_ylabel("rel. error of reconstructed vector (median, IQR)")
    axes[0].set_title(f"Readout error vs. budget, n_c={nc}, {len(res)} resolvable systems", fontsize=9); axes[0].legend(fontsize=6)
    aa = q[q.recon == "aa_shots"]
    if not aa.empty:
        s = aa.groupby(["n_clock", "shots_per_setting"]).aa_overhead_per_sample.median().unstack()
        for ncc in s.index: axes[1].plot(s.columns, s.loc[ncc], marker="o", color=plt.cm.Blues(0.35 + 0.6 * (list(s.index).index(ncc)) / max(1, len(s.index) - 1)), label=f"n_c={ncc}")
        axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].set_xlabel("accepted samples per setting"); axes[1].set_ylabel("expected circuit repetitions per accepted sample = 1/√P (median)"); axes[1].set_title("Amplitude-amplification overhead (assumption-dependent)"); axes[1].legend(fontsize=6)
    ideal_all = QD[(QD.experiment == "Q1_captured") & (QD.status == "ok") & (QD.recon == "ideal_statevector") & (QD.ir_round == 0)]
    sc_ = axes[2].scatter(ideal_all.kappa_target, ideal_all.p_postselect_exact.clip(lower=1e-9), c=ideal_all.n_clock, cmap="Blues", s=20, edgecolors="#52514e", linewidths=0.3)
    axes[2].set_xscale("log"); axes[2].set_yscale("log"); axes[2].set_xlabel("κ of the HHL target system"); axes[2].set_ylabel("P(anc = 1) exact"); axes[2].set_title("Postselection probability (all settings; colour = n_c)"); fig.colorbar(sc_, ax=axes[2], label="clock qubits")
    savefig(fig, "09_quantum_readout_and_postselection.png")

@with_data
def plot_quantum_regularization():
    q = QD[QD.experiment.str.startswith("Q4_regularization") & (QD.status == "ok")]
    if q.empty: raise ValueError("no Q4 data")
    corrs = list(q.corrector.unique()); fig, axes = plt.subplots(1, len(corrs) + 1, figsize=(5.6 * (len(corrs) + 1), 4.3))
    cols = ["#0d366b", "#1baf7a", "#eda100", "#e34948", "#4a3aa7"]
    for ax, corr in zip(axes, corrs):
        g0 = q[q.corrector == corr]
        for i, rg in enumerate(sorted(g0.regularization.unique())):
            s = g0[g0.regularization == rg].groupby("ir_round").rel_resid_original.quantile([0.25, 0.5, 0.75]).unstack()
            ax.plot(s.index, s[0.5].clip(lower=1e-17), marker="o", color=cols[i % 5], label=f"δ = {rg:g}"); ax.fill_between(s.index, s[0.25].clip(lower=1e-17), s[0.75].clip(lower=1e-17), color=cols[i % 5], alpha=0.12, linewidth=0)
        ax.set_yscale("log"); ax.set_xlabel("refinement round"); ax.set_ylabel("‖K̃ũ−r̃‖/‖r̃‖, original system (median, IQR)")
        ax.set_title(f"corrector: {corr}" + (" (classical diagnostic)" if "classical" in corr else " (ideal statevector)") + f", {g0[['instance_id','factor_index']].drop_duplicates().shape[0]} systems", fontsize=9); ax.legend(fontsize=7)
    ax = axes[-1]; g1 = q[(q.corrector == corrs[0]) & (q.ir_round == 0)]
    for i, rg in enumerate(sorted(g1.regularization.unique())):
        gg = g1[g1.regularization == rg]; ax.scatter(gg.kappa_scaled, gg.kappa_target, s=22, color=cols[i % 5], label=f"δ = {rg:g}", edgecolors="none")
    lim = [g1.kappa_scaled.min() * 0.5, g1.kappa_scaled.max() * 2]; ax.plot(lim, lim, color="#c3c2b7", linewidth=1)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("κ of the Ruiz-scaled system"); ax.set_ylabel("κ of the regularised scaled target"); ax.set_title("Regularisation bounds κ by ≈ (1+δ)/δ"); ax.legend(fontsize=7)
    savefig(fig, "10_regularization_refinement.png")

@with_data
def plot_integrated():
    ls = LS[LS.method.str.startswith("hhl")]
    if ls.empty: raise ValueError("no integrated HHL solves")
    runs = list(ls.run_id.unique())[:6]; ncol = min(3, len(runs)); nrow = int(math.ceil(len(runs) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 4.0 * nrow), squeeze=False)
    for ax, rid in zip(axes.ravel(), runs):
        g = ls[ls.run_id == rid].sort_values("solve_index")
        ax.plot(g.solve_index, g.crit_initial.clip(lower=1e-16), marker="D", color="#4a3aa7", label="criterion value of the HHL direction (before refinement)")
        ax.plot(g.solve_index, g.crit_final.clip(lower=1e-16), marker="o", color="#1baf7a", label="criterion value of the direction used")
        rej = g[(~g.accepted) & (~g.fallback)]
        if len(rej): ax.scatter(rej.solve_index, rej.crit_final.clip(lower=1e-16), marker="s", facecolors="none", edgecolors="#eda100", s=70, label="rejected direction used (fallback off)", zorder=6)
        fb = g[g.fallback]
        if len(fb): ax.scatter(fb.solve_index, fb.crit_initial.clip(lower=1e-16), marker="x", color="#e34948", s=50, label="classical fallback", zorder=5)
        ax.axhline(g.accept_rtol.iloc[0], color="#c3c2b7", linestyle=":", label="acceptance threshold")
        ax.set_yscale("log"); ax.set_xlabel("KKT solve index"); ax.set_ylabel("criterion value (combined: max of block/η and backward/tol)"); ax.set_title(rid.replace("__", "\n"), fontsize=7); ax.legend(fontsize=5)
    for ax in axes.ravel()[len(runs):]: ax.axis("off")
    savefig(fig, "11_hhl_inside_coneqp.png")

@with_data
def plot_quantum_simtime():
    q = QD[(QD.experiment == "Q1_captured") & (QD.status == "ok") & (QD.recon == "ideal_statevector") & (QD.ir_round == 0)]
    if q.empty: raise ValueError("no Q1 data")
    fig, ax = plt.subplots(figsize=(6.5, 4.2)); s = q.groupby("n_qubits")[["t_prep", "t_circuit_sim", "t_readout_probs"]].median()
    for col, colr in [("t_prep", "#9ec5f4"), ("t_circuit_sim", "#2a78d6"), ("t_readout_probs", "#0d366b")]: ax.plot(s.index, s[col], marker="o", color=colr, label=col)
    ax.set_yscale("log"); ax.set_xlabel("total qubits"); ax.set_ylabel("seconds (median), classical statevector simulation"); ax.set_title("Simulator time components vs. circuit size"); ax.legend()
    savefig(fig, "12_quantum_simulation_time.png")

@with_data
def plot_sdplib():
    d = DF[DF.experiment == "S1_sdplib"]
    if d.empty: raise ValueError("no SDPLIB data")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2)); piv = d.pivot_table(index="instance_id", columns="method", values="wall_method", aggfunc="median")
    piv.plot.bar(ax=axes[0], color=[method_color(c) for c in piv.columns], width=0.8, edgecolor="#fcfcfb"); axes[0].set_yscale("log"); axes[0].set_ylabel("wall time [s]"); axes[0].set_title("SDPLIB: runtime per method"); axes[0].legend(fontsize=6)
    k = d[d.method == "custom_exact"].set_index("instance_id")[["kappa_first", "kappa_max"]]
    k.plot.bar(ax=axes[1], color=["#9ec5f4", "#0d366b"], width=0.7, edgecolor="#fcfcfb"); axes[1].set_yscale("log"); axes[1].set_ylabel("κ₂(K̃)"); axes[1].set_title("SDPLIB: KKT conditioning (first vs. worst iteration)")
    savefig(fig, "13_sdplib.png")

@with_data
def plot_refinement_and_matvec():
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
    o = OREF if "OREF" in globals() else pd.DataFrame()
    if len(o) and "err" in o:
        cols = ["#0b0b0b", "#e34948", "#eb6834", "#2a78d6", "#4a3aa7", "#9085e9", "#1baf7a"]
        for i, (lab, g) in enumerate(o.groupby("inner_solver")):
            m = g.groupby("round").err.median(); axes[0].plot(m.index, m.values.clip(1e-17), marker="o", color=cols[i % len(cols)], label=lab)
        axes[0].axhline(1e-7, color="#c3c2b7", linestyle=":"); axes[0].set_yscale("log"); axes[0].set_xlabel("outer refinement round (inner tolerance 1e-3)"); axes[0].set_ylabel("max(pres, dres, valid rel. gap), median")
        axes[0].set_title("E5: optimisation-level refinement by inner KKT solver"); axes[0].legend(fontsize=6)
    d = DF[(DF.experiment == "E3_scaling") & (DF.family == "sparse_banded") & DF.method.str.startswith(("minres_base", "minres_fused", "minres_comb", "gmres_stalelu"))]
    for meth, g in d.groupby("method"):
        m = g.groupby("kkt_dim").krylov_iters_total.median()
        if (m > 0).any(): axes[1].plot(m.index, m.values, marker="o", color=method_color(meth), label=meth)
    axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].set_xlabel("KKT dimension"); axes[1].set_ylabel("Σ Krylov iterations per run (median)"); axes[1].set_title("Krylov work by method, sparse_banded"); axes[1].legend(fontsize=6)
    mv = MV if "MV" in globals() else pd.DataFrame()
    if len(mv):
        axes[2].plot(mv.kkt_dim, mv.t_matvec_cpu, marker="o", color="#2a78d6", label="CPU mat-vec"); axes[2].plot(mv.kkt_dim, mv.t_matvec_gpu, marker="o", color="#d03b3b", label="GPU mat-vec")
        axes[2].set_xscale("log"); axes[2].set_yscale("log"); axes[2].set_xlabel("KKT dimension"); axes[2].set_ylabel("seconds per sparse mat-vec"); axes[2].set_title("E6: CPU vs GPU mat-vec"); axes[2].legend()
    else:
        axes[2].axis("off"); axes[2].set_title("E6 GPU mat-vec: skipped (no GPU in this run)")
    savefig(fig, "14_outer_refinement_and_matvec.png")

@with_data
def plot_kernels_and_spectrum():
    fig, axes = plt.subplots(1, 4, figsize=(22, 4.4)); S_ = DF[DF.experiment == "E3_scaling"]
    for ax, fam, ttl in [(axes[0], "sparse_banded", "high-fill sparse family"), (axes[1], "sparse_lowfill", "low-fill control family")]:
        d = S_[(S_.family == fam) & S_.method.isin(["cvxopt_default", "custom_exact_reduced", "minres_base_comb_cpu", "minres_fused_comb_cpu", "minres_base_comb_gpu", "minres_fused_comb_gpu"])]
        for meth, g in d.groupby("method"):
            m = g.groupby("kkt_dim").wall_method.median(); ax.plot(m.index, m.values, marker="o", color=method_color(meth), label=meth, linestyle="--" if meth in ("minres_base_comb_cpu", "minres_base_comb_gpu") else "-")
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("KKT dimension"); ax.set_ylabel("wall time [s] (median)"); ax.set_title(ttl + " (dashed = baseline kernel)"); ax.legend(fontsize=6)
    if "SPEC7" in globals() and len(SPEC7):
        ax = axes[2]; ax.plot(SPEC7.ipm_iteration, SPEC7.minres_its_augmented, marker="o", color="#2a78d6", label="MINRES its, augmented (Ruiz)"); ax.plot(SPEC7.ipm_iteration, SPEC7.minres_its_reduced_jacobi, marker="o", color="#e34948", label="MINRES its, reduced (Jacobi, matrix-free)")
        ax.set_yscale("log"); ax.set_xlabel("IPM iteration"); ax.set_ylabel("Krylov iterations (2 solves)"); ax2 = ax.twinx(); ax2.plot(SPEC7.ipm_iteration, SPEC7.kappa_scaled, marker="s", color="#0b0b0b", linestyle=":", label="κ(scaled KKT)"); ax2.set_yscale("log"); ax2.set_ylabel("κ of Ruiz-scaled KKT")
        ax.set_title("E7: target system and κ along one IPM run"); ax.legend(fontsize=6, loc="upper left"); ax2.legend(fontsize=6, loc="lower right")
    else: axes[2].axis("off")
    if "DEFL7" in globals() and len(DEFL7):
        axes[3].plot(DEFL7.n_deflated, DEFL7.minres_its, marker="o", color="#1baf7a"); axes[3].set_yscale("log"); axes[3].set_xlabel("exactly deflated smallest-|λ| eigenvectors"); axes[3].set_ylabel("MINRES iterations"); axes[3].set_title("E7: deflation upper bound (last IPM iteration)")
    else: axes[3].axis("off")
    savefig(fig, "15_kernels_lowfill_spectrum.png")

def performance_profile(d, methods, time_col="wall_method"):
    """Dolan-More performance profile: per instance the ratio r = t_m / min_m' t_m' (runs that were not accepted count as failed: r = inf);
    rho_m(tau) = fraction of instances with r <= tau.  Returns (long DataFrame [method, tau, rho], per-instance ratio table)."""
    g = d[d.method.isin(methods)].groupby(["instance_id", "method"]).agg(t=(time_col, "median"), ok=("accepted", "all")).reset_index()
    g.loc[~g.ok, "t"] = np.inf; T = g.pivot_table(index="instance_id", columns="method", values="t", aggfunc="first").reindex(columns=[m for m in methods if m in set(g.method)])
    T = T.dropna(how="all"); tmin = T.min(axis=1); R = T.div(tmin, axis=0); R[np.isinf(R) | R.isna()] = np.inf
    fin = R.values[np.isfinite(R.values)]; tmax = float(fin.max()) if fin.size else 1.0; taus = np.unique(np.concatenate([[1.0], np.geomspace(1.0, max(2.0, 1.05 * tmax), 300)]))
    rows = [{"method": m, "tau": t, "rho": float((R[m] <= t).mean()), "instances": int(len(R))} for m in R.columns for t in taus]
    return pd.DataFrame(rows), R

PP_ALL = []
@with_data
def plot_performance_profiles():
    panels = [("E3_scaling", "sparse_banded", "sparse high-fill family (all seeds)"), ("E3_scaling", "sparse_lowfill", "sparse low-fill family (all seeds)"), ("E8_maros_meszaros", "maros_meszaros", "Maros-Meszaros (real-world QPs)")]
    meths = ["cvxopt_default", "custom_exact", "custom_exact_reduced", "gmres_stalelu_comb", "minres_fused_comb_cpu", "minres_fused_comb_switch_cpu", "minres_fused_comb_gpu", "minres_fused_comb_gpu_mm", "minres_fused_comb_auto"]
    have = [(e, f, t) for e, f, t in panels if len(DF[(DF.experiment == e) & (DF.family == f)])]
    if not have: raise ValueError("no data for performance profiles")
    fig, axes = plt.subplots(1, len(have), figsize=(5.6 * len(have), 4.3), squeeze=False); axes = axes[0]
    for ax, (e, f, ttl) in zip(axes, have):
        d = DF[(DF.experiment == e) & (DF.family == f)]; pp, R = performance_profile(d, meths)
        if pp.empty: ax.axis("off"); continue
        pp["experiment"] = e; pp["family"] = f; PP_ALL.append(pp)
        for m, g in pp.groupby("method", sort=False): ax.step(g.tau, g.rho, where="post", color=method_color(m), label=m, linewidth=1.8)
        ax.set_xscale("log", base=2); ax.set_ylim(-0.02, 1.02); ax.set_xlabel("τ  (time within a factor τ of the fastest accepted method)"); ax.set_ylabel("fraction of instances ρ(τ)"); ax.set_title(f"Performance profile: {ttl}, {pp.instances.iloc[0]} instances"); ax.legend(fontsize=6, loc="lower right")
    savefig(fig, "16_performance_profiles.png")
    if PP_ALL: pd.concat(PP_ALL, ignore_index=True).to_csv(os.path.join(RESULTS_DIR, "performance_profiles.csv"), index=False)

@with_data
def plot_seeds_mm_memory():
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
    # (a) seed-to-seed variability of the time ratio MINRES / CVXOPT on the sparse families
    ax = axes[0]; S_ = DF[(DF.experiment == "E3_scaling") & DF.family.isin(["sparse_banded", "sparse_lowfill"])]
    for fam, mk in [("sparse_banded", "o"), ("sparse_lowfill", "s")]:
        for meth, colr in [("minres_fused_comb_cpu", "#5598e7"), ("minres_fused_comb_gpu", "#eb6834"), ("custom_exact_reduced", "#c98500")]:
            a_ = S_[(S_.family == fam) & (S_.method == meth)].groupby(["instance_id", "kkt_dim"]).wall_method.median().reset_index(); b_ = S_[(S_.family == fam) & (S_.method == "cvxopt_default")].groupby("instance_id").wall_method.median()
            if a_.empty or b_.empty: continue
            a_["ratio"] = a_.wall_method / a_.instance_id.map(b_); a_ = a_.dropna()
            if a_.empty: continue
            ax.scatter(a_.kkt_dim, a_.ratio, color=colr, marker=mk, s=18, alpha=0.6, label=f"{meth} / cvxopt, {fam}")
            med = a_.groupby("kkt_dim").ratio.median(); ax.plot(med.index, med.values, color=colr, linestyle="-" if fam == "sparse_banded" else ":", linewidth=1.2)
    ax.axhline(1.0, color="#0b0b0b", linewidth=0.8, linestyle="--"); ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("KKT dimension"); ax.set_ylabel("wall-time ratio vs cvxopt_default (one point per seed)"); ax.set_title("Seed-to-seed variability (E3)"); ax.legend(fontsize=5.5)
    # (b) Maros-Meszaros: time of each method vs the unmodified solver
    ax = axes[1]; d8 = DF[DF.experiment == "E8_maros_meszaros"]
    if len(d8):
        b_ = d8[d8.method == "cvxopt_default"].set_index("instance_id").wall_method
        for meth, g in d8[d8.method != "cvxopt_default"].groupby("method"):
            g = g.set_index("instance_id"); x_ = b_.reindex(g.index); ok = g.accepted.values
            ax.scatter(x_[ok], g.wall_method[ok], color=method_color(meth), s=22, label=f"{meth} (accepted)"); ax.scatter(x_[~ok], g.wall_method[~ok], color=method_color(meth), s=30, marker="x", label=f"{meth} (not accepted)" if (~ok).any() else None)
        lim = [max(1e-3, min(b_.min(), d8.wall_method.min()) * 0.7), max(b_.max(), d8.wall_method.max()) * 1.4]; ax.plot(lim, lim, color="#0b0b0b", linewidth=0.8, linestyle="--"); ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel("cvxopt_default wall time [s]"); ax.set_ylabel("method wall time [s]"); ax.set_title(f"Maros-Meszaros ({d8.instance_id.nunique()} QPs): below the diagonal = faster"); ax.legend(fontsize=5.5)
    else: ax.axis("off"); ax.set_title("E8: no Maros-Meszaros runs")
    # (c) peak memory increase per run vs size
    ax = axes[2]; M_ = DF[DF.experiment.isin(["E3_scaling", "E8_maros_meszaros"]) & DF.mem_rss_increase_mb.notna()] if "mem_rss_increase_mb" in DF else pd.DataFrame()
    if len(M_):
        for meth in ["cvxopt_default", "custom_exact", "custom_exact_reduced", "minres_fused_comb_cpu", "minres_fused_comb_gpu"]:
            g = M_[(M_.method == meth) & (M_.family == "sparse_banded")].groupby("kkt_dim").mem_rss_increase_mb.median()
            if len(g): ax.plot(g.index, g.values.clip(0.1), marker="o", color=method_color(meth), label=f"{meth} (RSS increase)")
        gg = M_[(M_.method == "minres_fused_comb_gpu") & (M_.family == "sparse_banded")]
        if len(gg) and "mem_gpu_pool_peak_mb" in gg and gg.mem_gpu_pool_peak_mb.notna().any():
            g = gg.groupby("kkt_dim").mem_gpu_pool_peak_mb.median(); ax.plot(g.index, g.values.clip(0.1), marker="s", linestyle=":", color="#eb6834", label="minres_fused_comb_gpu (CuPy pool peak)")
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("KKT dimension"); ax.set_ylabel("peak memory [MB] (median over runs)"); ax.set_title("Peak memory per coneqp call (high-fill family)"); ax.legend(fontsize=6)
    else: ax.axis("off")
    savefig(fig, "17_seeds_maros_meszaros_memory.png")

@with_data
def plot_tolerance_shots_nullspace():
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
    ax = axes[0]; tsd = DF[DF.experiment == "E4b_tolerance_sweep"].copy()
    if len(tsd):
        tsd["base"] = tsd.method.str.split("@").str[0]; tsd["delta"] = tsd.base.str.extract(r"noise_(?:iso|block)_(.*)$")[0].astype(float)
        for model, colr in [("iso", "#e34948"), ("block", "#8b1a1a")]:
            g = tsd[tsd.base.str.contains(f"_{model}_")]
            for j, (tau, gt) in enumerate(sorted(g.groupby("ipm_tol_level"), key=lambda kv: kv[0])):
                s = gt.groupby("delta").accepted.mean().sort_index(); ax.plot(s.index, s.values, marker="o", color=colr, alpha=0.35 + 0.3 * j, label=f"{model} noise, τ={tau:.0e}")
        ax.set_xscale("log"); ax.set_ylim(-0.05, 1.05); ax.set_xlabel("injected relative direction error δ"); ax.set_ylabel("fraction accepted (thresholds scaled with τ)"); ax.set_title("E4b: tolerated direction error vs IPM tolerance τ"); ax.legend(fontsize=5.5)
    else: ax.axis("off"); ax.set_title("E4b: no data")
    ax = axes[1]; d5e = DF[DF.experiment == "Q5e_shot_sweep"]
    if len(d5e):
        g = d5e.groupby("sweep_samples").agg(strict=("strict_method_success", "mean"), lo=("strict_method_success", lambda x: x.groupby(d5e.loc[x.index, "sweep_seed"]).mean().min()), hi=("strict_method_success", lambda x: x.groupby(d5e.loc[x.index, "sweep_seed"]).mean().max()), its=("iterations", "median"), ref=("n_refine_total", "median")).sort_index()
        ax.plot(g.index, g.strict, marker="o", color="#4a3aa7", label="strict success rate (mean over seeds x instances)"); ax.fill_between(g.index, g.lo, g.hi, color="#4a3aa7", alpha=0.15, linewidth=0, label="range over sampling seeds")
        ax.set_xscale("log"); ax.set_ylim(-0.05, 1.05); ax.set_xlabel("accepted samples per refinement round (both settings)"); ax.set_ylabel("strict success rate"); ax2 = ax.twinx(); ax2.plot(g.index, g.ref, marker="s", linestyle=":", color="#c98500", label="median refinement rounds per run"); ax2.set_ylabel("refinement rounds per run")
        ax.set_title(f"Q5e: finite-shot polynomial solver, {d5e.instance_id.nunique()} QPs x {d5e.sweep_seed.nunique()} seeds"); ax.legend(fontsize=6, loc="lower right"); ax2.legend(fontsize=6, loc="upper left")
    else: ax.axis("off"); ax.set_title("Q5e: no data")
    ax = axes[2]; q5d = Q5D if "Q5D" in globals() and len(Q5D) else pd.DataFrame()
    if len(q5d):
        ax.scatter(q5d.kappa_augmented_scaled, q5d.kappa_reduced_jacobi, color="#0d366b", s=24, label="κ: null-space (Jacobi) vs augmented (Ruiz)")
        lim = [min(q5d.kappa_augmented_scaled.min(), q5d.kappa_reduced_jacobi.min()) * 0.7, max(q5d.kappa_augmented_scaled.max(), q5d.kappa_reduced_jacobi.max()) * 1.4]; ax.plot(lim, lim, color="#0b0b0b", linewidth=0.8, linestyle="--")
        ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("κ of the Ruiz-scaled augmented KKT matrix"); ax.set_ylabel("κ of the Jacobi-scaled null-space matrix ZᵀSZ")
        c3 = [c_ for c_ in q5d.columns if c_.startswith("degree_augmented_eps")]
        if c3:
            e_ = c3[0].split("eps")[1]; ax2 = ax.twinx(); ax2.scatter(q5d.kappa_augmented_scaled, q5d[f"degree_augmented_eps{e_}"] / q5d[f"degree_oss_eps{e_}"], color="#1baf7a", marker="^", s=24, label=f"degree ratio augmented / null-space at ε={e_}"); ax2.set_yscale("log"); ax2.set_ylabel("walk-step ratio (odd polynomial / SPD polynomial)"); ax2.legend(fontsize=6, loc="lower right")
        ax.set_title("Q5d: target system for the polynomial solver"); ax.legend(fontsize=6, loc="upper left")
    else: ax.axis("off"); ax.set_title("Q5d: no data")
    savefig(fig, "18_tolerance_shots_nullspace.png")

for f in [plot_kernels_and_spectrum, plot_refinement_and_matvec, plot_runtime_vs_size, plot_accuracy_vs_runtime, plot_convergence, plot_conditioning, plot_criterion_ablation, plot_variability, plot_noise,
          plot_quantum_error_vs_clock, plot_quantum_readout, plot_quantum_regularization, plot_integrated, plot_quantum_simtime, plot_sdplib,
          plot_performance_profiles, plot_seeds_mm_memory, plot_tolerance_shots_nullspace]:
    f()
print("plots done:", sorted(os.listdir(PLOTS_DIR)))

## 23. Summary report

`results/summary_report.md` holds one table per experiment, computed from the result tables of this run: success at the three levels, iterations, Krylov work, fallbacks and median times per method, and the corresponding summaries of the quantum experiments. It contains no interpretation; the paper discusses the results.

In [ ]:
# Summary report: one table per experiment, computed from the result tables of this run, written to results/summary_report.md.
DF = REC.df("benchmark_results"); LS = REC.df("linear_solve_logs"); QD = REC.df("quantum_results"); ST = REC.df("experiment_status")
L = []
def add(s=""): L.append(s)

def _cell(v):
    if isinstance(v, (bool, np.bool_)): return str(bool(v))
    if isinstance(v, (int, np.integer)): return str(int(v))
    if isinstance(v, (float, np.floating)): return "" if np.isnan(v) else f"{v:.3g}"
    return str(v)

def table(df):
    if df is None or len(df) == 0: return "(no data)\n"
    df = df.reset_index() if not isinstance(df.index, pd.RangeIndex) else df
    cols = [str(c) for c in df.columns]
    rows = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)] + ["| " + " | ".join(_cell(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(rows) + "\n"

def run_summary(d):
    return d.groupby("method").agg(runs=("run_id", "size"), accepted=("accepted", "mean"), fallback_free=("pure_method_success", "mean"), strict=("strict_method_success", "mean"),
                                   ipm_iters=("iterations", "median"), krylov_iters=("krylov_iters_total", "median"), fallbacks=("n_fallback_total", "sum"),
                                   wall_s=("wall_method", "median")).reindex([m for m in order_methods(list(d.method.unique()))])

def section(title, fn):
    add(f"## {title}\n")
    try: add(fn())
    except Exception as e: add(f"(not available: {type(e).__name__}: {e})\n")

add("# Summary report\n")
add(f"Version {CODE_VERSION}, profile `{CONFIG['run_profile']}`, device `{'gpu' if GPU_OK else 'cpu'}`, config hash `{CONFIG_HASH}`, seed {GLOBAL_SEED}, "
    f"{time.strftime('%Y-%m-%d %H:%M')}. {len(DF)} optimisation runs, {len(LS)} KKT solves, {len(QD)} quantum records. "
    f"CPU: {ENV['hardware'].get('cpu_model', '?')} ({ENV['hardware'].get('cpu_count')} cores); GPU: {GPU_STATUS.get('name') or 'none'}.\n")
add("Acceptance of the final iterate: " + ", ".join(f"{k} <= {v:g}" for k, v in CONFIG["accept"].items()) + ". "
    "`fallback_free` = accepted without classical fallback; `strict` = in addition every direction passed the acceptance test.\n")

section("Experiment status", lambda: table(ST[["experiment", "status", "reason"]]))
for exp, title in [("E1_contract_tiny", "E1, contract check"), ("E2_main", "E2, acceptance-test ablation"), ("E8_maros_meszaros", "E8, Maros-Meszaros QPs"),
                   ("E4_noise", "E4, injected direction error"), ("Q3_integrated", "Q3, HHL inside coneqp"),
                   ("Q5_integrated_cks", "Q5c, polynomial solver inside coneqp (no fallback)"), ("S1_sdplib", "S1, SDPLIB")]:
    if (DF.experiment == exp).any(): section(title, lambda exp=exp: table(run_summary(DF[DF.experiment == exp])))

def _e3():
    d = DF[DF.experiment == "E3_scaling"]; out = []
    for fam, g in d.groupby("family"):
        out.append(f"Median wall time [s], family `{fam}` (rows: method, columns: KKT dimension; fallback-free share in the last column)\n")
        piv = g.pivot_table(index="method", columns="kkt_dim", values="wall_method", aggfunc="median")
        piv["fallback_free"] = g.groupby("method").pure_method_success.mean(); out.append(table(piv.reindex([m for m in order_methods(list(piv.index))])))
    return "\n".join(out)
section("E3, scaling", _e3)

def _e4b():
    d = DF[DF.experiment == "E4b_tolerance_sweep"].copy(); d["base"] = d.method.str.split("@").str[0]
    return table(d.pivot_table(index="base", columns="ipm_tol_level", values="accepted", aggfunc="mean"))
if (DF.experiment == "E4b_tolerance_sweep").any(): section("E4b, share of accepted runs by IPM tolerance level", _e4b)

def _q5e():
    d = DF[DF.experiment == "Q5e_shot_sweep"]
    return table(d.groupby("sweep_samples").agg(runs=("run_id", "size"), strict=("strict_method_success", "mean"), accepted=("accepted", "mean"),
                                                ipm_iters=("iterations", "median"), refinement_rounds=("n_refine_total", "median")))
if (DF.experiment == "Q5e_shot_sweep").any(): section("Q5e, finite-shot polynomial solver over sampling seeds", _q5e)

def _q1():
    q = QD[(QD.experiment == "Q1_captured") & (QD.status == "ok") & (QD.recon == "ideal_statevector") & (QD.ir_round == 0)]
    return table(q.pivot_table(index="n_clock", columns=["scaling", "reciprocal_mode"], values="rel_err_original", aggfunc="median"))
section("Q1, HHL forward error on captured systems (ideal readout, median)", _q1)

def _q4():
    q = QD[QD.experiment.astype(str).str.startswith("Q4_regularization") & (QD.status == "ok")]
    last = q.sort_values("ir_round").groupby(["experiment", "instance_id", "factor_index", "regularization"]).tail(1)
    return table(last.groupby(["experiment", "regularization"]).agg(systems=("instance_id", "size"), final_rel_resid=("rel_resid_original", "median"), rounds=("ir_round", "median")))
section("Q4, regularised target with refinement (residual after the last round, median)", _q4)

section("Q5a, forward error of the quantum solvers on captured systems (ideal readout, median)", lambda: table(
    Q5A[Q5A.status == "ok"].groupby(["solver", "poly", "eps_target"], dropna=False).agg(systems=("rel_err_ideal", "size"), rel_err=("rel_err_ideal", "median"),
                                                                                         walk_steps=("oracle_queries", "median"), p_success=("p_success", "median"))))
section("Q5b, cost of refinement to a 1e-10 residual", lambda: table(
    Q5B.groupby(["poly", "eps_circuit", "samples_per_round"]).agg(systems=("rounds", "size"), reached=("reached_1e-10", "mean"), rounds=("rounds", "median"),
                                                                  total_samples=("total_samples", "median"), queries=("block_encoding_queries_total_with_AA", "median"))))
section("Q5d, augmented against null-space target", lambda: table(Q5D))
section("E5, outer refinement: median max(pres, dres, relgap) by round", lambda: table(OREF.pivot_table(index="round", columns="inner_solver", values="err", aggfunc="median")))
section("E7, spectrum along one IPM run", lambda: table(SPEC7.drop(columns=["instance_id"], errors="ignore")))
section("E7, exact-deflation bound at the last IPM iteration", lambda: table(DEFL7.drop(columns=["instance_id"], errors="ignore")))
section("D1, dequantisation probe (median)", lambda: table(DEQ.drop(columns=["instance_id", "factor_index"], errors="ignore").median().to_frame("median")))

REPORT = "\n".join(L)
with open(os.path.join(RESULTS_DIR, "summary_report.md"), "w") as f: f.write(REPORT)
print(f"summary_report.md: {len(REPORT.splitlines())} lines")

## 24. Export

Final CSV tables, the tiny and small instances and all captured KKT systems as `.npz`, `results/README.md` (column and method glossary), and the bundle `results_bundle_<cpu|gpu>_<profile>.zip` in the output directory.

In [ ]:
# Export: final CSV tables, instances and captured KKT systems as .npz, results/README.md, and the zip bundle.
REC.flush(final=True)

# instances and captured KKT systems
for inst_id, caps in CAPTURED.items():
    inst = INSTANCES[inst_id]
    data = {"K": np.stack([c["K"] for c in caps]), "factor_index": np.array([c["factor_index"] for c in caps]), "kappa": np.array([c["kappa"] for c in caps]), "nt_proxy": np.array([c["nt_proxy"] for c in caps]),
            "di": np.stack([c["di"] for c in caps]) if caps[0]["di"].size else np.zeros((len(caps), 0)), "rhs_first_solve": np.stack([c["solves"][0]["rhs"] for c in caps]), "u_first_solve": np.stack([c["solves"][0]["u"] for c in caps]),
            "rhs_all": np.array([np.stack([s["rhs"] for s in c["solves"]]) for c in caps], dtype=object), "u_all": np.array([np.stack([s["u"] for s in c["solves"]]) for c in caps], dtype=object),
            "n": inst["n"], "p": inst["p"], "m_packed": inst["kkt_dim"] - inst["n"] - inst["p"], "dims_l": inst["dims"]["l"], "dims_s": np.array(inst["dims"]["s"])}
    if inst["kind"] == "coneqp":
        data.update({"P": np.asarray(inst["P"].todense() if sp.issparse(inst["P"]) else inst["P"]), "q": inst["q"], "G": np.asarray(inst["G"].todense() if sp.issparse(inst["G"]) else inst["G"]), "h": inst["h"],
                     "A": np.asarray(inst["A"].todense() if (inst["p"] and sp.issparse(inst["A"])) else (inst["A"] if inst["p"] else np.zeros((0, inst["n"])))), "b": inst["b"] if inst["p"] else np.zeros(0),
                     "x_star": inst["x_star"], "y_star": inst["y_star"], "z_star": inst["z_star"], "s_star": inst["s_star"], "f_star": inst["f_star"]})
    else:
        data.update({"c": inst["q"], "G": inst["G"], "h": inst["h"], "block_sizes": np.array(inst["block_sizes"]), "f_star_sdplib": inst["f_star"]})
    np.savez_compressed(os.path.join(NPZ_DIR, f"{inst_id}_instance_and_kkt.npz"), **data)
for inst in list(INSTANCES.values()):
    if inst["family"] in ("dense_kappa", "dense_noeq") and inst["seed"] == 0 and inst["n"] <= 40:
        np.savez_compressed(os.path.join(NPZ_DIR, f"{inst['id']}.npz"), P=np.asarray(inst["P"]), q=inst["q"], G=np.asarray(inst["G"]), h=inst["h"], A=np.asarray(inst["A"]) if inst["p"] else np.zeros((0, inst["n"])),
                            b=inst["b"] if inst["p"] else np.zeros(0), x_star=inst["x_star"], y_star=inst["y_star"], z_star=inst["z_star"], s_star=inst["s_star"], f_star=inst["f_star"])
    if inst["family"] == "sparse_banded" and inst["seed"] == 0 and inst["n"] <= 400:
        sp.save_npz(os.path.join(NPZ_DIR, f"{inst['id']}_P.npz"), sp.csr_matrix(inst["P"])); sp.save_npz(os.path.join(NPZ_DIR, f"{inst['id']}_G.npz"), sp.csr_matrix(inst["G"]))
        if inst["p"]: sp.save_npz(os.path.join(NPZ_DIR, f"{inst['id']}_A.npz"), sp.csr_matrix(inst["A"]))
        np.savez_compressed(os.path.join(NPZ_DIR, f"{inst['id']}_vectors.npz"), q=inst["q"], h=inst["h"], b=inst["b"] if inst["p"] else np.zeros(0), x_star=inst["x_star"], y_star=inst["y_star"], z_star=inst["z_star"], s_star=inst["s_star"], f_star=inst["f_star"])
print("npz files:", len(os.listdir(NPZ_DIR)))

README = f"""# Results

Written by the notebook, version {CODE_VERSION}, profile `{CONFIG['run_profile']}`, device `{'gpu' if GPU_OK else 'cpu'}`, config hash `{CONFIG_HASH}`, seed {GLOBAL_SEED}, {time.strftime('%Y-%m-%d')}.
Times are wall-clock seconds; residuals and errors are relative unless the column name says otherwise. A missing value means that the quantity
was not computed for the method, was above a size limit (`cond_limit`, `max_qubits`), or that the run failed (`error`, `failures.csv`).
Skipped experiments and the reason are listed in `experiment_status.csv`. Failed runs are kept in every table.

## Identifiers

`run_id` = `<instance_id>__<method>__r<repeat>`. Benchmark rows carry `code_version`, `config_hash`, `experiment`, `instance_id`, `seed`, `method`,
`criterion`, `scaling`, `regularization` and `target_tol`. Solve logs and iteration logs join on `run_id`; quantum rows are identified by
`instance_id`, `factor_index` (the captured KKT system) and the solver settings. `instances.csv` lists n, m, p, kappa(P), structure and nnz.

## Methods

Tolerances in method names are the nominal tolerance tau; the relative-residual and backward-error tests accept at 10 tau.

* `cvxopt_default`: unmodified `coneqp` / `conelp`. `custom_exact`: LU of the full KKT matrix through the callback.
  `custom_exact_reduced`: CHOLMOD on the reduced quasi-definite system.
* `minres_relres_1e-06`, `minres_be_<tau>`, `minres_block_<eta>`, `minres_comb_0.1_1e-10`, `minres_rrfloor_1e-06`: SciPy MINRES with the
  relative-residual, backward-error, block, combined and floored relative-residual test, Ruiz-scaled; `_noscale` without scaling.
* `minres_adaptive_nt_be`: backward-error tolerance tied to the NT-scaling spread. `cg_reduced_be_1e-10`: CG on the reduced system (p = 0).
* `minres_base_comb_<cpu|gpu>`: baseline MINRES kernel with the combined test (eta = 0.1, tau = 1e-10); `_ws` warm start.
  `minres_fused_comb_<cpu|gpu|auto>`: latency-optimised kernel; `_auto` uses the GPU from KKT dimension {CONFIG['inexact']['gpu_auto_min_dim']};
  `_floor8` block floor 1e-8; `_gpu_mm` GPU from KKT dimension 2 000 (Maros-Meszaros runs).
* `_switch`: exact solver for the rest of the run after the first rejected direction (`utilisation` = share of solves by the method).
* `gmres_stalelu_comb`: GMRES preconditioned with the factors of an earlier IPM iteration (`n_factorizations` = factorisations made).
* `lsqr_sqrt_comb`: LSQR on the square-root factor (p = 0, dense P). `oss_cg_comb`: CG on the null-space system (`kappa_reduced_max`, `reduced_dim`).
* `exact_noise_iso_<delta>`, `exact_noise_block_<delta>`: exact direction plus an injected relative error (sensitivity only).
  In E4b the names are `<method>@tau<level>` with columns `ipm_tol_level` and `feastol_option`.
* `hhl_*` (Q3): HHL with `noscale_res` (no scaling, C = Delta) or `scaled_kk` (Ruiz, kappa known), `nc<k>` clock qubits, `fb` / `nofb`
  with or without classical fallback, `aa<n>` emulated amplitude amplification with n accepted samples per setting, `blockonly` block test alone.
  Without fallback a rejected direction is passed to CVXOPT and counted in `n_solves_unaccepted_used`.
* `lcu_opt_*`, `lcu_cks_*` (Q5c): polynomial solver with the optimal polynomial or the CKS expansion at circuit precision `eps`;
  `lcu_oss_*`: polynomial solver on the null-space target; `hhl_*_ir60_nofb`: HHL with the same refinement budget. Q5e rows carry
  `sweep_samples` and `sweep_seed`.

## Files

* `benchmark_results.csv`: one row per optimisation run, with timing components, solve counts (`n_solves_accepted_direct`,
  `n_solves_accepted_refined`, `n_solves_unaccepted_used`, `n_fallback_total`), recomputed metrics (`pres_rec`, `dres_rec`, `relgap_valid`,
  `obj_rel_err`, `x_rel_err`), CVXOPT's own values (`*_cvx`), condition numbers, peak memory (`mem_*`) and the three success levels
  `accepted`, `pure_method_success` and `strict_method_success`.
* `iteration_logs.csv`: CVXOPT's per-iteration table joined with the factorisation at that iterate.
* `linear_solve_logs.csv`: one row per KKT solve with all three test values before and after refinement, fallback, Krylov iterations,
  timings and, for the quantum solvers, the `q_*` columns (qubits, kappa of the target, post-selection probability, readout, walk steps).
* `quantum_results.csv`: HHL on captured systems (Q1), Qiskit cross-check (Q2) and regularisation (Q4). `recon` is `ideal_statevector`,
  `shots` or `aa_shots`; `ir_round` is the refinement round; errors are given on the target and on the original system. In Q4 the
  corrector `classical_lu_regularized` is a classical diagnostic, not a quantum solve.
* `qls_comparison_hhl_vs_cks.csv` (Q5a), `cks_refinement.csv` (Q5b), `oss_target_comparison.csv` (Q5d), `quantum_resource_estimates.csv`.
* `spectrum_along_ipm.csv`, `deflation_bound.csv` (E7), `outer_refinement.csv` (E5), `dequantisation_probe.csv` (D1),
  `gpu_matvec_benchmark.csv` (E6, GPU runs only), `performance_profiles.csv`, `maros_meszaros_instances.csv`.
* `summary_report.md`, `failures.csv`, `experiment_status.csv`, `config.json`, `environment.json`, `instances.csv`, `plots/`,
  `instances_npz/` (tiny instances with their captured KKT systems, small dense instances, sparse instances as SciPy `.npz`).
"""
with open(os.path.join(RESULTS_DIR, "README.md"), "w") as f: f.write(README)

bundle = os.path.join(CONFIG["output_root"], f"results_bundle_{'gpu' if GPU_OK else 'cpu'}_{RUN_PROFILE}.zip")
if os.path.exists(bundle): os.remove(bundle)
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(RESULTS_DIR):
        if root.endswith("partial"): continue
        for fn in files:
            full = os.path.join(root, fn); zf.write(full, os.path.relpath(full, CONFIG["output_root"]))
print(f"bundle: {bundle} ({os.path.getsize(bundle) / 2**20:.2f} MB)")
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(bundle) if not bundle.startswith("/kaggle") else bundle))
except Exception:
    pass
print(f"\nTotal time: {time.time() - NOTEBOOK_T0:.0f}s"); print(REC.df("experiment_status")[["experiment", "status", "reason"]].to_string(index=False))

## 25. Outputs

On Kaggle the bundle appears under *Output* once the notebook version is saved; the link above downloads it directly. It contains every table, the figures, the captured KKT systems, the configuration, the environment record and the summary report. If a session stops before the end, the tables written so far are in `results/partial/`.

The tables and figures of the paper are produced from one or more bundles by the scripts in `scripts/` of the repository (see its README).